# SQM Complete Validation, Comparative Benchmarks and Ablation — V8

This is the complete end-to-end validation notebook for **Selective-Quotient Modulation (SQM)**. It retains all V7 reproduction, mathematical audit, ablation, native-family benchmarks, and decisive SQM-vs-HWI tests, and adds a publication-oriented **complete 1+4 comparison matrix** in which SQM is shown together with all four selected competitors:

- Hardware-aware Grassmannian,
- Cube-Split,
- SE-MOCZ,
- IM-MOCZ.

The new matrix reports all five schemes under four progressively harder operating conditions: clean flat fading, flat fading with hardware uncertainty, clean two-tap multipath, and combined two-tap multipath + CFO + I/Q + DC. A dedicated headline plot shows all five schemes under the combined stress test.

Where a competitor has no native receiver for a nuisance in the SQM model, the notebook does **not** silently invent equivalence: the adaptation is stated explicitly and, for the MOCZ-family hardware-uncertainty stress tests, an **oracle front-end-cleaned upper reference** is used. This deliberately favors the competitor and makes any surviving SQM advantage more conservative.


## Part I — Reproduce and audit the released results

The following cells are the original independent reproduction workflow and are retained for provenance.


In [ ]:
import sys
import os
import platform

print("=== Python Environment Check ===")
print("Python version :", sys.version.split()[0])
print("Python executable:", sys.executable)
print("Working directory:", os.getcwd())
print("Platform:", platform.platform())

print("\n=== Key Packages ===")

packages = ["numpy", "pandas", "scipy", "matplotlib", "torch"]

for name in packages:
    try:
        module = __import__(name)
        version = getattr(module, "__version__", "unknown")
        print(f"{name:12s} OK   version={version}")
    except Exception as e:
        print(f"{name:12s} MISSING/ERROR   {e}")


In [ ]:
from pathlib import Path
import os
import shutil

# Repository root
ROOT = Path.cwd()

# Dedicated reproduction workspace
REPRO = ROOT / "Reproduction_Work"
RESULTS = REPRO / "results"
TMP = REPRO / "tmp"
CACHE = REPRO / "cache"

for folder in [REPRO, RESULTS, TMP, CACHE]:
    folder.mkdir(parents=True, exist_ok=True)

# Direct temporary/cache files to the external Cyber Range disk
os.environ["TMPDIR"] = str(TMP)
os.environ["TMP"] = str(TMP)
os.environ["TEMP"] = str(TMP)
os.environ["XDG_CACHE_HOME"] = str(CACHE)
os.environ["TORCH_HOME"] = str(CACHE / "torch")
os.environ["PIP_CACHE_DIR"] = str(CACHE / "pip")

# Disk-space check
usage = shutil.disk_usage(ROOT)

print("=== Reproduction Workspace ===")
print("Repository :", ROOT)
print("Results    :", RESULTS)
print("Temporary  :", TMP)
print("Cache      :", CACHE)

print("\n=== External Disk Space ===")
print(f"Total : {usage.total / 1024**3:.1f} GB")
print(f"Used  : {usage.used / 1024**3:.1f} GB")
print(f"Free  : {usage.free / 1024**3:.1f} GB")


In [ ]:
from pathlib import Path

ROOT = Path.cwd()

required_files = [
    "Code/run_final_headtohead.py",
    "Code/evaluate_online_adaptive.py",
    "Code/evaluate_online_adaptive_revision.py",
    "Code/online_adaptive_diagnostics.py",
    "Code/make_adaptive_revision_figures.py",
    "Data/final_rerun_fractional_tdla_headtohead.csv",
    "Data/online_adaptive_revision_summary.csv",
    "Data/online_adaptive_revision_seed_metrics.csv",
    "Data/online_adaptive_packet_results.csv",
    "Data/online_regime_rank_sweeps.csv",
    "Artifact/fractional_incidence_codebook.npz",
    "Artifact/stage3_codebooks.npz",
    "requirements.txt",
    "environment.txt",
    "README.md",
]

print("=== Repository Integrity Check ===\n")

all_ok = True

for rel_path in required_files:
    path = ROOT / rel_path
    status = "OK" if path.exists() else "MISSING"
    print(f"{status:8s} {rel_path}")

    if not path.exists():
        all_ok = False

print("\nOverall status:", "PASS" if all_ok else "CHECK MISSING FILES")


In [ ]:
import subprocess
import sys

print("=== Python Code Compilation Check ===\n")

result = subprocess.run(
    [sys.executable, "-m", "compileall", "-q", "Code"],
    capture_output=True,
    text=True
)

if result.stdout:
    print("STDOUT:")
    print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

print("Return code:", result.returncode)

if result.returncode == 0:
    print("\nOverall status: PASS")
    print("All Python files compiled successfully.")
else:
    print("\nOverall status: FAIL")
    print("One or more Python files have compilation errors.")


In [ ]:
import subprocess
import sys
from pathlib import Path

print("=== Smoke Test: Main Head-to-Head Simulation ===\n")

smoke_summary = RESULTS / "smoke_summary.csv"
smoke_seed = RESULTS / "smoke_per_seed.csv"
smoke_paired = RESULTS / "smoke_paired.csv"

cmd = [
    sys.executable,
    "Code/run_final_headtohead.py",
    "--blocks", "2",
    "--seeds", "13",
    "--snrs", "10",
    "--output", str(smoke_summary.relative_to(ROOT)),
    "--per-seed-output", str(smoke_seed.relative_to(ROOT)),
    "--paired-output", str(smoke_paired.relative_to(ROOT)),
]

print("Running:")
print(" ".join(cmd))
print()

result = subprocess.run(
    cmd,
    cwd=ROOT,
    capture_output=True,
    text=True,
    env=os.environ.copy()
)

print("=== STDOUT ===")
print(result.stdout)

if result.stderr:
    print("=== STDERR ===")
    print(result.stderr)

print("Return code:", result.returncode)

if result.returncode == 0:
    print("\nOverall status: PASS")
    print("Kumar's main simulation executed successfully.")
else:
    print("\nOverall status: FAIL")


In [ ]:
import subprocess
import sys
import os
import time
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

print("=== Full Reproduction: Main Head-to-Head Experiment ===\n")

# Manuscript settings
SNRS = [5, 7, 10, 12, 15]
SEEDS = "13,59,151"
BLOCKS = 1200

run_dir = RESULTS / "main_headtohead"
run_dir.mkdir(parents=True, exist_ok=True)

all_summary = []
all_seed = []
all_paired = []

start_all = time.time()

for snr in tqdm(SNRS, desc="Reproducing SNR points", unit="SNR"):

    summary_file = run_dir / f"summary_{snr}dB.csv"
    seed_file = run_dir / f"per_seed_{snr}dB.csv"
    paired_file = run_dir / f"paired_{snr}dB.csv"
    log_file = run_dir / f"log_{snr}dB.txt"

    cmd = [
        sys.executable,
        "Code/run_final_headtohead.py",
        "--blocks", str(BLOCKS),
        "--seeds", SEEDS,
        "--snrs", str(snr),
        "--output", str(summary_file.relative_to(ROOT)),
        "--per-seed-output", str(seed_file.relative_to(ROOT)),
        "--paired-output", str(paired_file.relative_to(ROOT)),
    ]

    t0 = time.time()

    result = subprocess.run(
        cmd,
        cwd=ROOT,
        capture_output=True,
        text=True,
        env=os.environ.copy()
    )

    elapsed = time.time() - t0

    # Save full log on external disk
    with open(log_file, "w") as f:
        f.write("COMMAND:\n")
        f.write(" ".join(cmd) + "\n\n")
        f.write("STDOUT:\n")
        f.write(result.stdout)
        f.write("\nSTDERR:\n")
        f.write(result.stderr)
        f.write(f"\nReturn code: {result.returncode}\n")
        f.write(f"Runtime seconds: {elapsed:.2f}\n")

    if result.returncode != 0:
        print(f"\nERROR at {snr} dB")
        print(result.stderr)
        raise RuntimeError(f"Simulation failed at {snr} dB")

    # Load newly generated results
    all_summary.append(pd.read_csv(summary_file))
    all_seed.append(pd.read_csv(seed_file))
    all_paired.append(pd.read_csv(paired_file))


# ------------------------------------------------------------
# Combine the five SNR runs into manuscript-style result files
# ------------------------------------------------------------

main_summary = pd.concat(all_summary, ignore_index=True)
main_seed = pd.concat(all_seed, ignore_index=True)
main_paired = pd.concat(all_paired, ignore_index=True)

final_summary_file = RESULTS / "main_reproduced_summary.csv"
final_seed_file = RESULTS / "main_reproduced_per_seed.csv"
final_paired_file = RESULTS / "main_reproduced_paired.csv"

main_summary.to_csv(final_summary_file, index=False)
main_seed.to_csv(final_seed_file, index=False)
main_paired.to_csv(final_paired_file, index=False)

elapsed_all = time.time() - start_all

print("\n=== REPRODUCTION COMPLETE ===")
print(f"Total runtime: {elapsed_all:.1f} seconds")

print("\n=== Reproduced Main Results ===")
display(main_summary)

print("\nFiles saved:")
print(final_summary_file)
print(final_seed_file)
print(final_paired_file)


In [ ]:
import pandas as pd
import numpy as np

print("=== Exact Reproducibility Check ===\n")

released_file = ROOT / "Data" / "final_rerun_fractional_tdla_headtohead.csv"
reproduced_file = RESULTS / "main_reproduced_summary.csv"

released = pd.read_csv(released_file)
reproduced = pd.read_csv(reproduced_file)

# Sort both identically before comparison
released = released.sort_values(
    ["snr_db", "method"]
).reset_index(drop=True)

reproduced = reproduced.sort_values(
    ["snr_db", "method"]
).reset_index(drop=True)

print("Released rows   :", len(released))
print("Reproduced rows :", len(reproduced))

print("\n=== Column Check ===")
print("Released   :", list(released.columns))
print("Reproduced :", list(reproduced.columns))

# ------------------------------------------------------------
# Check labels / experiment structure
# ------------------------------------------------------------

same_structure = (
    released["snr_db"].equals(reproduced["snr_db"])
    and released["method"].equals(reproduced["method"])
    and released["blocks"].equals(reproduced["blocks"])
)

# ------------------------------------------------------------
# Numerical differences
# ------------------------------------------------------------

comparison = released[
    ["snr_db", "method", "PER_mean", "PER_95CI_halfwidth"]
].copy()

comparison = comparison.rename(
    columns={
        "PER_mean": "released_PER",
        "PER_95CI_halfwidth": "released_CI"
    }
)

comparison["reproduced_PER"] = reproduced["PER_mean"]
comparison["reproduced_CI"] = reproduced["PER_95CI_halfwidth"]

comparison["PER_difference"] = (
    comparison["reproduced_PER"] - comparison["released_PER"]
)

comparison["CI_difference"] = (
    comparison["reproduced_CI"] - comparison["released_CI"]
)

display(comparison)

# ------------------------------------------------------------
# Exact / numerical agreement
# ------------------------------------------------------------

per_match = np.allclose(
    released["PER_mean"],
    reproduced["PER_mean"],
    rtol=1e-12,
    atol=1e-12
)

ci_match = np.allclose(
    released["PER_95CI_halfwidth"],
    reproduced["PER_95CI_halfwidth"],
    rtol=1e-12,
    atol=1e-12
)

max_per_diff = comparison["PER_difference"].abs().max()
max_ci_diff = comparison["CI_difference"].abs().max()

print("\n=== Reproducibility Verdict ===")
print("Same experiment structure :", same_structure)
print("PER values match          :", per_match)
print("CI values match           :", ci_match)

print(f"Maximum PER difference    : {max_per_diff:.16g}")
print(f"Maximum CI difference     : {max_ci_diff:.16g}")

if same_structure and per_match and ci_match:
    print("\nSUCCESS: Main published results reproduced numerically.")
else:
    print("\nCHECK: Differences detected.")


In [ ]:
import shutil
from pathlib import Path

print("=== Preserve Released Adaptive Results ===\n")

source_data = ROOT / "Data"
reference_data = REPRO / "Data_Kumar_Original"

if reference_data.exists():
    print("Reference snapshot already exists:")
    print(reference_data)
else:
    shutil.copytree(source_data, reference_data)
    print("Created untouched reference snapshot:")
    print(reference_data)

print("\nOriginal Data folder:")
print(source_data)

print("\nReference copy:")
print(reference_data)

print("\nStatus: READY FOR ADAPTIVE REPRODUCTION")


In [ ]:
import subprocess
import sys
import os
import time
from pathlib import Path
from tqdm.auto import tqdm

print("=== Adaptive-Rank Reproduction ===\n")

adaptive_log = RESULTS / "adaptive_reproduction_log.txt"

cmd = [
    sys.executable,
    "Code/evaluate_online_adaptive_revision.py"
]

print("Running Kumar's original script:")
print(" ".join(cmd))
print("\nOutputs will be written into Data/")
print("Original Data/ has already been preserved in:")
print(reference_data)
print()

start = time.time()

# Write all process output directly to the external disk
with open(adaptive_log, "w") as log:

    process = subprocess.Popen(
        cmd,
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
        env=os.environ.copy()
    )

    # tqdm activity indicator
    with tqdm(
        desc="Adaptive experiment running",
        unit="s",
        bar_format="{desc}: {n:.0f} s elapsed"
    ) as pbar:

        while process.poll() is None:
            time.sleep(1)
            pbar.update(1)

    return_code = process.returncode

elapsed = time.time() - start

print("\n=== Adaptive Run Finished ===")
print("Return code :", return_code)
print(f"Runtime     : {elapsed:.1f} seconds")
print("Log saved   :", adaptive_log)

if return_code == 0:
    print("\nOverall status: PASS")
else:
    print("\nOverall status: FAIL")

# Show final part of log
print("\n=== Last lines of execution log ===")

with open(adaptive_log, "r") as f:
    lines = f.readlines()

for line in lines[-30:]:
    print(line, end="")


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

print("=== Adaptive-Rank Reproducibility Check ===\n")

# Kumar's untouched released results
REF = REPRO / "Data_Kumar_Original"

# Results generated by our fresh adaptive run
NEW = ROOT / "Data"

files_to_compare = [
    "online_adaptive_revision_summary.csv",
    "online_adaptive_revision_seed_metrics.csv",
    "online_adaptive_revision_by_segment.csv",
    "online_adaptive_packet_results.csv",
    "online_regime_rank_sweeps.csv",
]

overall_pass = True

for filename in files_to_compare:

    ref_file = REF / filename
    new_file = NEW / filename

    print(f"\n--- {filename} ---")

    if not ref_file.exists():
        print("Reference file missing")
        overall_pass = False
        continue

    if not new_file.exists():
        print("Reproduced file missing")
        overall_pass = False
        continue

    ref = pd.read_csv(ref_file)
    new = pd.read_csv(new_file)

    print("Reference shape :", ref.shape)
    print("Reproduced shape:", new.shape)

    same_columns = list(ref.columns) == list(new.columns)
    same_shape = ref.shape == new.shape

    print("Same columns:", same_columns)
    print("Same shape  :", same_shape)

    if not (same_columns and same_shape):
        overall_pass = False
        continue

    # Compare numeric columns
    numeric_cols = ref.select_dtypes(include=[np.number]).columns

    if len(numeric_cols) > 0:
        a = ref[numeric_cols].to_numpy(dtype=float)
        b = new[numeric_cols].to_numpy(dtype=float)

        numeric_match = np.allclose(
            a,
            b,
            rtol=1e-12,
            atol=1e-12,
            equal_nan=True
        )

        max_diff = np.nanmax(np.abs(a - b))

        print("Numeric match :", numeric_match)
        print("Max abs diff :", max_diff)

        if not numeric_match:
            overall_pass = False

    # Compare non-numeric columns
    other_cols = [
        c for c in ref.columns
        if c not in numeric_cols
    ]

    text_match = True

    for col in other_cols:
        if not ref[col].fillna("").equals(
            new[col].fillna("")
        ):
            text_match = False

    print("Text match    :", text_match)

    if not text_match:
        overall_pass = False


print("\n======================================")
print("ADAPTIVE REPRODUCTION VERDICT")
print("======================================")

if overall_pass:
    print("SUCCESS: Adaptive released results reproduced exactly.")
else:
    print("CHECK: One or more adaptive files differ.")


In [ ]:
import subprocess
import sys
import os
import time
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

print("=== Table IV Reproduction: Coded OFDM Controls at 10 dB ===\n")

TABLE4 = RESULTS / "table_iv_coded_ofdm"
TABLE4.mkdir(parents=True, exist_ok=True)

SEEDS = [13, 59, 151]
SNR = 10
BLOCKS = 1200

per_seed_frames = []

start_all = time.time()

for seed in tqdm(SEEDS, desc="Reproducing Table IV", unit="seed"):

    seed_per = TABLE4 / f"per_seed_{seed}.csv"
    seed_summary = TABLE4 / f"summary_{seed}.csv"
    seed_paired = TABLE4 / f"paired_{seed}.csv"
    seed_log = TABLE4 / f"log_seed_{seed}.txt"

    cmd = [
        sys.executable,
        "Code/run_reviewer_baselines.py",
        "--snr", str(SNR),
        "--blocks", str(BLOCKS),
        "--seeds", str(seed),
        "--rank", "4",
        "--output", str(seed_per.relative_to(ROOT)),
        "--summary-output", str(seed_summary.relative_to(ROOT)),
        "--paired-output", str(seed_paired.relative_to(ROOT)),
    ]

    t0 = time.time()

    result = subprocess.run(
        cmd,
        cwd=ROOT,
        capture_output=True,
        text=True,
        env=os.environ.copy()
    )

    elapsed = time.time() - t0

    with open(seed_log, "w") as f:
        f.write("COMMAND:\n")
        f.write(" ".join(cmd) + "\n\n")
        f.write("STDOUT:\n")
        f.write(result.stdout)
        f.write("\nSTDERR:\n")
        f.write(result.stderr)
        f.write(f"\nReturn code: {result.returncode}\n")
        f.write(f"Runtime seconds: {elapsed:.2f}\n")

    if result.returncode != 0:
        print(f"\nERROR for seed {seed}")
        print(result.stderr)
        raise RuntimeError(f"Table IV run failed for seed {seed}")

    per_seed_frames.append(pd.read_csv(seed_per))


# ------------------------------------------------------------
# Combine independently regenerated seed-level results
# ------------------------------------------------------------

per_seed = pd.concat(per_seed_frames, ignore_index=True)

combined_per_seed = TABLE4 / "coded_reproduced_per_seed.csv"
per_seed.to_csv(combined_per_seed, index=False)


# ------------------------------------------------------------
# Reproduce Kumar's three-seed t-based summary
# ------------------------------------------------------------

T_CRIT_DF2 = 4.302652729911275

summary_rows = []

for (snr, method), g in per_seed.groupby(["snr_db", "method"]):

    x = g["PER"].to_numpy(float)

    mean = x.mean()
    half = T_CRIT_DF2 * x.std(ddof=1) / np.sqrt(len(x))

    summary_rows.append({
        "snr_db": snr,
        "method": method,
        "PER_mean": mean,
        "PER_95CI_halfwidth_t_df2": half,
        "seeds": len(x),
        "blocks_total": int(g["blocks"].sum())
    })

summary = pd.DataFrame(summary_rows)


# ------------------------------------------------------------
# Paired comparisons: SQM minus each OFDM receiver
# ------------------------------------------------------------

paired_rows = []

for snr, g in per_seed.groupby("snr_db"):

    pivot = g.pivot(
        index="seed",
        columns="method",
        values="PER"
    )

    for method in pivot.columns:

        if method == "SQM_rank4":
            continue

        delta = (
            pivot["SQM_rank4"] - pivot[method]
        ).dropna().to_numpy(float)

        half = (
            T_CRIT_DF2
            * delta.std(ddof=1)
            / np.sqrt(len(delta))
        )

        paired_rows.append({
            "snr_db": snr,
            "comparison": "SQM - " + method,
            "paired_delta_mean": delta.mean(),
            "paired_95CI_halfwidth_t_df2": half,
            "seeds": len(delta)
        })

paired = pd.DataFrame(paired_rows)

combined_summary = TABLE4 / "coded_reproduced_summary.csv"
combined_paired = TABLE4 / "coded_reproduced_paired.csv"

summary.to_csv(combined_summary, index=False)
paired.to_csv(combined_paired, index=False)

elapsed_all = time.time() - start_all


print("\n=== TABLE IV REPRODUCTION COMPLETE ===")
print(f"Total runtime: {elapsed_all:.1f} seconds")

print("\n=== Per-seed Results ===")
display(per_seed)

print("\n=== Three-seed Summary ===")
display(summary)

print("\n=== Paired Comparisons ===")
display(paired)

print("\nSaved under:")
print(TABLE4)


In [ ]:
import pandas as pd
import numpy as np

print("=== Table IV Exact Reproducibility Check ===\n")

REF = REPRO / "Data_Kumar_Original"

# Kumar's released data
ref_seed = pd.read_csv(
    REF / "reviewer_baselines_per_seed.csv"
)

ref_summary = pd.read_csv(
    REF / "reviewer_baselines_summary.csv"
)

ref_paired = pd.read_csv(
    REF / "reviewer_baselines_paired.csv"
)

# Restrict released data to Table IV: 10 dB
ref_seed = ref_seed[
    np.isclose(ref_seed["snr_db"], 10.0)
].copy()

ref_summary = ref_summary[
    np.isclose(ref_summary["snr_db"], 10.0)
].copy()

ref_paired = ref_paired[
    np.isclose(ref_paired["snr_db"], 10.0)
].copy()


# Our freshly reproduced data
new_seed = pd.read_csv(
    TABLE4 / "coded_reproduced_per_seed.csv"
)

new_summary = pd.read_csv(
    TABLE4 / "coded_reproduced_summary.csv"
)

new_paired = pd.read_csv(
    TABLE4 / "coded_reproduced_paired.csv"
)


def compare_frames(name, ref, new, sort_cols):

    print(f"\n--- {name} ---")

    ref = ref.sort_values(sort_cols).reset_index(drop=True)
    new = new.sort_values(sort_cols).reset_index(drop=True)

    print("Reference shape :", ref.shape)
    print("Reproduced shape:", new.shape)

    same_columns = list(ref.columns) == list(new.columns)
    same_shape = ref.shape == new.shape

    print("Same columns:", same_columns)
    print("Same shape  :", same_shape)

    if not (same_columns and same_shape):
        return False

    numeric_cols = ref.select_dtypes(
        include=[np.number]
    ).columns

    a = ref[numeric_cols].to_numpy(dtype=float)
    b = new[numeric_cols].to_numpy(dtype=float)

    numeric_match = np.allclose(
        a,
        b,
        rtol=1e-12,
        atol=1e-12,
        equal_nan=True
    )

    max_diff = np.nanmax(np.abs(a - b))

    other_cols = [
        c for c in ref.columns
        if c not in numeric_cols
    ]

    text_match = True

    for col in other_cols:
        if not ref[col].fillna("").equals(
            new[col].fillna("")
        ):
            text_match = False

    print("Numeric match :", numeric_match)
    print("Max abs diff  :", max_diff)
    print("Text match    :", text_match)

    return (
        same_columns
        and same_shape
        and numeric_match
        and text_match
    )


ok_seed = compare_frames(
    "Per-seed results",
    ref_seed,
    new_seed,
    ["snr_db", "seed", "method"]
)

ok_summary = compare_frames(
    "Three-seed summary",
    ref_summary,
    new_summary,
    ["snr_db", "method"]
)

ok_paired = compare_frames(
    "Paired comparisons",
    ref_paired,
    new_paired,
    ["snr_db", "comparison"]
)


print("\n======================================")
print("TABLE IV REPRODUCTION VERDICT")
print("======================================")

if ok_seed and ok_summary and ok_paired:
    print(
        "SUCCESS: Table IV coded-OFDM results "
        "reproduced exactly."
    )
else:
    print(
        "CHECK: One or more released Table IV "
        "results differ."
    )


In [ ]:
import json
from pathlib import Path

print("=== Reconstruct Missing Code/config.json ===\n")

config_path = ROOT / "Code" / "config.json"
backup_copy = REPRO / "reconstructed_config.json"

config = {
    "method_name": "Selective SQM",
    "messages": 256,
    "block_length": 16,
    "nfft": 12,
    "cp_length": 4,
    "pilot_bins": [0, 3, 6, 9],
    "data_bins": [1, 2, 4, 5, 7, 8, 10, 11],
    "physical_cfo_support_deg_per_sample": [-4, 4],
    "cfo_search_grid_deg_per_sample": [-5, 5, 1]
}

if config_path.exists():
    print("config.json already exists:")
    print(config_path)
else:
    config_path.write_text(
        json.dumps(config, indent=2),
        encoding="utf-8"
    )
    print("Created missing config.json:")
    print(config_path)

# Keep an explicit copy in Reproduction_Work for provenance
backup_copy.write_text(
    json.dumps(config, indent=2),
    encoding="utf-8"
)

print("\nProvenance copy saved to:")
print(backup_copy)

print("\n=== Config contents ===")
print(json.dumps(config, indent=2))

print("\nStatus: READY")


In [ ]:
import subprocess
import sys

print("=== Verify Reconstructed experiment_config ===\n")

result = subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import sys; "
            "sys.path.insert(0, 'Code'); "
            "import experiment_config as c; "
            "print('METHOD_NAME =', c.METHOD_NAME); "
            "print('M =', c.M); "
            "print('L =', c.L); "
            "print('NFFT =', c.NFFT); "
            "print('CP =', c.CP); "
            "print('PILOT_BINS =', c.PILOT_BINS.tolist()); "
            "print('DATA_BINS =', c.DATA_BINS.tolist()); "
            "print('Physical CFO =', c.PHYSICAL_CFO_MIN, c.PHYSICAL_CFO_MAX); "
            "print('Decoder grid =', c.GRID_MIN, c.GRID_MAX, c.GRID_STEP)"
        )
    ],
    cwd=ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

print("Return code:", result.returncode)

if result.returncode == 0:
    print("\nOverall status: PASS")
else:
    print("\nOverall status: FAIL")


In [ ]:
import subprocess
import sys
import os
import time
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

print("=== Figure 4 Reproduction: Delay and PDP Robustness ===\n")

FIG4 = RESULTS / "figure4_robustness"
FIG4.mkdir(parents=True, exist_ok=True)

experiments = [
    {
        "name": "Delay-spread sweep",
        "mode": "delay",
        "output": FIG4 / "delay_spread_reproduced.csv",
    },
    {
        "name": "PDP mismatch sweep",
        "mode": "pdp",
        "output": FIG4 / "pdp_mismatch_reproduced.csv",
    },
]

for exp in tqdm(experiments, desc="Reproducing Figure 4", unit="experiment"):

    log_file = FIG4 / f"{exp['mode']}_run.log"

    cmd = [
        sys.executable,
        "Code/run_delay_pdp_sweep.py",
        "--mode", exp["mode"],
        "--snr", "10",
        "--blocks", "200",
        "--seeds", "13,59",
        "--rank", "4",
        "--output", str(exp["output"].relative_to(ROOT)),
    ]

    print(f"\nRunning: {exp['name']}")
    print(" ".join(cmd))

    start = time.time()

    result = subprocess.run(
        cmd,
        cwd=ROOT,
        capture_output=True,
        text=True,
        env=os.environ.copy()
    )

    elapsed = time.time() - start

    with open(log_file, "w") as f:
        f.write("COMMAND:\n")
        f.write(" ".join(cmd) + "\n\n")
        f.write("STDOUT:\n")
        f.write(result.stdout)
        f.write("\nSTDERR:\n")
        f.write(result.stderr)
        f.write(f"\nReturn code: {result.returncode}\n")
        f.write(f"Runtime seconds: {elapsed:.2f}\n")

    if result.returncode != 0:
        print(result.stderr)
        raise RuntimeError(f"{exp['name']} failed")

    print(f"Completed in {elapsed:.1f} s")

delay = pd.read_csv(
    FIG4 / "delay_spread_reproduced.csv"
)

pdp = pd.read_csv(
    FIG4 / "pdp_mismatch_reproduced.csv"
)

delay_summary = (
    delay
    .groupby(["delay_spread_ns", "method"], as_index=False)["PER"]
    .mean()
)

pdp_summary = (
    pdp
    .groupby(["pdp_power_exponent", "method"], as_index=False)["PER"]
    .mean()
)

print("\n=== Delay-spread sweep ===")
display(delay_summary)

print("\n=== PDP mismatch sweep ===")
display(pdp_summary)

print("\nFiles saved in:")
print(FIG4)


In [ ]:
import pandas as pd
import numpy as np

print("=== Figure 4 Exact Reproducibility Check ===\n")

REF = REPRO / "Data_Kumar_Original"

checks = [
    (
        "Delay-spread sweep",
        REF / "delay_spread_sweep.csv",
        FIG4 / "delay_spread_reproduced.csv",
        ["delay_spread_ns", "seed", "method"],
    ),
    (
        "PDP mismatch sweep",
        REF / "pdp_mismatch_sweep.csv",
        FIG4 / "pdp_mismatch_reproduced.csv",
        ["pdp_power_exponent", "seed", "method"],
    ),
]

overall_pass = True

for name, ref_file, new_file, sort_cols in checks:

    print(f"\n--- {name} ---")

    ref = pd.read_csv(ref_file)
    new = pd.read_csv(new_file)

    ref = ref.sort_values(sort_cols).reset_index(drop=True)
    new = new.sort_values(sort_cols).reset_index(drop=True)

    print("Reference shape :", ref.shape)
    print("Reproduced shape:", new.shape)

    same_columns = list(ref.columns) == list(new.columns)
    same_shape = ref.shape == new.shape

    print("Same columns:", same_columns)
    print("Same shape  :", same_shape)

    if not (same_columns and same_shape):
        overall_pass = False
        continue

    numeric_cols = ref.select_dtypes(
        include=[np.number]
    ).columns

    a = ref[numeric_cols].to_numpy(dtype=float)
    b = new[numeric_cols].to_numpy(dtype=float)

    numeric_match = np.allclose(
        a,
        b,
        rtol=1e-12,
        atol=1e-12,
        equal_nan=True
    )

    max_diff = np.nanmax(np.abs(a - b))

    text_cols = [
        c for c in ref.columns
        if c not in numeric_cols
    ]

    text_match = all(
        ref[c].fillna("").equals(
            new[c].fillna("")
        )
        for c in text_cols
    )

    print("Numeric match :", numeric_match)
    print("Max abs diff  :", max_diff)
    print("Text match    :", text_match)

    if not (
        same_columns
        and same_shape
        and numeric_match
        and text_match
    ):
        overall_pass = False


print("\n======================================")
print("FIGURE 4 REPRODUCTION VERDICT")
print("======================================")

if overall_pass:
    print(
        "SUCCESS: Figure 4 delay/PDP robustness "
        "results reproduced exactly."
    )
else:
    print(
        "CHECK: One or more Figure 4 results differ."
    )


In [ ]:
import subprocess
import sys
import os
import time
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

print("=== Figure 5 Reproduction: Selective-Margin Radius Test ===\n")

FIG5 = RESULTS / "figure5_margin_radius"
FIG5.mkdir(parents=True, exist_ok=True)

output_file = FIG5 / "theorem3_validation_reproduced.csv"
log_file = FIG5 / "theorem3_validation.log"

cmd = [
    sys.executable,
    "Code/evaluate_theorem3.py",
    "--cases", "30",
    "--noise-draws", "20",
    "--ranks", "4,6",
    "--fractions", "0.5,1.0,1.5,2.0",
    "--seed", "20260929",
    "--output", str(output_file.relative_to(ROOT)),
]

print("Running Kumar's original script:")
print(" ".join(cmd))
print()

start = time.time()

with open(log_file, "w") as log:

    process = subprocess.Popen(
        cmd,
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
        env=os.environ.copy()
    )

    # Activity indicator only — elapsed time, not percentage complete
    with tqdm(
        desc="Figure 5 experiment running",
        unit="s",
        bar_format="{desc}: {n:.0f} s elapsed"
    ) as pbar:

        while process.poll() is None:
            time.sleep(1)
            pbar.update(1)

return_code = process.returncode
elapsed = time.time() - start

print("\n=== Figure 5 Run Finished ===")
print("Return code :", return_code)
print(f"Runtime     : {elapsed:.1f} seconds")
print("Output      :", output_file)
print("Log         :", log_file)

if return_code != 0:
    print("\n=== Last lines of log ===")
    print(log_file.read_text()[-4000:])
    raise RuntimeError("Figure 5 experiment failed.")

df5 = pd.read_csv(output_file)

summary5 = (
    df5
    .groupby(
        ["rank", "noise_fraction_of_certified_radius"],
        as_index=False
    )
    .agg(
        empirical_error_rate=("empirical_error_rate", "mean"),
        cases=("case", "nunique"),
        noise_draws=("noise_draws", "first")
    )
)

print("\n=== Reproduced Figure 5 Results ===")
display(summary5)

print("\nRows generated:", len(df5))

print("\nOverall status: PASS")


In [ ]:
import pandas as pd
import numpy as np

print("=== Figure 5 Exact Reproducibility Check ===\n")

ref_file = REPRO / "Data_Kumar_Original" / "theorem3_validation.csv"
new_file = FIG5 / "theorem3_validation_reproduced.csv"

ref = pd.read_csv(ref_file)
new = pd.read_csv(new_file)

print("Reference shape :", ref.shape)
print("Reproduced shape:", new.shape)

same_columns = list(ref.columns) == list(new.columns)
same_shape = ref.shape == new.shape

print("Same columns:", same_columns)
print("Same shape  :", same_shape)

# Sort identically before comparison
sort_cols = [
    c for c in [
        "case",
        "rank",
        "noise_fraction_of_certified_radius"
    ]
    if c in ref.columns
]

if sort_cols:
    ref = ref.sort_values(sort_cols).reset_index(drop=True)
    new = new.sort_values(sort_cols).reset_index(drop=True)

numeric_cols = ref.select_dtypes(include=[np.number]).columns

a = ref[numeric_cols].to_numpy(dtype=float)
b = new[numeric_cols].to_numpy(dtype=float)

numeric_match = np.allclose(
    a,
    b,
    rtol=1e-12,
    atol=1e-12,
    equal_nan=True
)

max_diff = np.nanmax(np.abs(a - b))

text_cols = [
    c for c in ref.columns
    if c not in numeric_cols
]

text_match = all(
    ref[c].fillna("").equals(new[c].fillna(""))
    for c in text_cols
)

print("Numeric match :", numeric_match)
print("Max abs diff  :", max_diff)
print("Text match    :", text_match)

print("\n======================================")
print("FIGURE 5 REPRODUCTION VERDICT")
print("======================================")

if (
    same_columns
    and same_shape
    and numeric_match
    and text_match
):
    print(
        "SUCCESS: Figure 5 selective-margin "
        "radius results reproduced exactly."
    )
else:
    print("CHECK: Figure 5 results differ.")


In [ ]:
import subprocess
import sys
import os
import time
import pandas as pd
from pathlib import Path

print("=== Figure 6 Reproduction: Payload Scaling ===\n")

FIG6 = RESULTS / "figure6_scaling"
FIG6.mkdir(parents=True, exist_ok=True)

output_file = FIG6 / "scaling_10dB_reproduced.csv"
log_file = FIG6 / "scaling_run.log"

cmd = [
    sys.executable,
    "Code/evaluate_scaling.py",
    "--bits", "6,8,10",
    "--snr", "10",
    "--blocks", "200",
    "--seed", "20260929",
    "--output", str(output_file.relative_to(ROOT)),
]

print("Running Kumar's original script:")
print(" ".join(cmd))

start = time.time()

result = subprocess.run(
    cmd,
    cwd=ROOT,
    capture_output=True,
    text=True,
    env=os.environ.copy()
)

elapsed = time.time() - start

with open(log_file, "w") as f:
    f.write("COMMAND:\n")
    f.write(" ".join(cmd) + "\n\n")
    f.write("STDOUT:\n")
    f.write(result.stdout)
    f.write("\nSTDERR:\n")
    f.write(result.stderr)
    f.write(f"\nReturn code: {result.returncode}\n")
    f.write(f"Runtime seconds: {elapsed:.2f}\n")

print("\n=== Figure 6 Run Finished ===")
print("Return code :", result.returncode)
print(f"Runtime     : {elapsed:.1f} seconds")

if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("Figure 6 experiment failed.")

df6 = pd.read_csv(output_file)

print("\n=== Reproduced Figure 6 Results ===")
display(df6)

print("\nOverall status: PASS")


In [ ]:
import pandas as pd
import numpy as np

print("=== Figure 6 Exact Reproducibility Check ===\n")

ref_file = (
    REPRO
    / "Data_Kumar_Original"
    / "scaling_10dB.csv"
)

new_file = (
    FIG6
    / "scaling_10dB_reproduced.csv"
)

ref = pd.read_csv(ref_file)
new = pd.read_csv(new_file)

print("Reference shape :", ref.shape)
print("Reproduced shape:", new.shape)

same_columns = list(ref.columns) == list(new.columns)
same_shape = ref.shape == new.shape

print("Same columns:", same_columns)
print("Same shape  :", same_shape)

# Sort identically
sort_cols = [
    c for c in ["bits", "messages", "snr_db", "rank"]
    if c in ref.columns
]

if sort_cols:
    ref = ref.sort_values(sort_cols).reset_index(drop=True)
    new = new.sort_values(sort_cols).reset_index(drop=True)

numeric_cols = ref.select_dtypes(
    include=[np.number]
).columns

a = ref[numeric_cols].to_numpy(dtype=float)
b = new[numeric_cols].to_numpy(dtype=float)

numeric_match = np.allclose(
    a,
    b,
    rtol=1e-12,
    atol=1e-12,
    equal_nan=True
)

max_diff = np.nanmax(
    np.abs(a - b)
)

text_cols = [
    c for c in ref.columns
    if c not in numeric_cols
]

text_match = all(
    ref[c].fillna("").equals(
        new[c].fillna("")
    )
    for c in text_cols
)

print("Numeric match :", numeric_match)
print("Max abs diff  :", max_diff)
print("Text match    :", text_match)

print("\n======================================")
print("FIGURE 6 REPRODUCTION VERDICT")
print("======================================")

if (
    same_columns
    and same_shape
    and numeric_match
    and text_match
):
    print(
        "SUCCESS: Figure 6 payload-scaling "
        "results reproduced exactly."
    )
else:
    print("CHECK: Figure 6 results differ.")


In [ ]:
import subprocess
import sys
import os
import time
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

print("=== Table V / Figure 7 Reproduction: BMOCZ Comparison ===\n")

FIG7 = RESULTS / "figure7_bmocz"
FIG7.mkdir(parents=True, exist_ok=True)

output_file = FIG7 / "dmm_vs_bmocz_reproduced.csv"
per_seed_file = FIG7 / "dmm_vs_bmocz_reproduced_per_seed.csv"
log_file = FIG7 / "bmocz_run.log"

cmd = [
    sys.executable,
    "Code/evaluate_bmocz_baseline.py",
    "--blocks", "800",
    "--out", str(output_file.relative_to(ROOT)),
]

print("Running Kumar's original script:")
print(" ".join(cmd))
print()
print("Script settings:")
print("  Seeds  : 17, 83, 173")
print("  SNRs   : 5, 7, 10, 12, 15 dB")
print("  CFO    : 0 and ±8 degrees/sample")
print("  Blocks : 800 per seed/SNR")

start = time.time()

with open(log_file, "w") as log:

    process = subprocess.Popen(
        cmd,
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
        env=os.environ.copy()
    )

    with tqdm(
        desc="BMOCZ experiment running",
        unit="s",
        bar_format="{desc}: {n:.0f} s elapsed"
    ) as pbar:

        while process.poll() is None:
            time.sleep(1)
            pbar.update(1)

return_code = process.returncode
elapsed = time.time() - start

print("\n=== BMOCZ Run Finished ===")
print("Return code :", return_code)
print(f"Runtime     : {elapsed:.1f} seconds")
print("Summary     :", output_file)
print("Per-seed    :", per_seed_file)
print("Log         :", log_file)

if return_code != 0:
    print("\n=== Last lines of log ===")
    print(log_file.read_text()[-5000:])
    raise RuntimeError("BMOCZ experiment failed.")

df7 = pd.read_csv(output_file)
ps7 = pd.read_csv(per_seed_file)

# Table V / Figure 7 uses ±8 degrees/sample CFO
table_v = (
    df7[
        df7["cfo_range_deg_per_sample"] == 8.0
    ]
    .sort_values(["snr_db", "method"])
    .reset_index(drop=True)
)

print("\n=== Reproduced Table V: CFO ±8 deg/sample ===")
display(table_v)

print("\n=== Reproduced per-seed results: CFO ±8 deg/sample ===")
display(
    ps7[
        ps7["cfo_range_deg_per_sample"] == 8.0
    ].sort_values(["snr_db", "seed", "method"])
)

print("\nOverall status: PASS")


In [ ]:
import pandas as pd
import numpy as np

print("=== Table V / Figure 7 Exact Reproducibility Check ===\n")

REF = REPRO / "Data_Kumar_Original"

checks = [
    (
        "BMOCZ summary",
        REF / "dmm_vs_bmocz_rerun.csv",
        FIG7 / "dmm_vs_bmocz_reproduced.csv",
        ["cfo_range_deg_per_sample", "snr_db", "method"],
    ),
    (
        "BMOCZ per-seed",
        REF / "dmm_vs_bmocz_rerun_per_seed.csv",
        FIG7 / "dmm_vs_bmocz_reproduced_per_seed.csv",
        ["cfo_range_deg_per_sample", "snr_db", "seed", "method"],
    ),
]

overall_pass = True

for name, ref_file, new_file, sort_cols in checks:

    print(f"\n--- {name} ---")

    ref = pd.read_csv(ref_file)
    new = pd.read_csv(new_file)

    ref = ref.sort_values(sort_cols).reset_index(drop=True)
    new = new.sort_values(sort_cols).reset_index(drop=True)

    print("Reference shape :", ref.shape)
    print("Reproduced shape:", new.shape)

    same_columns = list(ref.columns) == list(new.columns)
    same_shape = ref.shape == new.shape

    print("Same columns:", same_columns)
    print("Same shape  :", same_shape)

    if not (same_columns and same_shape):
        overall_pass = False
        continue

    numeric_cols = ref.select_dtypes(include=[np.number]).columns

    a = ref[numeric_cols].to_numpy(dtype=float)
    b = new[numeric_cols].to_numpy(dtype=float)

    numeric_match = np.allclose(
        a,
        b,
        rtol=1e-12,
        atol=1e-12,
        equal_nan=True
    )

    max_diff = np.nanmax(np.abs(a - b))

    text_cols = [
        c for c in ref.columns
        if c not in numeric_cols
    ]

    text_match = all(
        ref[c].fillna("").equals(
            new[c].fillna("")
        )
        for c in text_cols
    )

    print("Numeric match :", numeric_match)
    print("Max abs diff  :", max_diff)
    print("Text match    :", text_match)

    if not (
        same_columns
        and same_shape
        and numeric_match
        and text_match
    ):
        overall_pass = False


print("\n======================================")
print("TABLE V / FIGURE 7 REPRODUCTION VERDICT")
print("======================================")

if overall_pass:
    print(
        "SUCCESS: BMOCZ comparison results "
        "reproduced exactly."
    )
else:
    print(
        "CHECK: One or more BMOCZ results differ."
    )


In [ ]:
# Cell 22 — inspect released Figure 8 data

import pandas as pd

print("=== Figure 8: Released Observable-Rank Data ===\n")

spectrum_file = ROOT / "Data" / "mean_singular_spectrum.csv"
rank_file = ROOT / "Data" / "effective_rank_summary.csv"

spectrum = pd.read_csv(spectrum_file)
effective_rank = pd.read_csv(rank_file)

print("=== Mean singular-energy spectrum ===")
display(spectrum)

print("\n=== Effective-rank summary ===")
display(effective_rank)

print("\nFirst two modes contain:")
print(
    f"{100 * spectrum.loc[:1, 'mean_energy_fraction'].sum():.3f}% "
    "of the mean weighted energy"
)


In [ ]:
import sys
import numpy as np
import pandas as pd

print("=== Figure 8 Independent Regeneration ===\n")

# Allow direct import of Kumar's code
CODE = ROOT / "Code"

if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))

from fractional_tdla_core import (
    tdla_operators,
    weighted_nuisance_dictionary,
)

FIG8 = RESULTS / "figure8_observable_rank"
FIG8.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Load Kumar's shipped DMM/SQM codebook
# ------------------------------------------------------------

artifact = np.load(
    ROOT / "Artifact" / "stage3_codebooks.npz"
)

X = artifact["X_dmm"]

print("Codebook shape:", X.shape)

# Convert real I/Q representation to complex waveforms
Xd = X[:, 0, :] + 1j * X[:, 1, :]

L = Xd.shape[1]

# Exact 30-ns TDL-A operators used by Kumar
_, powers, D = tdla_operators(L)

# ------------------------------------------------------------
# Singular-energy spectrum for every message
# ------------------------------------------------------------

all_energy = []

for x in Xd:

    F = weighted_nuisance_dictionary(
        x=x,
        omega=0.0,
        powers=powers,
        D=D
    )

    singular_values = np.linalg.svd(
        F,
        compute_uv=False
    )

    energy = singular_values**2
    energy = energy / energy.sum()

    all_energy.append(energy)

all_energy = np.asarray(all_energy)

mean_energy = all_energy.mean(axis=0)
mean_cumulative = np.cumsum(mean_energy)

spectrum_reproduced = pd.DataFrame({
    "mode": np.arange(1, len(mean_energy) + 1),
    "mean_energy_fraction": mean_energy,
    "mean_cumulative_energy": mean_cumulative,
})

# ------------------------------------------------------------
# Effective rank statistics
# ------------------------------------------------------------

thresholds = [0.90, 0.95, 0.99, 0.999]

rank_rows = []

cumulative_per_message = np.cumsum(
    all_energy,
    axis=1
)

for threshold in thresholds:

    ranks = (
        np.argmax(
            cumulative_per_message >= threshold,
            axis=1
        )
        + 1
    )

    rank_rows.append({
        "energy_fraction": threshold,
        "median_rank": float(np.median(ranks)),
        "p90_rank": float(np.percentile(ranks, 90)),
        "max_rank": int(np.max(ranks)),
    })

rank_reproduced = pd.DataFrame(rank_rows)

# ------------------------------------------------------------
# Save our independently regenerated outputs
# ------------------------------------------------------------

spectrum_out = (
    FIG8 / "mean_singular_spectrum_reproduced.csv"
)

rank_out = (
    FIG8 / "effective_rank_summary_reproduced.csv"
)

spectrum_reproduced.to_csv(
    spectrum_out,
    index=False
)

rank_reproduced.to_csv(
    rank_out,
    index=False
)

print("\n=== Reproduced singular-energy spectrum ===")
display(spectrum_reproduced)

print("\n=== Reproduced effective-rank summary ===")
display(rank_reproduced)

print(
    "\nFirst two modes contain:",
    f"{100 * mean_energy[:2].sum():.3f}%"
)

print("\nSaved:")
print(spectrum_out)
print(rank_out)


In [ ]:
# Cell 25 — Figure 8 exact reproducibility check

import pandas as pd
import numpy as np

print("=== Figure 8 Exact Reproducibility Check ===\n")

checks = [
    (
        "Mean singular spectrum",
        ROOT / "Data" / "mean_singular_spectrum.csv",
        FIG8 / "mean_singular_spectrum_reproduced.csv",
        ["mode"],
    ),
    (
        "Effective-rank summary",
        ROOT / "Data" / "effective_rank_summary.csv",
        FIG8 / "effective_rank_summary_reproduced.csv",
        ["energy_fraction"],
    ),
]

overall_pass = True

for name, ref_file, new_file, sort_cols in checks:

    print(f"\n--- {name} ---")

    ref = pd.read_csv(ref_file)
    new = pd.read_csv(new_file)

    ref = ref.sort_values(sort_cols).reset_index(drop=True)
    new = new.sort_values(sort_cols).reset_index(drop=True)

    print("Reference shape :", ref.shape)
    print("Reproduced shape:", new.shape)

    same_columns = list(ref.columns) == list(new.columns)
    same_shape = ref.shape == new.shape

    print("Same columns:", same_columns)
    print("Same shape  :", same_shape)

    if not (same_columns and same_shape):
        overall_pass = False
        continue

    numeric_cols = ref.select_dtypes(include=[np.number]).columns

    a = ref[numeric_cols].to_numpy(dtype=float)
    b = new[numeric_cols].to_numpy(dtype=float)

    numeric_match = np.allclose(
        a,
        b,
        rtol=1e-12,
        atol=1e-12,
        equal_nan=True
    )

    max_diff = np.nanmax(np.abs(a - b))

    text_cols = [
        c for c in ref.columns
        if c not in numeric_cols
    ]

    text_match = all(
        ref[c].fillna("").equals(
            new[c].fillna("")
        )
        for c in text_cols
    )

    print("Numeric match :", numeric_match)
    print("Max abs diff  :", max_diff)
    print("Text match    :", text_match)

    if not (
        same_columns
        and same_shape
        and numeric_match
        and text_match
    ):
        overall_pass = False


print("\n======================================")
print("FIGURE 8 REPRODUCTION VERDICT")
print("======================================")

if overall_pass:
    print(
        "SUCCESS: Figure 8 observable-rank "
        "results reproduced exactly."
    )
else:
    print("CHECK: Figure 8 results differ.")


In [ ]:
import subprocess
import sys
import os
import time
import pandas as pd
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

print("=== Figure 10 Reproduction: I/Q Condition-Number Control ===\n")

FIG10 = RESULTS / "figure10_iq_condition"
FIG10.mkdir(parents=True, exist_ok=True)

output_file = FIG10 / "operating_iq_sweep_reproduced.csv"
log_file = FIG10 / "iq_condition_run.log"

cmd = [
    sys.executable,
    "Code/run_iq_correction.py",
    "--blocks", "1000",
    "--seeds", "13,59,151",
    "--output", str(output_file.relative_to(ROOT)),
]

print("Running Kumar's original script:")
print(" ".join(cmd))
print()

start = time.time()

with open(log_file, "w") as log:

    process = subprocess.Popen(
        cmd,
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
        env=os.environ.copy()
    )

    with tqdm(
        desc="Figure 10 experiment running",
        unit="s",
        bar_format="{desc}: {n:.0f} s elapsed"
    ) as pbar:

        while process.poll() is None:
            time.sleep(1)
            pbar.update(1)

return_code = process.returncode
elapsed = time.time() - start

print("\n=== Figure 10 Run Finished ===")
print("Return code :", return_code)
print(f"Runtime     : {elapsed:.1f} seconds")
print("Output      :", output_file)
print("Log         :", log_file)

if return_code != 0:
    print(log_file.read_text()[-5000:])
    raise RuntimeError("Figure 10 experiment failed.")

df10 = pd.read_csv(output_file)

print("\n=== Raw reproduced results ===")
display(df10)

# Publication-style summary
T_CRIT_DF2 = 4.302652729911275

summary_rows = []

for (kappa, receiver), g in df10.groupby(
    ["condition_number", "receiver"]
):

    x = g["PER"].to_numpy(float)

    summary_rows.append({
        "condition_number": kappa,
        "receiver": receiver,
        "PER_mean": x.mean(),
        "PER_95CI_halfwidth":
            T_CRIT_DF2 * x.std(ddof=1) / np.sqrt(len(x)),
        "seeds": len(x)
    })

summary10 = pd.DataFrame(summary_rows)

print("\n=== Three-seed summary ===")
display(summary10)

print("\nOverall status: PASS")


In [ ]:
import subprocess
import sys
import os
import time
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

print("=== Figure 10 Reproduction: I/Q Condition-Number Control ===\n")

FIG10 = RESULTS / "figure10_iq_condition"
FIG10.mkdir(parents=True, exist_ok=True)

output_file = FIG10 / "operating_iq_sweep_reproduced.csv"
log_file = FIG10 / "iq_condition_run.log"

cmd = [
    sys.executable,
    "Code/run_iq_correction.py",
    "--blocks", "1200",
    "--seeds", "13,59,151",
    "--output", str(output_file.relative_to(ROOT)),
]

print("Running Kumar's original script:")
print(" ".join(cmd))

start = time.time()

with open(log_file, "w") as log:
    process = subprocess.Popen(
        cmd,
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
        env=os.environ.copy()
    )

    with tqdm(
        desc="Figure 10 experiment running",
        unit="s",
        bar_format="{desc}: {n:.0f} s elapsed"
    ) as pbar:
        while process.poll() is None:
            time.sleep(1)
            pbar.update(1)

return_code = process.returncode
elapsed = time.time() - start

print("\n=== Figure 10 Run Finished ===")
print("Return code :", return_code)
print(f"Runtime     : {elapsed:.1f} seconds")

if return_code != 0:
    print(log_file.read_text()[-5000:])
    raise RuntimeError("Figure 10 experiment failed.")

df10 = pd.read_csv(output_file)

T_CRIT_DF2 = 4.302652729911275

summary_rows = []

for (kappa, receiver), g in df10.groupby(
    ["condition_number", "receiver"]
):
    x = g["PER"].to_numpy(float)

    summary_rows.append({
        "condition_number": kappa,
        "receiver": receiver,
        "PER_mean": x.mean(),
        "PER_95CI_halfwidth":
            T_CRIT_DF2 * x.std(ddof=1) / np.sqrt(len(x)),
        "seeds": len(x)
    })

summary10 = pd.DataFrame(summary_rows)

print("\n=== Reproduced Figure 10 Results ===")
display(df10)

print("\n=== Three-seed summary ===")
display(summary10)

print("\nOverall status: PASS")


In [ ]:
import pandas as pd
import numpy as np

print("=== Figure 10 Exact Reproducibility Check ===\n")

ref_file = (
    REPRO
    / "Data_Kumar_Original"
    / "operating_iq_sweep_covaware.csv"
)

new_file = (
    FIG10
    / "operating_iq_sweep_reproduced.csv"
)

ref = pd.read_csv(ref_file)
new = pd.read_csv(new_file)

print("Reference shape :", ref.shape)
print("Reproduced shape:", new.shape)

same_columns = list(ref.columns) == list(new.columns)
same_shape = ref.shape == new.shape

print("Same columns:", same_columns)
print("Same shape  :", same_shape)

sort_cols = [
    c for c in [
        "condition_number",
        "seed",
        "receiver"
    ]
    if c in ref.columns
]

ref = ref.sort_values(sort_cols).reset_index(drop=True)
new = new.sort_values(sort_cols).reset_index(drop=True)

numeric_cols = ref.select_dtypes(
    include=[np.number]
).columns

a = ref[numeric_cols].to_numpy(dtype=float)
b = new[numeric_cols].to_numpy(dtype=float)

numeric_match = np.allclose(
    a,
    b,
    rtol=1e-12,
    atol=1e-12,
    equal_nan=True
)

max_diff = np.nanmax(np.abs(a - b))

text_cols = [
    c for c in ref.columns
    if c not in numeric_cols
]

text_match = all(
    ref[c].fillna("").equals(
        new[c].fillna("")
    )
    for c in text_cols
)

print("Numeric match :", numeric_match)
print("Max abs diff  :", max_diff)
print("Text match    :", text_match)

print("\n======================================")
print("FIGURE 10 REPRODUCTION VERDICT")
print("======================================")

if (
    same_columns
    and same_shape
    and numeric_match
    and text_match
):
    print(
        "SUCCESS: Figure 10 I/Q condition-number "
        "results reproduced exactly."
    )
else:
    print("CHECK: Figure 10 results differ.")


In [ ]:
# Cell 29 — Figure 11 reproducibility status

import pandas as pd

print("=== Figure 11: CFO-Range Sensitivity ===\n")

cfo_file = ROOT / "Data" / "operating_cfo_sweep.csv"
cfo = pd.read_csv(cfo_file)

display(cfo)

print("\nReproducibility status:")
print(
    "Released numerical data are available and can reproduce the figure, "
    "but the downloaded repository does not contain the simulation script "
    "that generated operating_cfo_sweep.csv."
)

print("\nTherefore:")
print("Figure reconstruction from released CSV : YES")
print("Independent regeneration from original experiment code : NOT VERIFIED")


In [ ]:
# Cell 30 — Figure 9: retained nuisance rank vs SNR

import pandas as pd

print("=== Figure 9: PER versus Retained Nuisance Rank ===\n")

rank_file = ROOT / "Data" / "rank_vs_snr.csv"

rank_snr = pd.read_csv(rank_file)

print("=== Kumar's released Figure 9 data ===")
display(rank_snr)

print("\n=== Best retained rank at each SNR ===")

best_rank = (
    rank_snr
    .loc[
        rank_snr.groupby("snr_db")["PER_mean"].idxmin(),
        ["snr_db", "rank", "PER_mean"]
    ]
    .sort_values("snr_db")
    .reset_index(drop=True)
)

display(best_rank)

print("\nReproducibility status:")
print("Released numerical data for Figure 9             : AVAILABLE")
print("Figure reconstruction from released CSV          : YES")
print("Current rank-sweep evaluator available            : YES")
print("Exact command/seeds for published rank_vs_snr.csv : NOT DOCUMENTED")
print("Exact independent regeneration                    : NOT VERIFIED")


In [ ]:
# Cell 31 — Figure 12: Coarse-to-fine decoder complexity

import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

print("=== Figure 12: Coarse-to-Fine Decoder Complexity ===\n")

FIG12 = RESULTS / "figure12_complexity"
FIG12.mkdir(parents=True, exist_ok=True)

# Use our untouched snapshot of Kumar's released data
complexity_file = (
    REPRO
    / "Data_Kumar_Original"
    / "fractional_decoder_complexity.csv"
)

df12 = pd.read_csv(complexity_file)

print("=== Kumar's released Figure 12 data ===")
display(df12)

# ------------------------------------------------------------
# Basic consistency checks
# ------------------------------------------------------------

same_per = df12["PER_mean"].nunique() == 1

print("\nConsistency checks:")
print("All decoder variants have same reported PER :", same_per)

full_scores = float(
    df12.loc[
        df12["decoder"] == "full search",
        "candidate_CFO_scores"
    ].iloc[0]
)

print("Full-search score evaluations              :", int(full_scores))

for _, row in df12.iterrows():
    print(
        f"{row['decoder']:25s} "
        f"scores={int(row['candidate_CFO_scores']):4d}  "
        f"relative={row['relative_score_count']:.4f}  "
        f"PER={row['PER_mean']:.6f}  "
        f"disagreement={row['disagreement_with_full']:.6f}"
    )

# ------------------------------------------------------------
# Reconstruct Kumar's Figure 12
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(4.7, 2.8))

labels = {
    "coarse-to-fine top-4": "top-4 shortlist",
    "coarse-to-fine top-8": "top-8 shortlist",
    "coarse-to-fine top-16": "top-16 shortlist",
    "full search": "full search",
}

for _, row in df12.iterrows():

    ax.errorbar(
        row["relative_score_count"],
        row["PER_mean"],
        yerr=row["PER_95CI_halfwidth"],
        fmt="o",
        capsize=3,
        label=labels[row["decoder"]]
    )

ax.set_xlabel("Relative candidate-CFO score count")
ax.set_ylabel("PER at 10 dB")
ax.grid(True, alpha=0.25)
ax.legend(frameon=True, fontsize=7, loc="lower right")

fig.tight_layout()

fig_file = FIG12 / "figure12_complexity_reconstructed.png"
fig.savefig(fig_file, dpi=300, bbox_inches="tight")

plt.show()

print("\nSaved reconstructed figure:")
print(fig_file)

print("\nReproducibility status:")
print("Released numerical data                      : AVAILABLE")
print("Figure reconstruction from released CSV       : YES")
print("Generating simulation script in repository     : NO")
print("Independent regeneration of underlying results : NOT VERIFIED")


In [ ]:
# Cell 32 — Grass-Lattice baseline: independent reproduction + exact check

import subprocess
import sys
import os
import time
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

print("=== Grass-Lattice Baseline Reproduction ===\n")

GRASS = RESULTS / "grass_lattice"
GRASS.mkdir(parents=True, exist_ok=True)

output_file = GRASS / "dmm_vs_grasslattice_flat_reproduced.csv"
log_file = GRASS / "grass_lattice_run.log"

cmd = [
    sys.executable,
    "Code/evaluate_grasslattice_baseline.py",
    "--blocks", "1800",
    "--out", str(output_file.relative_to(ROOT)),
]

print("Running Kumar's original script:")
print(" ".join(cmd))
print()
print("Internal script settings:")
print("  Seeds  : 17, 73, 157, 229")
print("  SNRs   : 5, 7, 10, 12, 15 dB")
print("  Blocks : 1800 per seed/SNR")
print()

start = time.time()

with open(log_file, "w") as log:

    process = subprocess.Popen(
        cmd,
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
        env=os.environ.copy()
    )

    with tqdm(
        desc="Grass-Lattice experiment running",
        unit="s",
        bar_format="{desc}: {n:.0f} s elapsed"
    ) as pbar:

        while process.poll() is None:
            time.sleep(1)
            pbar.update(1)

return_code = process.returncode
elapsed = time.time() - start

print("\n=== Grass-Lattice Run Finished ===")
print("Return code :", return_code)
print(f"Runtime     : {elapsed:.1f} seconds")
print("Output      :", output_file)
print("Log         :", log_file)

if return_code != 0:
    print("\n=== Last lines of log ===")
    print(log_file.read_text()[-5000:])
    raise RuntimeError("Grass-Lattice experiment failed.")

# ------------------------------------------------------------
# Show regenerated data
# ------------------------------------------------------------

new = pd.read_csv(output_file)

print("\n=== Reproduced Grass-Lattice Results ===")
display(new)

# ------------------------------------------------------------
# Exact comparison with Kumar's preserved release
# ------------------------------------------------------------

ref_file = (
    REPRO
    / "Data_Kumar_Original"
    / "dmm_vs_grasslattice_flat.csv"
)

ref = pd.read_csv(ref_file)

sort_cols = ["snr_db", "method"]

ref = ref.sort_values(sort_cols).reset_index(drop=True)
new = new.sort_values(sort_cols).reset_index(drop=True)

print("\n=== Exact Reproducibility Check ===")

print("Reference shape :", ref.shape)
print("Reproduced shape:", new.shape)

same_columns = list(ref.columns) == list(new.columns)
same_shape = ref.shape == new.shape

print("Same columns:", same_columns)
print("Same shape  :", same_shape)

numeric_cols = ref.select_dtypes(
    include=[np.number]
).columns

a = ref[numeric_cols].to_numpy(dtype=float)
b = new[numeric_cols].to_numpy(dtype=float)

numeric_match = np.allclose(
    a,
    b,
    rtol=1e-12,
    atol=1e-12,
    equal_nan=True
)

max_diff = np.nanmax(np.abs(a - b))

text_cols = [
    c for c in ref.columns
    if c not in numeric_cols
]

text_match = all(
    ref[c].fillna("").equals(
        new[c].fillna("")
    )
    for c in text_cols
)

print("Numeric match :", numeric_match)
print("Max abs diff  :", max_diff)
print("Text match    :", text_match)

print("\n======================================")
print("GRASS-LATTICE REPRODUCTION VERDICT")
print("======================================")

if (
    same_columns
    and same_shape
    and numeric_match
    and text_match
):
    print(
        "SUCCESS: Grass-Lattice baseline "
        "results reproduced exactly."
    )
else:
    print(
        "CHECK: Grass-Lattice results differ."
    )


In [ ]:
# Cell 33 — Resolve Grass-Lattice label-only mismatch

print("=== Grass-Lattice Label Audit ===\n")

ref = pd.read_csv(
    REPRO / "Data_Kumar_Original" / "dmm_vs_grasslattice_flat.csv"
)

new = pd.read_csv(
    GRASS / "dmm_vs_grasslattice_flat_reproduced.csv"
)

print("Released method labels:")
print(ref["method"].unique())

print("\nCurrent-script method labels:")
print(new["method"].unique())

# Current script renamed the SQM method only
label_map = {
    "Incidence-trained SQM": "Incidence-DMM"
}

new_normalized = new.copy()
new_normalized["method"] = (
    new_normalized["method"]
    .replace(label_map)
)

sort_cols = ["snr_db", "method"]

ref = ref.sort_values(sort_cols).reset_index(drop=True)
new_normalized = (
    new_normalized
    .sort_values(sort_cols)
    .reset_index(drop=True)
)

same_after_normalization = ref.equals(new_normalized)

print("\nAfter normalizing the renamed SQM label:")
print("Complete dataframe match:", same_after_normalization)

print("\n======================================")
print("GRASS-LATTICE FINAL VERDICT")
print("======================================")

if same_after_normalization:
    print(
        "SUCCESS: Results reproduced exactly. "
        "Only the SQM method label changed from "
        "'Incidence-DMM' to 'Incidence-trained SQM'."
    )
else:
    print("CHECK: Differences remain after label normalization.")


In [ ]:
# Cell 34 — Manifold-geometry diagnostic reproduction

import subprocess
import sys
import os
import time
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

print("=== Manifold-Geometry Diagnostic Reproduction ===\n")

MANIFOLD = RESULTS / "manifold_geometry"
MANIFOLD.mkdir(parents=True, exist_ok=True)

output_file = (
    MANIFOLD / "manifold_geometry_reproduced.csv"
)

log_file = (
    MANIFOLD / "manifold_geometry_run.log"
)

cmd = [
    sys.executable,
    "Code/evaluate_manifold_geometry.py",
    "--pairs", "20",
    "--seed", "20260929",
    "--output", str(output_file.relative_to(ROOT)),
]

print("Running Kumar's original script:")
print(" ".join(cmd))
print()
print("Settings:")
print("  Message pairs : 20")
print("  Seed          : 20260929")

start = time.time()

with open(log_file, "w") as log:

    process = subprocess.Popen(
        cmd,
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
        env=os.environ.copy()
    )

    with tqdm(
        desc="Manifold experiment running",
        unit="s",
        bar_format="{desc}: {n:.0f} s elapsed"
    ) as pbar:

        while process.poll() is None:
            time.sleep(1)
            pbar.update(1)

return_code = process.returncode
elapsed = time.time() - start

print("\n=== Run Finished ===")
print("Return code :", return_code)
print(f"Runtime     : {elapsed:.1f} seconds")
print("Output      :", output_file)
print("Log         :", log_file)

if return_code != 0:
    print(log_file.read_text()[-5000:])
    raise RuntimeError(
        "Manifold-geometry experiment failed."
    )

new = pd.read_csv(output_file)

print("\n=== Reproduced Results ===")
display(new)

# ------------------------------------------------------------
# Compare against Kumar's released data
# ------------------------------------------------------------

ref_file = (
    REPRO
    / "Data_Kumar_Original"
    / "manifold_geometry.csv"
)

ref = pd.read_csv(ref_file)

ref = ref.sort_values("case").reset_index(drop=True)
new = new.sort_values("case").reset_index(drop=True)

print("\n=== Exact Reproducibility Check ===")

print("Reference shape :", ref.shape)
print("Reproduced shape:", new.shape)

same_columns = list(ref.columns) == list(new.columns)
same_shape = ref.shape == new.shape

print("Same columns:", same_columns)
print("Same shape  :", same_shape)

numeric_cols = ref.select_dtypes(
    include=[np.number]
).columns

a = ref[numeric_cols].to_numpy(dtype=float)
b = new[numeric_cols].to_numpy(dtype=float)

numeric_match = np.allclose(
    a,
    b,
    rtol=1e-11,
    atol=1e-12,
    equal_nan=True
)

max_diff = np.nanmax(
    np.abs(a - b)
)

print("Numeric match :", numeric_match)
print("Max abs diff  :", max_diff)

# Check discrete/integer quantities separately
discrete_cols = [
    "case",
    "message_i",
    "message_j",
    "ambient_grassmann_dimension",
    "tangent_dim_i",
    "tangent_dim_j",
    "generic_intersection_lower_bound",
]

discrete_match = all(
    ref[c].equals(new[c])
    for c in discrete_cols
)

print("Discrete match:", discrete_match)

print("\n======================================")
print("MANIFOLD-GEOMETRY REPRODUCTION VERDICT")
print("======================================")

if (
    same_columns
    and same_shape
    and numeric_match
    and discrete_match
):
    print(
        "SUCCESS: Manifold-geometry diagnostic "
        "reproduced numerically."
    )
else:
    print(
        "CHECK: Manifold-geometry results differ."
    )


In [ ]:
# Cell 35 — Tuned OFDM + Oracle-Gap Reproduction

import subprocess
import sys
import os
import time
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

print("=== Tuned OFDM + Oracle-Gap Reproduction ===\n")

OFDMGAP = RESULTS / "ofdm_tuned_gap"
OFDMGAP.mkdir(parents=True, exist_ok=True)

output_file = (
    OFDMGAP
    / "ofdm_tuned_and_gap_reproduced.csv"
)

log_file = (
    OFDMGAP
    / "ofdm_tuned_and_gap_run.log"
)

cmd = [
    sys.executable,
    "Code/run_ofdm_tuned_and_gap.py",
    "--snr", "10",
    "--blocks", "300",
    "--seeds", "13,59",
    "--output", str(output_file.relative_to(ROOT)),
]

print("Running Kumar's original script:")
print(" ".join(cmd))

print("\nSettings:")
print("  SNR    : 10 dB")
print("  Blocks : 300 per seed")
print("  Seeds  : 13, 59")

start = time.time()

with open(log_file, "w") as log:

    process = subprocess.Popen(
        cmd,
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
        env=os.environ.copy()
    )

    with tqdm(
        desc="Tuned OFDM experiment running",
        unit="s",
        bar_format="{desc}: {n:.0f} s elapsed"
    ) as pbar:

        while process.poll() is None:
            time.sleep(1)
            pbar.update(1)

return_code = process.returncode
elapsed = time.time() - start

print("\n=== Run Finished ===")
print("Return code :", return_code)
print(f"Runtime     : {elapsed:.1f} seconds")
print("Output      :", output_file)
print("Log         :", log_file)

if return_code != 0:
    print(log_file.read_text()[-5000:])
    raise RuntimeError(
        "Tuned OFDM experiment failed."
    )

# ------------------------------------------------------------
# Reproduced results
# ------------------------------------------------------------

new = pd.read_csv(output_file)

print("\n=== Reproduced Raw Results ===")
display(new)

print("\n=== Mean PER by experiment/method ===")

summary = (
    new
    .groupby(["experiment", "method"], as_index=False)
    ["PER"]
    .mean()
    .sort_values(["experiment", "PER"])
    .reset_index(drop=True)
)

display(summary)

# ------------------------------------------------------------
# Exact comparison against Kumar's released data
# ------------------------------------------------------------

ref_file = (
    REPRO
    / "Data_Kumar_Original"
    / "ofdm_tuned_and_gap.csv"
)

ref = pd.read_csv(ref_file)

sort_cols = [
    "experiment",
    "method",
    "PER",
    "snr_db",
    "blocks"
]

ref = (
    ref
    .sort_values(sort_cols)
    .reset_index(drop=True)
)

new = (
    new
    .sort_values(sort_cols)
    .reset_index(drop=True)
)

print("\n=== Exact Reproducibility Check ===")

print("Reference shape :", ref.shape)
print("Reproduced shape:", new.shape)

same_columns = list(ref.columns) == list(new.columns)
same_shape = ref.shape == new.shape

print("Same columns:", same_columns)
print("Same shape  :", same_shape)

numeric_cols = ref.select_dtypes(
    include=[np.number]
).columns

a = ref[numeric_cols].to_numpy(dtype=float)
b = new[numeric_cols].to_numpy(dtype=float)

numeric_match = np.allclose(
    a,
    b,
    rtol=1e-12,
    atol=1e-12,
    equal_nan=True
)

max_diff = np.nanmax(
    np.abs(a - b)
)

text_cols = [
    c for c in ref.columns
    if c not in numeric_cols
]

text_match = all(
    ref[c].fillna("").equals(
        new[c].fillna("")
    )
    for c in text_cols
)

print("Numeric match :", numeric_match)
print("Max abs diff  :", max_diff)
print("Text match    :", text_match)

print("\n======================================")
print("TUNED OFDM / ORACLE-GAP VERDICT")
print("======================================")

if (
    same_columns
    and same_shape
    and numeric_match
    and text_match
):
    print(
        "SUCCESS: Tuned OFDM and oracle-gap "
        "results reproduced exactly."
    )
else:
    print(
        "CHECK: Tuned OFDM/oracle-gap "
        "results differ."
    )


In [ ]:
# Cell 36 — Delay-only online rank-control reproduction

import subprocess
import sys
import os
import time
import json
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

print("=== Delay-Only Online Rank-Control Reproduction ===\n")

DELAYCTRL = RESULTS / "delay_only_controller"
DELAYCTRL.mkdir(parents=True, exist_ok=True)

log_file = DELAYCTRL / "delay_only_controller_run.log"

# This Kumar script has no CLI arguments and writes directly to Data/
cmd = [
    sys.executable,
    "Code/evaluate_delay_transition.py"
]

print("Running Kumar's original script:")
print(" ".join(cmd))
print()
print("Experiment:")
print("  SNR trace   : 10, 10, 10, 10 dB")
print("  Delay trace : 30, 60, 100, 30 ns")
print("  Seeds       : 13, 59, 151")
print("  300 packets per regime")
print("  4 regimes -> 1200 packets per seed")
print()

start = time.time()

with open(log_file, "w") as log:

    process = subprocess.Popen(
        cmd,
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
        env=os.environ.copy()
    )

    with tqdm(
        desc="Delay-only controller running",
        unit="s",
        bar_format="{desc}: {n:.0f} s elapsed"
    ) as pbar:

        while process.poll() is None:
            time.sleep(1)
            pbar.update(1)

return_code = process.returncode
elapsed = time.time() - start

print("\n=== Run Finished ===")
print("Return code :", return_code)
print(f"Runtime     : {elapsed:.1f} seconds")
print("Log         :", log_file)

if return_code != 0:
    print(log_file.read_text()[-5000:])
    raise RuntimeError(
        "Delay-only controller experiment failed."
    )

# ------------------------------------------------------------
# Files written by Kumar's script
# ------------------------------------------------------------

generated = {
    "Summary":
        ROOT / "Data" / "online_delay_only_summary.csv",

    "Seed metrics":
        ROOT / "Data" / "online_delay_only_seed_metrics.csv",

    "By segment":
        ROOT / "Data" / "online_delay_only_by_segment.csv",
}

reference = {
    name:
        REPRO / "Data_Kumar_Original" / path.name
    for name, path in generated.items()
}

print("\n=== Reproduced Summary ===")
display(pd.read_csv(generated["Summary"]))

# ------------------------------------------------------------
# Exact CSV comparisons
# ------------------------------------------------------------

sort_keys = {
    "Summary":
        ["method"],

    "Seed metrics":
        ["seed", "method"],

    "By segment":
        ["seed", "method", "segment"],
}

overall_pass = True

print("\n=== Exact Reproducibility Check ===")

for name in generated:

    print(f"\n--- {name} ---")

    ref = pd.read_csv(reference[name])
    new = pd.read_csv(generated[name])

    keys = sort_keys[name]

    ref = (
        ref.sort_values(keys)
        .reset_index(drop=True)
    )

    new = (
        new.sort_values(keys)
        .reset_index(drop=True)
    )

    print("Reference shape :", ref.shape)
    print("Reproduced shape:", new.shape)

    same_columns = list(ref.columns) == list(new.columns)
    same_shape = ref.shape == new.shape

    print("Same columns:", same_columns)
    print("Same shape  :", same_shape)

    if not (same_columns and same_shape):
        overall_pass = False
        continue

    numeric_cols = ref.select_dtypes(
        include=[np.number]
    ).columns

    a = ref[numeric_cols].to_numpy(dtype=float)
    b = new[numeric_cols].to_numpy(dtype=float)

    numeric_match = np.allclose(
        a,
        b,
        rtol=1e-12,
        atol=1e-12,
        equal_nan=True
    )

    max_diff = np.nanmax(
        np.abs(a - b)
    )

    text_cols = [
        c for c in ref.columns
        if c not in numeric_cols
    ]

    text_match = all(
        ref[c].fillna("").equals(
            new[c].fillna("")
        )
        for c in text_cols
    )

    print("Numeric match :", numeric_match)
    print("Max abs diff  :", max_diff)
    print("Text match    :", text_match)

    if not (
        numeric_match
        and text_match
    ):
        overall_pass = False

# ------------------------------------------------------------
# Config JSON comparison
# ------------------------------------------------------------

print("\n--- Configuration JSON ---")

ref_json_file = (
    REPRO
    / "Data_Kumar_Original"
    / "online_delay_only_config.json"
)

new_json_file = (
    ROOT
    / "Data"
    / "online_delay_only_config.json"
)

with open(ref_json_file) as f:
    ref_json = json.load(f)

with open(new_json_file) as f:
    new_json = json.load(f)

json_match = ref_json == new_json

print("Parsed JSON match:", json_match)

if not json_match:
    overall_pass = False

# ------------------------------------------------------------
# Final verdict
# ------------------------------------------------------------

print("\n======================================")
print("DELAY-ONLY CONTROLLER VERDICT")
print("======================================")

if overall_pass:
    print(
        "SUCCESS: Delay-only online rank-control "
        "results reproduced exactly."
    )
else:
    print(
        "CHECK: One or more delay-only controller "
        "outputs differ."
    )


In [ ]:
# Cell 37 — Resolve delay-only summary CI inconsistency

import math
import pandas as pd
import numpy as np

print("=== Delay-Only Summary CI Audit ===\n")

seed_file = ROOT / "Data" / "online_delay_only_seed_metrics.csv"
ref_summary_file = (
    REPRO / "Data_Kumar_Original" / "online_delay_only_summary.csv"
)

seed = pd.read_csv(seed_file)
ref = pd.read_csv(ref_summary_file)

T_CRIT_DF2 = 4.302652729911275

rows = []

for method, g in seed.groupby("method", sort=False):

    rows.append({
        "method": method,
        "PER": g["PER"].mean(),
        "PER_ci95":
            T_CRIT_DF2
            * g["PER"].std(ddof=1)
            / math.sqrt(len(g)),
        "mean_rank":
            g["mean_selected_rank"].mean(),
        "switch_rate":
            g["switch_rate"].mean(),
        "adaptation_delay":
            g["mean_adaptation_delay_packets"].mean(),
        "mean_evals":
            g["mean_score_evals"].mean(),
    })

recomputed = pd.DataFrame(rows)

print("=== Recomputed summary using t(df=2) ===")
display(recomputed)

ref = (
    ref.sort_values("method")
    .reset_index(drop=True)
)

recomputed = (
    recomputed.sort_values("method")
    .reset_index(drop=True)
)

numeric_cols = ref.select_dtypes(
    include=[np.number]
).columns

a = ref[numeric_cols].to_numpy(float)
b = recomputed[numeric_cols].to_numpy(float)

numeric_match = np.allclose(
    a,
    b,
    rtol=1e-12,
    atol=1e-12,
    equal_nan=True
)

max_diff = np.nanmax(np.abs(a - b))

text_match = ref["method"].equals(
    recomputed["method"]
)

print("\n=== Corrected Comparison ===")
print("Numeric match :", numeric_match)
print("Max abs diff  :", max_diff)
print("Text match    :", text_match)

print("\n======================================")
print("DELAY-ONLY FINAL VERDICT")
print("======================================")

if numeric_match and text_match:
    print(
        "SUCCESS: Underlying delay-only results "
        "reproduced exactly."
    )
    print(
        "Released summary uses t(df=2) 95% CI; "
        "current script uses 1.96."
    )
    print(
        "Difference is statistical post-processing only, "
        "not simulation output."
    )
else:
    print("CHECK: Differences remain.")


In [ ]:
# Cell 38 — Adaptive diagnostics + Figure 15 sensitivity reproduction

import subprocess
import sys
import os
import time
import json
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

print("=== Adaptive Diagnostics / Sensitivity Reproduction ===\n")

DIAG = RESULTS / "adaptive_diagnostics"
DIAG.mkdir(parents=True, exist_ok=True)

log_file = DIAG / "online_adaptive_diagnostics_run.log"

cmd = [
    sys.executable,
    "Code/online_adaptive_diagnostics.py"
]

print("Running Kumar's original script:")
print(" ".join(cmd))
print()
print("Outputs:")
print("  online_proxy_recommender.csv")
print("  online_proxy_diagnostics.csv")
print("  online_controller_sensitivity.csv")
print("  online_diagnostics_config.json")

start = time.time()

with open(log_file, "w") as log:

    process = subprocess.Popen(
        cmd,
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
        env=os.environ.copy()
    )

    with tqdm(
        desc="Adaptive diagnostics running",
        unit="s",
        bar_format="{desc}: {n:.0f} s elapsed"
    ) as pbar:

        while process.poll() is None:
            time.sleep(1)
            pbar.update(1)

return_code = process.returncode
elapsed = time.time() - start

print("\n=== Run Finished ===")
print("Return code :", return_code)
print(f"Runtime     : {elapsed:.1f} seconds")
print("Log         :", log_file)

if return_code != 0:
    print(log_file.read_text()[-6000:])
    raise RuntimeError(
        "Adaptive diagnostics experiment failed."
    )

# ------------------------------------------------------------
# Generated files
# ------------------------------------------------------------

generated = {
    "Proxy recommender":
        ROOT / "Data" / "online_proxy_recommender.csv",

    "Proxy diagnostics":
        ROOT / "Data" / "online_proxy_diagnostics.csv",

    "Controller sensitivity":
        ROOT / "Data" / "online_controller_sensitivity.csv",
}

references = {
    name:
        REPRO / "Data_Kumar_Original" / path.name
    for name, path in generated.items()
}

print("\n=== Reproduced Figure 15 Sensitivity Results ===")
display(
    pd.read_csv(
        generated["Controller sensitivity"]
    )
)

# ------------------------------------------------------------
# Exact comparisons
# ------------------------------------------------------------

sort_keys = {
    "Proxy recommender":
        ["split", "seed"],

    "Proxy diagnostics":
        ["seed", "feature", "quartile"],

    "Controller sensitivity":
        ["policy", "setting"],
}

overall_pass = True

print("\n=== Exact Reproducibility Checks ===")

for name in generated:

    print(f"\n--- {name} ---")

    ref = pd.read_csv(references[name])
    new = pd.read_csv(generated[name])

    keys = sort_keys[name]

    # Convert sorting columns to strings where mixed types occur
    for c in keys:
        if c in ref.columns:
            ref[c] = ref[c].astype(str)
            new[c] = new[c].astype(str)

    ref = (
        ref.sort_values(keys)
        .reset_index(drop=True)
    )

    new = (
        new.sort_values(keys)
        .reset_index(drop=True)
    )

    print("Reference shape :", ref.shape)
    print("Reproduced shape:", new.shape)

    same_columns = list(ref.columns) == list(new.columns)
    same_shape = ref.shape == new.shape

    print("Same columns:", same_columns)
    print("Same shape  :", same_shape)

    if not (same_columns and same_shape):
        overall_pass = False
        continue

    numeric_cols = [
        c for c in ref.columns
        if c not in keys
        and pd.api.types.is_numeric_dtype(ref[c])
    ]

    if numeric_cols:

        a = ref[numeric_cols].to_numpy(float)
        b = new[numeric_cols].to_numpy(float)

        numeric_match = np.allclose(
            a,
            b,
            rtol=1e-12,
            atol=1e-12,
            equal_nan=True
        )

        diff = np.abs(a - b)

        max_diff = (
            np.nanmax(diff)
            if not np.all(np.isnan(diff))
            else 0.0
        )

    else:
        numeric_match = True
        max_diff = 0.0

    text_cols = [
        c for c in ref.columns
        if c not in numeric_cols
    ]

    text_match = all(
        ref[c].fillna("").astype(str).equals(
            new[c].fillna("").astype(str)
        )
        for c in text_cols
    )

    print("Numeric match :", numeric_match)
    print("Max abs diff  :", max_diff)
    print("Text match    :", text_match)

    if not (numeric_match and text_match):
        overall_pass = False


# ------------------------------------------------------------
# Configuration JSON
# ------------------------------------------------------------

print("\n--- Diagnostics configuration ---")

ref_cfg_file = (
    REPRO
    / "Data_Kumar_Original"
    / "online_diagnostics_config.json"
)

new_cfg_file = (
    ROOT
    / "Data"
    / "online_diagnostics_config.json"
)

with open(ref_cfg_file) as f:
    ref_cfg = json.load(f)

with open(new_cfg_file) as f:
    new_cfg = json.load(f)

config_match = ref_cfg == new_cfg

print("JSON configuration match:", config_match)

if not config_match:
    overall_pass = False


# ------------------------------------------------------------
# Check Kumar's revision sensitivity file too
# ------------------------------------------------------------

print("\n--- Sensitivity revision duplicate check ---")

revision_file = (
    REPRO
    / "Data_Kumar_Original"
    / "online_controller_sensitivity_revision.csv"
)

generated_sens = pd.read_csv(
    generated["Controller sensitivity"]
)

revision_sens = pd.read_csv(revision_file)

revision_match = (
    generated_sens.shape == revision_sens.shape
    and list(generated_sens.columns)
        == list(revision_sens.columns)
    and np.allclose(
        generated_sens.select_dtypes(
            include=[np.number]
        ).to_numpy(float),
        revision_sens.select_dtypes(
            include=[np.number]
        ).to_numpy(float),
        rtol=1e-12,
        atol=1e-12,
        equal_nan=True
    )
)

print(
    "Generated sensitivity matches "
    "online_controller_sensitivity_revision.csv:",
    revision_match
)

if not revision_match:
    overall_pass = False


# ------------------------------------------------------------
# Final verdict
# ------------------------------------------------------------

print("\n======================================")
print("ADAPTIVE DIAGNOSTICS / FIGURE 15 VERDICT")
print("======================================")

if overall_pass:
    print(
        "SUCCESS: Adaptive diagnostics and "
        "Figure 15 sensitivity results reproduced exactly."
    )
else:
    print(
        "CHECK: One or more adaptive diagnostic "
        "outputs differ."
    )


In [ ]:
# Cell 39 — Diagnose Adaptive Diagnostics final CHECK

import json
import math
import pandas as pd
import numpy as np

print("=== Adaptive Diagnostics Final Audit ===\n")

def compare_csv(ref_file, new_file, sort_cols=None):
    ref = pd.read_csv(ref_file)
    new = pd.read_csv(new_file)

    if sort_cols:
        ref = ref.sort_values(sort_cols).reset_index(drop=True)
        new = new.sort_values(sort_cols).reset_index(drop=True)

    same_columns = list(ref.columns) == list(new.columns)
    same_shape = ref.shape == new.shape

    numeric_cols = ref.select_dtypes(include=[np.number]).columns

    if same_columns and same_shape and len(numeric_cols):
        a = ref[numeric_cols].to_numpy(float)
        b = new[numeric_cols].to_numpy(float)

        numeric_match = np.allclose(
            a, b,
            rtol=1e-12,
            atol=1e-12,
            equal_nan=True
        )

        d = np.abs(a - b)
        max_diff = (
            np.nanmax(d)
            if not np.all(np.isnan(d))
            else 0.0
        )
    else:
        numeric_match = same_columns and same_shape
        max_diff = np.nan

    text_cols = [
        c for c in ref.columns
        if c not in numeric_cols
    ]

    text_match = (
        same_columns
        and same_shape
        and all(
            ref[c].fillna("").astype(str).equals(
                new[c].fillna("").astype(str)
            )
            for c in text_cols
        )
    )

    return {
        "same_columns": same_columns,
        "same_shape": same_shape,
        "numeric_match": numeric_match,
        "max_diff": max_diff,
        "text_match": text_match,
    }


tests = [
    (
        "Proxy recommender",
        REPRO / "Data_Kumar_Original" /
        "online_proxy_recommender.csv",
        ROOT / "Data" /
        "online_proxy_recommender.csv",
        None,
    ),
    (
        "Proxy diagnostics",
        REPRO / "Data_Kumar_Original" /
        "online_proxy_diagnostics.csv",
        ROOT / "Data" /
        "online_proxy_diagnostics.csv",
        None,
    ),
    (
        "Controller sensitivity",
        REPRO / "Data_Kumar_Original" /
        "online_controller_sensitivity.csv",
        ROOT / "Data" /
        "online_controller_sensitivity.csv",
        ["policy", "setting"],
    ),
    (
        "Sensitivity revision copy",
        REPRO / "Data_Kumar_Original" /
        "online_controller_sensitivity_revision.csv",
        ROOT / "Data" /
        "online_controller_sensitivity.csv",
        ["policy", "setting"],
    ),
]

csv_pass = True

for name, ref_file, new_file, sort_cols in tests:

    result = compare_csv(
        ref_file,
        new_file,
        sort_cols
    )

    print(f"--- {name} ---")
    for k, v in result.items():
        print(f"{k:15s}: {v}")
    print()

    if not (
        result["same_columns"]
        and result["same_shape"]
        and result["numeric_match"]
        and result["text_match"]
    ):
        csv_pass = False


# ------------------------------------------------------------
# JSON comparison with tolerant floating-point handling
# ------------------------------------------------------------

print("--- Diagnostics configuration JSON ---")

with open(
    REPRO / "Data_Kumar_Original" /
    "online_diagnostics_config.json"
) as f:
    ref_cfg = json.load(f)

with open(
    ROOT / "Data" /
    "online_diagnostics_config.json"
) as f:
    new_cfg = json.load(f)


def json_close(a, b, path="root"):

    differences = []

    if isinstance(a, dict) and isinstance(b, dict):

        if set(a) != set(b):
            differences.append(
                (path, "different keys")
            )
            return False, differences

        ok = True

        for key in a:
            same, diff = json_close(
                a[key],
                b[key],
                path + "." + str(key)
            )
            ok &= same
            differences.extend(diff)

        return ok, differences

    if isinstance(a, list) and isinstance(b, list):

        if len(a) != len(b):
            return False, [
                (path, f"length {len(a)} vs {len(b)}")
            ]

        ok = True

        for i, (x, y) in enumerate(zip(a, b)):
            same, diff = json_close(
                x, y,
                f"{path}[{i}]"
            )
            ok &= same
            differences.extend(diff)

        return ok, differences

    if (
        isinstance(a, (int, float))
        and isinstance(b, (int, float))
    ):
        same = math.isclose(
            float(a),
            float(b),
            rel_tol=1e-12,
            abs_tol=1e-12
        )

        if not same:
            return False, [
                (path, f"{a} vs {b}")
            ]

        return True, []

    same = a == b

    return (
        same,
        [] if same
        else [(path, f"{a!r} vs {b!r}")]
    )


json_match, json_differences = json_close(
    ref_cfg,
    new_cfg
)

print("Tolerant JSON match:", json_match)

if not json_match:
    print("\nDifferences:")
    for item in json_differences:
        print(item)


print("\n======================================")
print("ADAPTIVE DIAGNOSTICS FINAL VERDICT")
print("======================================")

if csv_pass and json_match:
    print(
        "SUCCESS: Adaptive diagnostics and "
        "Figure 15 sensitivity reproduced."
    )
else:
    print(
        "CSV results exact:", csv_pass
    )
    print(
        "Configuration match:", json_match
    )


In [ ]:
# Cell 40 — Show only diagnostics JSON differences

import json
import math

ref_file = (
    REPRO / "Data_Kumar_Original"
    / "online_diagnostics_config.json"
)

new_file = (
    ROOT / "Data"
    / "online_diagnostics_config.json"
)

with open(ref_file) as f:
    ref = json.load(f)

with open(new_file) as f:
    new = json.load(f)


def diff_json(a, b, path="root"):

    diffs = []

    if isinstance(a, dict) and isinstance(b, dict):

        for key in sorted(set(a) | set(b)):

            if key not in a:
                diffs.append(
                    (f"{path}.{key}", "MISSING IN REFERENCE", b[key])
                )

            elif key not in b:
                diffs.append(
                    (f"{path}.{key}", a[key], "MISSING IN NEW")
                )

            else:
                diffs.extend(
                    diff_json(
                        a[key],
                        b[key],
                        f"{path}.{key}"
                    )
                )

    elif isinstance(a, list) and isinstance(b, list):

        if len(a) != len(b):
            diffs.append(
                (path, f"length={len(a)}", f"length={len(b)}")
            )

        else:
            for i, (x, y) in enumerate(zip(a, b)):
                diffs.extend(
                    diff_json(
                        x, y,
                        f"{path}[{i}]"
                    )
                )

    elif (
        isinstance(a, (int, float))
        and isinstance(b, (int, float))
    ):

        if not math.isclose(
            float(a),
            float(b),
            rel_tol=1e-12,
            abs_tol=1e-12
        ):
            diffs.append(
                (path, a, b)
            )

    elif a != b:

        diffs.append(
            (path, a, b)
        )

    return diffs


diffs = diff_json(ref, new)

print("=== Diagnostics JSON Differences ===")
print("Number of differing fields:", len(diffs))

for path, old, new_value in diffs:
    print()
    print(path)
    print("  released :", old)
    print("  reproduced:", new_value)


print("\n======================================")

if len(diffs) == 0:
    print("JSON MATCHES within numerical tolerance.")
else:
    print(
        "Scientific CSV results still reproduce exactly; "
        "differences above are configuration metadata."
    )


In [ ]:
# Cell 41 — Final Figure 15 verdict

print("======================================")
print("FIGURE 15 / ADAPTIVE DIAGNOSTICS FINAL VERDICT")
print("======================================")

print(
    "SUCCESS: All released scientific CSV results "
    "reproduced exactly."
)

print(
    "The diagnostics JSON contains only negligible "
    "floating-point differences (~1e-9) in proxy-model "
    "centroids and normalization parameters."
)

print(
    "These differences do not affect any reproduced "
    "controller decisions or reported performance."
)


In [ ]:
# Cell 42 — Verify remaining adaptive-revision outputs

import json
import math
import pandas as pd
import numpy as np

print("=== Remaining Adaptive-Revision Output Check ===\n")

checks = [
    (
        "Transient seed metrics",
        REPRO / "Data_Kumar_Original" / "online_transient_seed_metrics.csv",
        ROOT / "Data" / "online_transient_seed_metrics.csv",
        ["sequence", "seed", "method", "transition_segment", "window"],
    ),
    (
        "Transient summary",
        REPRO / "Data_Kumar_Original" / "online_transient_summary.csv",
        ROOT / "Data" / "online_transient_summary.csv",
        ["sequence", "method", "transition_segment", "window"],
    ),
]

overall_pass = True

for name, ref_file, new_file, keys in checks:

    print(f"--- {name} ---")

    ref = pd.read_csv(ref_file)
    new = pd.read_csv(new_file)

    ref = ref.sort_values(keys).reset_index(drop=True)
    new = new.sort_values(keys).reset_index(drop=True)

    print("Reference shape :", ref.shape)
    print("Reproduced shape:", new.shape)

    same_columns = list(ref.columns) == list(new.columns)
    same_shape = ref.shape == new.shape

    print("Same columns:", same_columns)
    print("Same shape  :", same_shape)

    numeric_cols = ref.select_dtypes(include=[np.number]).columns

    a = ref[numeric_cols].to_numpy(float)
    b = new[numeric_cols].to_numpy(float)

    numeric_match = np.allclose(
        a, b,
        rtol=1e-12,
        atol=1e-12,
        equal_nan=True
    )

    max_diff = np.nanmax(np.abs(a - b))

    text_cols = [c for c in ref.columns if c not in numeric_cols]

    text_match = all(
        ref[c].fillna("").equals(new[c].fillna(""))
        for c in text_cols
    )

    print("Numeric match :", numeric_match)
    print("Max abs diff  :", max_diff)
    print("Text match    :", text_match)
    print()

    overall_pass &= (
        same_columns
        and same_shape
        and numeric_match
        and text_match
    )


# ------------------------------------------------------------
# Revision JSON config
# ------------------------------------------------------------

print("--- Adaptive revision config ---")

ref_cfg_file = (
    REPRO
    / "Data_Kumar_Original"
    / "online_adaptive_revision_config.json"
)

new_cfg_file = (
    ROOT
    / "Data"
    / "online_adaptive_revision_config.json"
)

with open(ref_cfg_file) as f:
    ref_cfg = json.load(f)

with open(new_cfg_file) as f:
    new_cfg = json.load(f)


def json_close(a, b):

    if isinstance(a, dict) and isinstance(b, dict):
        return (
            set(a) == set(b)
            and all(json_close(a[k], b[k]) for k in a)
        )

    if isinstance(a, list) and isinstance(b, list):
        return (
            len(a) == len(b)
            and all(json_close(x, y) for x, y in zip(a, b))
        )

    if isinstance(a, (int, float)) and isinstance(b, (int, float)):
        return math.isclose(
            float(a), float(b),
            rel_tol=1e-12,
            abs_tol=1e-12
        )

    return a == b


config_match = json_close(ref_cfg, new_cfg)

print("Configuration match:", config_match)

overall_pass &= config_match


print("\n======================================")
print("ADAPTIVE REVISION COMPLETENESS VERDICT")
print("======================================")

if overall_pass:
    print(
        "SUCCESS: Transient metrics, transient summary, "
        "and revision configuration reproduced."
    )
else:
    print(
        "CHECK: One or more remaining adaptive-revision "
        "outputs differ."
    )


In [ ]:
# Cell 43 — Original/legacy adaptive-controller reproduction

import subprocess
import sys
import os
import time
import json
import math
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

print("=== Original Adaptive Controller Reproduction ===\n")

LEGACY = RESULTS / "legacy_adaptive"
LEGACY.mkdir(parents=True, exist_ok=True)

results_file = LEGACY / "online_adaptive_results_reproduced.csv"
summary_file = LEGACY / "online_adaptive_summary_reproduced.csv"
segment_file = LEGACY / "online_adaptive_summary_reproduced_by_segment.csv"
config_file = LEGACY / "online_adaptive_config_reproduced.json"
log_file = LEGACY / "legacy_adaptive_run.log"

cmd = [
    sys.executable,
    "Code/evaluate_online_adaptive.py",
    "--blocks-per-segment", "300",
    "--val-blocks-per-segment", "250",
    "--seeds", "13,59,151",
    "--output", str(results_file.relative_to(ROOT)),
    "--summary-output", str(summary_file.relative_to(ROOT)),
    "--config-output", str(config_file.relative_to(ROOT)),
]

print("Running Kumar's original script:")
print(" ".join(cmd))

print("\nSettings:")
print("  Final blocks / segment      : 300")
print("  Validation blocks / segment : 250")
print("  Seeds                       : 13, 59, 151")
print("  SNR trace                   : 5, 10, 15, 7 dB")
print("  Delay trace                 : 30, 60, 100, 30 ns")
print("  Schemes                     : S0 through S8")

start = time.time()

with open(log_file, "w") as log:

    process = subprocess.Popen(
        cmd,
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
        env=os.environ.copy()
    )

    with tqdm(
        desc="Legacy adaptive experiment running",
        unit="s",
        bar_format="{desc}: {n:.0f} s elapsed"
    ) as pbar:

        while process.poll() is None:
            time.sleep(1)
            pbar.update(1)

return_code = process.returncode
elapsed = time.time() - start

print("\n=== Run Finished ===")
print("Return code :", return_code)
print(f"Runtime     : {elapsed:.1f} seconds")
print("Results     :", results_file)
print("Summary     :", summary_file)
print("By segment  :", segment_file)
print("Config      :", config_file)
print("Log         :", log_file)

if return_code != 0:
    print(log_file.read_text()[-6000:])
    raise RuntimeError(
        "Legacy adaptive experiment failed."
    )


# ============================================================
# Inspect reproduced summary
# ============================================================

summary_new = pd.read_csv(summary_file)

print("\n=== Reproduced Legacy Adaptive Summary ===")
display(summary_new)


# ============================================================
# Exact CSV comparison helper
# ============================================================

def compare_csv(name, ref_file, new_file, sort_cols):

    print(f"\n--- {name} ---")

    ref = pd.read_csv(ref_file)
    new = pd.read_csv(new_file)

    ref = (
        ref.sort_values(sort_cols)
        .reset_index(drop=True)
    )

    new = (
        new.sort_values(sort_cols)
        .reset_index(drop=True)
    )

    print("Reference shape :", ref.shape)
    print("Reproduced shape:", new.shape)

    same_columns = list(ref.columns) == list(new.columns)
    same_shape = ref.shape == new.shape

    print("Same columns:", same_columns)
    print("Same shape  :", same_shape)

    if not (same_columns and same_shape):
        return False

    numeric_cols = ref.select_dtypes(
        include=[np.number]
    ).columns

    a = ref[numeric_cols].to_numpy(float)
    b = new[numeric_cols].to_numpy(float)

    numeric_match = np.allclose(
        a,
        b,
        rtol=1e-12,
        atol=1e-12,
        equal_nan=True
    )

    diff = np.abs(a - b)

    max_diff = (
        np.nanmax(diff)
        if not np.all(np.isnan(diff))
        else 0.0
    )

    text_cols = [
        c for c in ref.columns
        if c not in numeric_cols
    ]

    text_match = all(
        ref[c].fillna("").astype(str).equals(
            new[c].fillna("").astype(str)
        )
        for c in text_cols
    )

    print("Numeric match :", numeric_match)
    print("Max abs diff  :", max_diff)
    print("Text match    :", text_match)

    return (
        same_columns
        and same_shape
        and numeric_match
        and text_match
    )


print("\n=== Exact Reproducibility Checks ===")

pass_results = compare_csv(
    "Per-seed adaptive results",
    REPRO / "Data_Kumar_Original"
          / "online_adaptive_results.csv",
    results_file,
    ["seed", "method"]
)

pass_summary = compare_csv(
    "Adaptive summary",
    REPRO / "Data_Kumar_Original"
          / "online_adaptive_summary.csv",
    summary_file,
    ["method"]
)

pass_segments = compare_csv(
    "Adaptive by-segment results",
    REPRO / "Data_Kumar_Original"
          / "online_adaptive_summary_by_segment.csv",
    segment_file,
    ["seed", "method", "segment"]
)


# ============================================================
# Configuration comparison
# ============================================================

print("\n--- Adaptive configuration ---")

with open(
    REPRO
    / "Data_Kumar_Original"
    / "online_adaptive_config.json"
) as f:
    ref_cfg = json.load(f)

with open(config_file) as f:
    new_cfg = json.load(f)


def json_close(a, b):

    if isinstance(a, dict) and isinstance(b, dict):
        return (
            set(a.keys()) == set(b.keys())
            and all(
                json_close(a[k], b[k])
                for k in a
            )
        )

    if isinstance(a, list) and isinstance(b, list):
        return (
            len(a) == len(b)
            and all(
                json_close(x, y)
                for x, y in zip(a, b)
            )
        )

    if (
        isinstance(a, (int, float))
        and isinstance(b, (int, float))
    ):
        return math.isclose(
            float(a),
            float(b),
            rel_tol=1e-12,
            abs_tol=1e-12
        )

    return a == b


config_match = json_close(
    ref_cfg,
    new_cfg
)

print("Configuration match:", config_match)


# ============================================================
# Final verdict
# ============================================================

overall_pass = (
    pass_results
    and pass_summary
    and pass_segments
    and config_match
)

print("\n======================================")
print("LEGACY ADAPTIVE CONTROLLER VERDICT")
print("======================================")

if overall_pass:

    print(
        "SUCCESS: Original adaptive-controller "
        "results reproduced exactly."
    )

else:

    print(
        "CHECK: One or more legacy adaptive "
        "outputs differ."
    )


In [ ]:
# Cell 44 — Reproduce Kumar's fully t-based legacy adaptive summary

import math
import pandas as pd
import numpy as np

print("=== Legacy Adaptive t-Based Summary Reproduction ===\n")

df = pd.read_csv(
    LEGACY / "online_adaptive_results_reproduced.csv"
)

T_CRIT_DF2 = 4.302652729911275

metrics = [
    "PER",
    "mean_selected_rank",
    "switch_rate",
    "mean_adaptation_delay_packets",
    "mean_score_evals",
    "p95_score_evals",
]

rows = []

for method, g in df.groupby("method", sort=False):

    row = {"method": method}

    for col in metrics:

        values = g[col]

        mean = values.mean()

        if values.notna().sum() < 2:
            ci = 0.0
        else:
            ci = (
                T_CRIT_DF2
                * values.std(ddof=1)
                / math.sqrt(values.notna().sum())
            )

        row[f"{col}_mean"] = mean
        row[f"{col}_t95"] = ci

    rows.append(row)

reproduced_t = pd.DataFrame(rows)

print("=== Reproduced t-based summary ===")
display(reproduced_t)

# ------------------------------------------------------------
# Compare with Kumar's released online_adaptive_summary_t.csv
# ------------------------------------------------------------

ref_t = pd.read_csv(
    REPRO
    / "Data_Kumar_Original"
    / "online_adaptive_summary_t.csv"
)

ref_t = (
    ref_t.sort_values("method")
    .reset_index(drop=True)
)

reproduced_t = (
    reproduced_t.sort_values("method")
    .reset_index(drop=True)
)

print("\nReference shape :", ref_t.shape)
print("Reproduced shape:", reproduced_t.shape)

same_columns = list(ref_t.columns) == list(reproduced_t.columns)
same_shape = ref_t.shape == reproduced_t.shape

numeric_cols = ref_t.select_dtypes(
    include=[np.number]
).columns

a = ref_t[numeric_cols].to_numpy(float)
b = reproduced_t[numeric_cols].to_numpy(float)

numeric_match = np.allclose(
    a,
    b,
    rtol=1e-12,
    atol=1e-12,
    equal_nan=True
)

max_diff = np.nanmax(np.abs(a - b))

text_match = ref_t["method"].equals(
    reproduced_t["method"]
)

print("Same columns :", same_columns)
print("Same shape   :", same_shape)
print("Numeric match:", numeric_match)
print("Max abs diff :", max_diff)
print("Text match   :", text_match)

print("\n======================================")
print("LEGACY ADAPTIVE t-SUMMARY VERDICT")
print("======================================")

if (
    same_columns
    and same_shape
    and numeric_match
    and text_match
):
    print(
        "SUCCESS: Kumar's fully t-based legacy "
        "adaptive summary reproduced exactly."
    )
else:
    print("CHECK: Differences remain.")


In [ ]:
# Cell 45 — Underlying Figure 1 selective-margin sampling audit

import subprocess
import sys
import os
import time
import pandas as pd
import numpy as np
from tqdm.auto import tqdm

print("=== Figure 1 Underlying Selective-Margin Sampling Audit ===\n")

MARGIN = RESULTS / "selective_margin"
MARGIN.mkdir(parents=True, exist_ok=True)

output_file = (
    MARGIN / "selective_margin_samples_reproduced.csv"
)

log_file = (
    MARGIN / "selective_margin_run.log"
)

cmd = [
    sys.executable,
    "Code/evaluate_selective_margin.py",
    "--cases", "500",
    "--wrong", "16",
    "--ranks", "2,3,4,5,6,8",
    "--seed", "20260918",
    "--output", str(output_file.relative_to(ROOT)),
]

print("Running Kumar's released evaluator:")
print(" ".join(cmd))

print("\nSettings:")
print("  Cases        : 500")
print("  Wrong msgs   : 16 sampled per case")
print("  Ranks        : 2,3,4,5,6,8")
print("  Seed         : 20260918")

start = time.time()

with open(log_file, "w") as log:

    process = subprocess.Popen(
        cmd,
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
        env=os.environ.copy()
    )

    with tqdm(
        desc="Selective-margin experiment running",
        unit="s",
        bar_format="{desc}: {n:.0f} s elapsed"
    ) as pbar:

        while process.poll() is None:
            time.sleep(1)
            pbar.update(1)

return_code = process.returncode
elapsed = time.time() - start

print("\n=== Run Finished ===")
print("Return code :", return_code)
print(f"Runtime     : {elapsed:.1f} seconds")
print("Output      :", output_file)
print("Log         :", log_file)

if return_code != 0:
    print(log_file.read_text()[-5000:])
    raise RuntimeError(
        "Selective-margin experiment failed."
    )


# ------------------------------------------------------------
# Load released and regenerated samples
# ------------------------------------------------------------

ref_file = (
    REPRO
    / "Data_Kumar_Original"
    / "selective_margin_samples.csv"
)

ref = pd.read_csv(ref_file)
new = pd.read_csv(output_file)

# Kumar's released sample file includes the certified radius.
# The current evaluator does not write it, but mathematically:
# certified radius = selective margin / 2
new["certified_noise_radius_norm"] = (
    new["selective_margin_norm"] / 2.0
)

new = new[ref.columns]

print("\nReference shape :", ref.shape)
print("Reproduced shape:", new.shape)

same_columns = list(ref.columns) == list(new.columns)
same_shape = ref.shape == new.shape

print("Same columns:", same_columns)
print("Same shape  :", same_shape)

ref = ref.sort_values(["case", "rank"]).reset_index(drop=True)
new = new.sort_values(["case", "rank"]).reset_index(drop=True)

numeric_cols = ref.select_dtypes(include=[np.number]).columns

a = ref[numeric_cols].to_numpy(float)
b = new[numeric_cols].to_numpy(float)

numeric_match = np.allclose(
    a,
    b,
    rtol=1e-12,
    atol=1e-12,
    equal_nan=True
)

max_diff = np.nanmax(np.abs(a - b))

print("Numeric match :", numeric_match)
print("Max abs diff  :", max_diff)


# ------------------------------------------------------------
# Recompute Figure 1 summary from current evaluator
# ------------------------------------------------------------

summary_rows = []

for rank, g in new.groupby("rank"):

    margin = g["selective_margin_norm"]

    summary_rows.append({
        "rank": rank,
        "mean_true_mismatch":
            g["true_mismatch_norm"].mean(),
        "p90_true_mismatch":
            g["true_mismatch_norm"].quantile(0.90),
        "mean_wrong_distance":
            g["nearest_wrong_dist_norm"].mean(),
        "p01_wrong_distance":
            g["nearest_wrong_dist_norm"].quantile(0.01),
        "mean_selective_margin":
            margin.mean(),
        "p01_selective_margin":
            margin.quantile(0.01),
        "frac_positive_margin":
            (margin > 0).mean(),
    })

summary_new = pd.DataFrame(summary_rows)

summary_ref = pd.read_csv(
    REPRO
    / "Data_Kumar_Original"
    / "selective_margin_summary.csv"
)

print("\n=== Released Figure 1 summary ===")
display(summary_ref)

print("\n=== Summary from current evaluator ===")
display(summary_new)

summary_match = np.allclose(
    summary_ref.select_dtypes(include=[np.number]).to_numpy(float),
    summary_new.select_dtypes(include=[np.number]).to_numpy(float),
    rtol=1e-12,
    atol=1e-12
)

summary_max_diff = np.max(
    np.abs(
        summary_ref.select_dtypes(include=[np.number]).to_numpy(float)
        -
        summary_new.select_dtypes(include=[np.number]).to_numpy(float)
    )
)

print("\nSummary numeric match:", summary_match)
print("Summary max abs diff :", summary_max_diff)


print("\n======================================")
print("FIGURE 1 UNDERLYING-SAMPLING VERDICT")
print("======================================")

if numeric_match and summary_match:
    print(
        "SUCCESS: Released selective-margin samples "
        "reproduced exactly."
    )
else:
    print(
        "NOT EXACTLY REPRODUCIBLE FROM CURRENT RELEASE."
    )
    print(
        "The released evaluator itself states that the "
        "manuscript CSV used a different/larger sampled "
        "wrong-message setup."
    )
    print(
        "Figure 1 can be reconstructed exactly from the "
        "released CSV, but its underlying 500-case sample "
        "generation cannot be recovered exactly from the "
        "current script/configuration."
    )


In [ ]:
# Cell 46 — Reference incidence-aware codebook training

import subprocess
import sys
import os
import time
import shutil
import numpy as np
from tqdm.auto import tqdm

print("=== Reference Incidence-Aware Codebook Training ===\n")

TRAIN = RESULTS / "reference_training"
TRAIN.mkdir(parents=True, exist_ok=True)

log_file = TRAIN / "train_selective_incidence.log"

# Kumar's script always writes here
generated_artifact = (
    ROOT / "Artifact" / "retrained_from_reference.npz"
)

# Copy it here afterwards for our independent record
saved_artifact = (
    TRAIN / "retrained_from_reference.npz"
)

cmd = [
    sys.executable,
    "Code/train_selective_incidence.py",
    "--steps", "180",
    "--batch", "64",
    "--rank", "4",
    "--seed", "20260918",
]

print("Running Kumar's reference training script:")
print(" ".join(cmd))

print("\nSettings:")
print("  Steps : 180")
print("  Batch : 64")
print("  Rank  : 4")
print("  Seed  : 20260918")
print("  Hard-negative-k : 0")
print()

start = time.time()

with open(log_file, "w") as log:

    process = subprocess.Popen(
        cmd,
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
        env=os.environ.copy()
    )

    with tqdm(
        desc="Reference training running",
        unit="s",
        bar_format="{desc}: {n:.0f} s elapsed"
    ) as pbar:

        while process.poll() is None:
            time.sleep(1)
            pbar.update(1)

return_code = process.returncode
elapsed = time.time() - start

print("\n=== Training Finished ===")
print("Return code :", return_code)
print(f"Runtime     : {elapsed:.1f} seconds")
print("Log         :", log_file)

if return_code != 0:
    print(log_file.read_text()[-6000:])
    raise RuntimeError(
        "Reference codebook training failed."
    )

if not generated_artifact.exists():
    raise FileNotFoundError(
        "Training completed but retrained artifact was not created."
    )

# Preserve independent copy
shutil.copy2(
    generated_artifact,
    saved_artifact
)

print("Generated :", generated_artifact)
print("Saved copy:", saved_artifact)


# ============================================================
# Inspect training log
# ============================================================

print("\n=== Training Loss Trace ===")

log_text = log_file.read_text()

for line in log_text.splitlines():
    print(line)


# ============================================================
# Validate generated codebook
# ============================================================

artifact = np.load(saved_artifact)

print("\nArtifact keys:", artifact.files)

X = artifact["X_opt"]

print("Codebook shape:", X.shape)
print("Finite values :", np.isfinite(X).all())

M, two, L = X.shape

# ------------------------------------------------------------
# Mathematical constraints imposed by Kumar's parametrization
# ------------------------------------------------------------

# 1. Temporal means should be zero because training uses
#    the centered (L-1)-dimensional basis.
temporal_means = X.mean(axis=2)

max_abs_mean = np.max(
    np.abs(temporal_means)
)

# 2. I and Q rows should be orthogonal with equal energy.
target_energy = L / 2.0

max_energy_error = 0.0
max_iq_inner_product = 0.0

for m in range(M):

    G = X[m] @ X[m].T

    energy_error = np.max(
        np.abs(
            np.diag(G) - target_energy
        )
    )

    iq_inner = abs(G[0, 1])

    max_energy_error = max(
        max_energy_error,
        energy_error
    )

    max_iq_inner_product = max(
        max_iq_inner_product,
        iq_inner
    )


print("\n=== Codebook Constraint Checks ===")

print(
    "Max |temporal mean|      :",
    max_abs_mean
)

print(
    "Max row-energy error     :",
    max_energy_error
)

print(
    "Max |I·Q inner product|  :",
    max_iq_inner_product
)

mean_ok = max_abs_mean < 1e-5
energy_ok = max_energy_error < 1e-4
orthogonal_ok = max_iq_inner_product < 1e-4


# ============================================================
# Compare with initialization and frozen paper artifact
# ============================================================

stage3 = np.load(
    ROOT / "Artifact" / "stage3_codebooks.npz"
)

X_init = stage3["X_dmm"]

frozen = np.load(
    ROOT / "Artifact" / "fractional_incidence_codebook.npz"
)

print("\nFrozen artifact keys:")
print(frozen.files)

print("\nDistance from initialization:")
print(
    "RMS change:",
    np.sqrt(
        np.mean(
            (X - X_init) ** 2
        )
    )
)

# We deliberately do NOT assert equality to the frozen artifact,
# because Kumar's README explicitly says the short reference run
# is not claimed to reproduce it exactly.


print("\n======================================")
print("REFERENCE TRAINING VERDICT")
print("======================================")

if (
    X.shape == (256, 2, 16)
    and np.isfinite(X).all()
    and mean_ok
    and energy_ok
    and orthogonal_ok
):
    print(
        "SUCCESS: Kumar's reference incidence-training "
        "pipeline executes correctly and produces a valid "
        "256-message codebook satisfying the intended constraints."
    )

    print(
        "Exact equality with the released optimized codebook "
        "is NOT expected or claimed by the repository."
    )

else:
    print(
        "CHECK: The generated codebook violates one or more "
        "expected structural constraints."
    )


In [ ]:
# Cell 47 — Master reproducibility audit

import pandas as pd

print("=== BASELINE / RELEASED-RESULT SQM REPRODUCIBILITY AUDIT ===\n")

audit = pd.DataFrame([
    {
        "item": "Main head-to-head / Table III / Fig. 2",
        "status": "EXACT",
        "comment": "PER and 95% CI reproduced exactly across all SNRs."
    },
    {
        "item": "Coded OFDM controls / Table IV",
        "status": "EXACT",
        "comment": "Per-seed, summary and paired comparisons reproduced exactly."
    },
    {
        "item": "Adaptive revision / Table VII / Figs. 13-14",
        "status": "EXACT RELEASED",
        "comment": "Released adaptive implementation reproduces exactly; V2 corrections below address delay/S6/statistics."
    },
    {
        "item": "Fig. 4 delay/PDP robustness",
        "status": "EXACT",
        "comment": "Delay-spread and PDP mismatch sweeps reproduced exactly."
    },
    {
        "item": "Fig. 5 selective-margin radius",
        "status": "EXACT",
        "comment": "240-row theorem/radius experiment reproduced to floating-point precision."
    },
    {
        "item": "Fig. 6 payload scaling",
        "status": "EXACT",
        "comment": "Payload-scaling results reproduced exactly."
    },
    {
        "item": "Table V / Fig. 7 BMOCZ",
        "status": "EXACT",
        "comment": "Summary and per-seed BMOCZ results reproduced exactly."
    },
    {
        "item": "Fig. 8 observable nuisance rank",
        "status": "EXACT",
        "comment": "Regenerated from Kumar's codebook/core functions; spectrum and ranks reproduced exactly."
    },
    {
        "item": "Fig. 9 rank vs SNR",
        "status": "RELEASED DATA ONLY",
        "comment": "Released CSV available; exact command/seeds that generated it are not documented."
    },
    {
        "item": "Fig. 10 I/Q condition-number sensitivity",
        "status": "EXACT",
        "comment": "30-row experiment reproduced exactly using 1200 blocks/seed."
    },
    {
        "item": "Fig. 11 CFO-range sensitivity",
        "status": "RELEASED DATA ONLY",
        "comment": "Released CSV available; generating simulation script absent."
    },
    {
        "item": "Fig. 12 coarse-to-fine complexity",
        "status": "RELEASED DATA ONLY",
        "comment": "Released CSV available; generating simulation script absent."
    },
    {
        "item": "Fig. 15 adaptive sensitivity",
        "status": "EXACT DATA",
        "comment": "Scientific CSVs reproduce exactly; V2 below corrects the sensitivity CI convention."
    },
    {
        "item": "Grass-Lattice baseline",
        "status": "EXACT NUMERIC",
        "comment": "Numerical results exact; SQM label changed from Incidence-DMM to Incidence-trained SQM."
    },
    {
        "item": "Manifold geometry diagnostic",
        "status": "EXACT",
        "comment": "20-case geometry diagnostic reproduced."
    },
    {
        "item": "Tuned OFDM + oracle-gap",
        "status": "EXACT",
        "comment": "All 18 released rows reproduced exactly."
    },
    {
        "item": "Delay-only online controller",
        "status": "EXACT UNDERLYING",
        "comment": "Seed/segment results exact; released summary uses t(df=2), current script uses 1.96."
    },
    {
        "item": "Legacy adaptive controller",
        "status": "EXACT UNDERLYING",
        "comment": "Per-seed results exact; Kumar's t-based summary reconstructed exactly."
    },
    {
        "item": "Fig. 1 selective-margin samples",
        "status": "NOT EXACTLY REGENERABLE",
        "comment": "Released 3000-row dataset reconstructs figure, but current sampler uses different wrong-message sampling."
    },
    {
        "item": "Reference incidence-aware training",
        "status": "PIPELINE VERIFIED",
        "comment": "Training runs successfully and satisfies codebook constraints; exact frozen codebook reproduction not claimed."
    },
])

display(audit)

print("\n=== Status counts ===")
display(
    audit.groupby("status")
         .size()
         .rename("count")
         .reset_index()
)

out_file = (
    REPRO
    / "results"
    / "master_reproducibility_audit.csv"
)

audit.to_csv(out_file, index=False)

print("\nSaved:")
print(out_file)



## Part II — corrected analyses

The cells below address the specific mathematical/statistical/implementation issues identified after the released-result audit. They do **not** alter Kumar's original source files or the preserved released-data snapshot.


In [ ]:

# V2 correction workspace
from pathlib import Path
import sys, os, json, math, time
import numpy as np
import pandas as pd

V2_DIR = RESULTS / "v2_corrected"
V2_DIR.mkdir(parents=True, exist_ok=True)

# Make repository code importable without installing anything.
CODE_DIR = ROOT / "Code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

T_CRIT_DF2 = 4.302652729911275

print("=== V2 Corrected-Analysis Workspace ===")
print("Repository :", ROOT)
print("V2 outputs :", V2_DIR)
print("Original source files will NOT be modified.")


### V2.1 — Mathematical sanity checks for the manuscript model


In [ ]:

# Eq. (15) projector / rank-saturation sanity check

import numpy as np
from fractional_tdla_core import tdla_operators, weighted_nuisance_dictionary

artifact = np.load(ROOT / "Artifact" / "fractional_incidence_codebook.npz")
X = artifact["X_opt"]
Xd = X[:, 0] + 1j * X[:, 1]

L = Xd.shape[1]
_, powers, D = tdla_operators(L)
P_paths = len(powers)

# One actual centered TDL nuisance dictionary from the released implementation.
Phi = weighted_nuisance_dictionary(
    Xd[0],
    omega=0.0,
    powers=powers,
    D=D
)

rank_phi = np.linalg.matrix_rank(Phi)
Gram = Phi @ Phi.T
rank_gram = np.linalg.matrix_rank(Gram)

print("=== Full TDL projector rank audit ===")
print("Phi shape                   :", Phi.shape)
print("Number of physical paths P  :", P_paths)
print("2P                          :", 2 * P_paths)
print("Centered observation dim    :", L - 1)
print("rank(Phi)                   :", rank_phi)
print("Gram shape                  :", Gram.shape)
print("rank(Phi Phi^T)             :", rank_gram)

r_full_bound = min(2 * P_paths, L - 1)
print("Correct full-rank bound     : min(2P, L-1) =", r_full_bound)

# Ordinary inverse cannot be valid when Gram is singular.
ordinary_inverse_valid = rank_gram == Gram.shape[0]
print("Ordinary inverse valid?     :", ordinary_inverse_valid)

# Correct projector via pseudoinverse.
P_pinv = Phi.T @ np.linalg.pinv(Gram) @ Phi

# Equivalent compact-SVD row-space projector.
U, s, Vh = np.linalg.svd(Phi, full_matrices=False)
tol = np.max(Phi.shape) * np.finfo(float).eps * s[0]
rho = int(np.sum(s > tol))
Vrho = Vh[:rho].T
P_svd = Vrho @ Vrho.T

print("\n=== Correct projector checks ===")
print("SVD numerical rank          :", rho)
print("pinv vs compact-SVD max diff:",
      np.max(np.abs(P_pinv - P_svd)))
print("Symmetry error              :",
      np.max(np.abs(P_pinv - P_pinv.T)))
print("Idempotence error           :",
      np.max(np.abs(P_pinv @ P_pinv - P_pinv)))

assert not ordinary_inverse_valid, (
    "Unexpected: Phi Phi^T was full rank for this TDL-A case."
)
assert rho <= min(2 * P_paths, L - 1)
assert np.allclose(P_pinv, P_svd, rtol=1e-6, atol=1e-6)

print("\nV2 conclusion:")
print("Use the Moore-Penrose pseudoinverse or compact-SVD projector.")
print("The full explanation-space dimension is rank(Phi), not blindly 2P.")


In [ ]:

# Static I/Q-noise covariance convention sanity check
#
# Stated simulation model:
#     Y = A S0 + d 1^T + N
# with isotropic additive N injected AFTER A.
#
# Therefore received noise remains isotropic.  If a coherent receiver
# subsequently applies A^{-1}, the equalized noise covariance becomes
# proportional to A^{-1} A^{-T}.

rng = np.random.default_rng(20261024)

kappa = 4.0
theta = 0.37

s1 = np.sqrt(2.0) * kappa / np.sqrt(1.0 + kappa**2)
s2 = np.sqrt(2.0) / np.sqrt(1.0 + kappa**2)

R = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta),  np.cos(theta)],
])
A = R @ np.diag([s1, s2])
Ainv = np.linalg.inv(A)

N_mc = 200_000
sigma = 1.0
N = rng.normal(0.0, sigma, size=(N_mc, 2))

cov_received = np.cov(N, rowvar=False, ddof=0)
N_equalized = N @ Ainv.T
cov_equalized = np.cov(N_equalized, rowvar=False, ddof=0)

theory_received = sigma**2 * np.eye(2)
theory_equalized = sigma**2 * (Ainv @ Ainv.T)

# For comparison only: if noise were inserted BEFORE A,
# its covariance after A would instead be A A^T.
N_pre_frontend = N @ A.T
cov_pre_frontend_case = np.cov(N_pre_frontend, rowvar=False, ddof=0)
theory_pre_frontend_case = sigma**2 * (A @ A.T)

print("=== I/Q-noise covariance audit ===")
print("||A||_F^2 =", np.sum(A*A))

print("\nNoise added AFTER A (the manuscript/simulation model)")
print("Empirical received covariance:\n", cov_received)
print("Expected covariance:\n", theory_received)
print("Max abs error:",
      np.max(np.abs(cov_received - theory_received)))

print("\nAfter coherent equalization by A^{-1}")
print("Empirical covariance:\n", cov_equalized)
print("Expected covariance A^{-1}A^{-T}:\n", theory_equalized)
print("Max abs error:",
      np.max(np.abs(cov_equalized - theory_equalized)))

print("\nCounterfactual: noise inserted BEFORE A")
print("Empirical covariance:\n", cov_pre_frontend_case)
print("Expected covariance A A^T:\n", theory_pre_frontend_case)

assert np.allclose(cov_received, theory_received, atol=1.5e-2)
assert np.allclose(cov_equalized, theory_equalized, atol=3e-2)

print("\nV2 conclusion:")
print("For Y = A S0 + N, do not describe the received-noise covariance as A A^T.")
print("A^{-1}A^{-T} is the relevant covariance after receiver equalization.")


### V2.2 — Correct adaptation-delay definition (true regime boundaries only)


In [ ]:

# Correct adaptation delay:
# measure ONLY after internal regime boundaries (segments 1, 2, 3),
# not from the initial segment 0.

packet_file = ROOT / "Data" / "online_adaptive_packet_results.csv"
packet = pd.read_csv(packet_file)

canonical = packet[
    packet["sequence"].eq("canonical")
].copy()

# Derive the held-out oracle target rank directly from S8.
oracle_targets = (
    canonical[
        canonical["method"].eq("S8_regime_oracle_rank")
    ]
    .groupby(["seed", "segment"])["selected_rank"]
    .first()
)

def adaptation_delay_boundaries_only(
    selected,
    segments,
    target_by_segment,
    *,
    window=10
):
    selected = np.asarray(selected, dtype=int)
    segments = np.asarray(segments, dtype=int)

    unique_segments = sorted(np.unique(segments))
    internal_segments = unique_segments[1:]  # skip segment 0

    delays = []

    for seg in internal_segments:
        ix = np.flatnonzero(segments == seg)

        if isinstance(target_by_segment, dict):
            target = int(target_by_segment[int(seg)])
        else:
            target = int(target_by_segment[int(seg)])

        hit = None

        for k in range(0, len(ix) - window + 1):
            if np.all(selected[ix[k:k+window]] == target):
                hit = k
                break

        delays.append(
            float(hit if hit is not None else len(ix))
        )

    return float(np.mean(delays)), delays


delay_rows = []

for (seed, method), g in canonical.groupby(
    ["seed", "method"],
    sort=False
):
    if method in ("S0_fixed_r4", "S1_best_fixed"):
        continue

    g = g.sort_values("packet")
    target_map = {
        int(seg): int(oracle_targets.loc[(seed, seg)])
        for seg in sorted(g["segment"].unique())
    }

    mean_delay, per_boundary = adaptation_delay_boundaries_only(
        g["selected_rank"].to_numpy(),
        g["segment"].to_numpy(),
        target_map,
        window=10
    )

    delay_rows.append({
        "seed": int(seed),
        "method": method,
        "adaptation_delay_v2": mean_delay,
        "boundary_1_delay": per_boundary[0],
        "boundary_2_delay": per_boundary[1],
        "boundary_3_delay": per_boundary[2],
    })

delay_v2 = pd.DataFrame(delay_rows)

delay_summary_v2 = (
    delay_v2
    .groupby("method", sort=False)["adaptation_delay_v2"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

delay_summary_v2["95CI_halfwidth_t_df2"] = (
    T_CRIT_DF2
    * delay_summary_v2["std"]
    / np.sqrt(delay_summary_v2["count"])
)

# Compare with the released canonical seed-level metric.
released_seed_metrics = pd.read_csv(
    ROOT / "Data" / "online_adaptive_revision_seed_metrics.csv"
)
released_canonical = released_seed_metrics[
    released_seed_metrics["sequence"].eq("canonical")
][
    ["seed", "method", "mean_adaptation_delay_packets"]
].copy()

delay_compare = released_canonical.merge(
    delay_v2[["seed", "method", "adaptation_delay_v2"]],
    on=["seed", "method"],
    how="inner"
)

delay_compare["difference_v2_minus_released"] = (
    delay_compare["adaptation_delay_v2"]
    - delay_compare["mean_adaptation_delay_packets"]
)

delay_v2.to_csv(
    V2_DIR / "adaptive_delay_corrected_per_seed.csv",
    index=False
)
delay_summary_v2.to_csv(
    V2_DIR / "adaptive_delay_corrected_summary.csv",
    index=False
)

print("=== Corrected per-seed adaptation delay ===")
display(delay_v2)

print("\n=== Corrected three-seed summary ===")
display(delay_summary_v2)

print("\n=== Released vs corrected ===")
display(delay_compare)

print("\nExpected corrected canonical means:")
for _, r in delay_summary_v2.iterrows():
    print(f"{r['method']:38s} {r['mean']:.6f} packets")


### V2.3 — Figure 14: seed-level transition aggregation and t(df=2) confidence intervals


In [ ]:

# Correct Figure 14 statistical unit:
# 1) average the three transitions within each seed;
# 2) treat the three seeds as the independent replicates;
# 3) use t_{0.975,2} for the 95% half-width.

import matplotlib.pyplot as plt

transient = pd.read_csv(
    ROOT / "Data" / "online_transient_seed_metrics.csv"
)

transient = transient[
    transient["sequence"].eq("canonical")
].copy()

# Average transitions 1,2,3 within each seed first.
transient_seedavg = (
    transient
    .groupby(
        ["sequence", "seed", "method", "window"],
        sort=False,
        as_index=False
    )["PER"]
    .mean()
    .rename(columns={"PER": "PER_seed_transition_mean"})
)

transient_v2 = (
    transient_seedavg
    .groupby(
        ["sequence", "method", "window"],
        sort=False
    )["PER_seed_transition_mean"]
    .agg(["mean", "std", "count"])
    .reset_index()
)

transient_v2["PER_95CI_halfwidth_t_df2"] = (
    T_CRIT_DF2
    * transient_v2["std"]
    / np.sqrt(transient_v2["count"])
)

transient_seedavg.to_csv(
    V2_DIR / "online_transient_seed_averages_v2.csv",
    index=False
)
transient_v2.to_csv(
    V2_DIR / "online_transient_summary_v2.csv",
    index=False
)

print("=== Corrected Figure 14 summary ===")
display(transient_v2)


# ------------------------------------------------------------
# Corrected Figure 14 plot
# ------------------------------------------------------------

window_order = [
    "first_25",
    "first_50",
    "first_100",
    "steady_after_100",
]
window_labels = [
    "First 25",
    "First 50",
    "First 100",
    "Packets 100+",
]

methods = [
    "S2_SNR_class_lookup",
    "S4_window_confidence_residual",
    "S5_EWMA_confidence_residual",
    "S8_regime_oracle_rank",
]
labels = {
    "S2_SNR_class_lookup": "SNR lookup",
    "S4_window_confidence_residual": "Window gap + residual",
    "S5_EWMA_confidence_residual": "EWMA gap + residual",
    "S8_regime_oracle_rank": "State oracle",
}

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "Nimbus Roman", "DejaVu Serif"],
    "font.size": 11,
    "axes.labelsize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
})

fig, ax = plt.subplots(figsize=(6.6, 4.2), dpi=220)
x = np.arange(len(window_order))

styles = ["o-", "s--", "^-.", "D:"]

for method, style in zip(methods, styles):
    q = (
        transient_v2[
            transient_v2["method"].eq(method)
        ]
        .set_index("window")
        .reindex(window_order)
    )

    ax.errorbar(
        x,
        q["mean"],
        yerr=q["PER_95CI_halfwidth_t_df2"],
        fmt=style,
        capsize=3,
        markersize=5,
        markerfacecolor="white",
        linewidth=1.4,
        label=labels[method],
    )

ax.set_xticks(x)
ax.set_xticklabels(window_labels)
ax.set_xlabel("Packets after each regime transition")
ax.set_ylabel("Packet error rate")
ax.grid(True, alpha=0.20)
ax.legend(frameon=True, ncol=2)
fig.tight_layout()

fig14_v2 = V2_DIR / "fig_14_post_transition_per_v2.png"
fig.savefig(fig14_v2, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", fig14_v2)


### V2.4 — Figure 15: sensitivity rerun with t(df=2) intervals and corrected delay


In [ ]:

# Recompute Figure 15 sensitivity using the same frozen validation procedure,
# but use:
#   * t(df=2) 95% intervals, not 1.96;
#   * adaptation delay from the three true boundaries only.

import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from evaluate_online_adaptive import (
    make_trace,
    all_rank_stats,
    rank_maps,
    thresholds_from_validation,
    simulate_controller,
    RANKS,
)
from online_adaptive_diagnostics import pooled_validation
from run_final_headtohead import precompute_dmm_bases

SNRS_ADAPT = [5.0, 10.0, 15.0, 7.0]
DELAYS_ADAPT = [30.0, 60.0, 100.0, 30.0]
SEEDS_ADAPT = [13, 59, 151]

# Build the same nominal decoder banks.
t0 = make_trace(10, 777, SNRS_ADAPT, DELAYS_ADAPT)
Xd0, powers0, D0 = t0[5], t0[6], t0[7]
bases_sens = {
    r: precompute_dmm_bases(
        Xd0, powers0, D0, r, 4.0
    )[1]
    for r in RANKS
}

# Validation-only parameter selection, identical to released diagnostics.
vstats, vidx, vsnr, vdelay, vseg = pooled_validation(
    bases_sens
)
best_fixed, snr_map, regime_map, rank_bias, segdf = rank_maps(
    vstats, vidx, vseg, vsnr, vdelay
)
th = thresholds_from_validation(
    vstats, vidx, vseg, W=8
)

def corrected_controller_metrics(
    res,
    segs,
    regime_map
):
    target_map = {
        int(seg): int(regime_map[int(seg)])
        for seg in sorted(np.unique(segs))
    }

    mean_delay, _ = adaptation_delay_boundaries_only(
        res["selected"],
        segs,
        target_map,
        window=10,
    )

    return {
        "PER": float(np.mean(res["errors"])),
        "switch_rate": float(
            res["switches"] / max(1, len(res["selected"]) - 1)
        ),
        "mean_adaptation_delay_packets": mean_delay,
    }


sensitivity_rows = []

# Precompute each final seed ONCE.  The released diagnostics script
# recalculates the same trace/rank statistics for every W/alpha setting;
# caching them here makes the corrected V2 rerun much faster without
# changing the experiment.
final_sensitivity_cache = {}

for seed in tqdm(
    SEEDS_ADAPT,
    desc="Precomputing V2 sensitivity seeds",
    unit="seed"
):
    tr = make_trace(
        300,
        120000 + seed,
        SNRS_ADAPT,
        DELAYS_ADAPT
    )
    st = all_rank_stats(tr[0], bases_sens)

    final_sensitivity_cache[int(seed)] = {
        "trace": tr,
        "stats": st,
    }

settings = (
    [("S4_window", f"W={W}", W, np.nan)
     for W in [2, 4, 8, 16, 32]]
    +
    [("S5_EWMA", f"alpha={a:g}", np.nan, a)
     for a in [0.05, 0.10, 0.20, 0.35, 0.50]]
)

for policy, setting, W, alpha in tqdm(
    settings,
    desc="V2 sensitivity settings",
    unit="setting"
):
    vals = []

    for seed in SEEDS_ADAPT:
        cached = final_sensitivity_cache[int(seed)]
        tr = cached["trace"]
        st = cached["stats"]

        if policy == "S4_window":
            res = simulate_controller(
                st,
                tr[1],
                tr[4],
                "S4",
                W=int(W),
                tau_g=th["tau_g"],
                tau_e_hi=th["tau_e_hi"],
                tau_e_lo=th["tau_e_lo"],
                min_dwell=2,
            )
        else:
            res = simulate_controller(
                st,
                tr[1],
                tr[4],
                "S5",
                alpha=float(alpha),
                tau_g=th["tau_g"],
                tau_e_hi=th["tau_e_hi"],
                tau_e_lo=th["tau_e_lo"],
                min_dwell=2,
            )

        vals.append(
            corrected_controller_metrics(
                res,
                tr[4],
                regime_map
            )
        )

    per = np.array(
        [x["PER"] for x in vals],
        dtype=float
    )

    sensitivity_rows.append({
        "policy": policy,
        "setting": setting,
        "W": W,
        "alpha": alpha,
        "PER_mean": float(per.mean()),
        "PER_ci95_t_df2": float(
            T_CRIT_DF2
            * per.std(ddof=1)
            / np.sqrt(len(per))
        ),
        "switch_rate_mean": float(
            np.mean(
                [x["switch_rate"] for x in vals]
            )
        ),
        "adaptation_delay_mean_v2": float(
            np.mean([
                x["mean_adaptation_delay_packets"]
                for x in vals
            ])
        ),
    })

sensitivity_v2 = pd.DataFrame(sensitivity_rows)

sensitivity_v2.to_csv(
    V2_DIR / "online_controller_sensitivity_v2.csv",
    index=False
)

released_sensitivity = pd.read_csv(
    ROOT / "Data" / "online_controller_sensitivity.csv"
)

sens_compare = released_sensitivity.merge(
    sensitivity_v2,
    on=["policy", "setting"],
    how="inner",
    suffixes=("_released", "_v2")
)

print("=== Corrected sensitivity results ===")
display(sensitivity_v2)

print("\n=== Released vs V2 sensitivity ===")
display(
    sens_compare[
        [
            "policy",
            "setting",
            "PER_mean_released",
            "PER_mean_v2",
            "PER_ci95",
            "PER_ci95_t_df2",
            "adaptation_delay_mean",
            "adaptation_delay_mean_v2",
        ]
    ]
)


# ------------------------------------------------------------
# Corrected Figure 15
# ------------------------------------------------------------

s4 = sensitivity_v2[
    sensitivity_v2["policy"].eq("S4_window")
].sort_values("W")

s5 = sensitivity_v2[
    sensitivity_v2["policy"].eq("S5_EWMA")
].sort_values("alpha")

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "Nimbus Roman", "DejaVu Serif"],
    "font.size": 10.5,
    "axes.labelsize": 12,
    "axes.titlesize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9.5,
    "axes.linewidth": 0.8,
})

fig, axes = plt.subplots(
    1, 2,
    figsize=(8.0, 3.6),
    dpi=220,
    sharey=True
)

axes[0].errorbar(
    s4["W"],
    s4["PER_mean"],
    yerr=s4["PER_ci95_t_df2"],
    fmt="o-",
    markersize=4.8,
    capsize=3,
    markerfacecolor="white",
    linewidth=1.5,
)
axes[0].set_xlabel("Window size $W$")
axes[0].set_ylabel("Packet error rate")
axes[0].set_title("Window-based controller")
axes[0].set_xticks(s4["W"])
axes[0].grid(True, alpha=0.20)

axes[1].errorbar(
    s5["alpha"],
    s5["PER_mean"],
    yerr=s5["PER_ci95_t_df2"],
    fmt="s--",
    color="tab:red",
    markersize=4.6,
    capsize=3,
    markerfacecolor="white",
    linewidth=1.5,
)
axes[1].set_xlabel(r"EWMA factor $\alpha$")
axes[1].set_title("EWMA-based controller")
axes[1].set_xticks(s5["alpha"])
axes[1].set_xticklabels(
    ["0.05", "0.10", "0.20", "0.35", "0.50"]
)
axes[1].grid(True, alpha=0.20)

fig.tight_layout()
fig15_v2 = V2_DIR / "fig_15_controller_sensitivity_v2.png"
fig.savefig(fig15_v2, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", fig15_v2)


### V2.5 — S6 genuine same-packet confidence-triggered refinement


In [ ]:

# Correct S6 semantics:
# the adjacent-rank comparison now changes the CURRENT packet's decoded
# prediction, selected rank, residual/gap record, and rank-aware MAC count.

from evaluate_online_adaptive import (
    make_trace,
    all_rank_stats,
    RANKS,
    FULL_EVALS,
)
import evaluate_online_adaptive_revision as adaptive_revision
from tqdm.auto import tqdm

L_BLOCK = 16

def simulate_s6_same_packet(
    stats,
    idx,
    *,
    tau_g,
    rank_bias,
    initial_rank=4
):
    n = len(idx)
    rank_to_i = {r: i for i, r in enumerate(RANKS)}

    selected = np.empty(n, dtype=int)
    pred = np.empty(n, dtype=int)
    gaps = np.empty(n, dtype=float)
    residuals = np.empty(n, dtype=float)
    evals = np.full(n, FULL_EVALS, dtype=float)
    approx_macs = np.empty(n, dtype=float)

    active = int(initial_rank)
    switches = 0

    for t in range(n):
        base_rank = active

        # Trigger is evaluated from the current active rank.
        base_gap = float(stats[base_rank][1][t])

        candidates = [base_rank]

        if base_gap < tau_g:
            i = rank_to_i[base_rank]

            if i > 0:
                candidates.append(RANKS[i - 1])

            if i < len(RANKS) - 1:
                candidates.append(RANKS[i + 1])

            def util(r):
                fit = 1.0 - float(stats[r][2][t])
                return (
                    float(stats[r][1][t])
                    - (rank_bias or {}).get(r, 0.0)
                    + 0.25 * (
                        fit
                        - (rank_bias or {}).get(
                            ("fit", r), 0.0
                        )
                    )
                )

            chosen = max(candidates, key=util)
        else:
            chosen = base_rank

        # SAME-PACKET correction:
        # use the winning candidate to decode packet t.
        selected[t] = int(chosen)
        pred[t] = int(stats[chosen][0][t])
        gaps[t] = float(stats[chosen][1][t])
        residuals[t] = float(stats[chosen][2][t])

        if chosen != base_rank:
            switches += 1

        # Carry the chosen rank forward.
        active = int(chosen)

        # Every candidate was scored over the full candidate/CFO bank.
        evals[t] = FULL_EVALS * len(candidates)

        # Rank-aware approximate MAC count:
        # sum cost over the ranks actually evaluated.
        approx_macs[t] = (
            FULL_EVALS
            * sum(2 * L_BLOCK * int(r) for r in candidates)
        )

    return {
        "scheme": "S6_same_packet_v2",
        "errors": (pred != idx).astype(float),
        "pred": pred,
        "selected": selected,
        "gap": gaps,
        "residual": residuals,
        "evals": evals,
        "approx_macs": approx_macs,
        "switches": int(switches),
    }


print("=== Calibrating corrected S6 on validation traces ===")

bases_s6 = adaptive_revision.build_bases()

(
    th_s6,
    rank_bias_s6,
    vstats_s6,
    vidx_s6,
    vsegs_s6,
    vsnrs_s6,
    vdelays_s6,
) = adaptive_revision.thresholds_and_rank_bias(
    bases_s6
)

s6_grid = np.quantile(
    np.concatenate(
        [vstats_s6[r][1] for r in RANKS]
    ),
    [0.10, 0.20, 0.30, 0.40]
)

s6_validation_rows = []

for tau in s6_grid:
    rr = simulate_s6_same_packet(
        vstats_s6,
        vidx_s6,
        tau_g=float(tau),
        rank_bias=rank_bias_s6,
        initial_rank=4,
    )

    s6_validation_rows.append({
        "tau_gap": float(tau),
        "validation_PER": float(
            rr["errors"].mean()
        ),
    })

s6_validation = pd.DataFrame(s6_validation_rows)
best_idx = s6_validation["validation_PER"].idxmin()
s6_tau_v2 = float(
    s6_validation.loc[best_idx, "tau_gap"]
)

display(s6_validation)
print("Selected V2 S6 threshold:", s6_tau_v2)


# ------------------------------------------------------------
# Final canonical three-seed rerun
# ------------------------------------------------------------

# Oracle target ranks are derived from the independently reproduced S8 trace.
target_by_seed_segment = oracle_targets

s6_seed_rows = []
s6_packet_rows = []

for seed in tqdm(
    [13, 59, 151],
    desc="V2 S6 final seeds",
    unit="seed"
):
    tr = make_trace(
        300,
        120000 + seed,
        [5.0, 10.0, 15.0, 7.0],
        [30.0, 60.0, 100.0, 30.0],
    )

    Y, idx, snrs, delays, segs = tr[:5]
    stats = all_rank_stats(Y, bases_s6)

    res = simulate_s6_same_packet(
        stats,
        idx,
        tau_g=s6_tau_v2,
        rank_bias=rank_bias_s6,
        initial_rank=4,
    )

    target_map = {
        int(seg): int(
            target_by_seed_segment.loc[
                (seed, seg)
            ]
        )
        for seg in sorted(np.unique(segs))
    }

    delay_mean, delay_each = (
        adaptation_delay_boundaries_only(
            res["selected"],
            segs,
            target_map,
            window=10,
        )
    )

    transitions = np.r_[
        False,
        res["selected"][1:]
        != res["selected"][:-1]
    ]

    s6_seed_rows.append({
        "sequence": "canonical",
        "seed": int(seed),
        "method": "S6_confidence_triggered_refinement",
        "blocks": len(idx),
        "PER": float(res["errors"].mean()),
        "mean_selected_rank": float(
            res["selected"].mean()
        ),
        "controller_switches": int(
            res["switches"]
        ),
        "controller_switch_rate": float(
            res["switches"]
            / max(1, len(idx) - 1)
        ),
        "selected_rank_transitions": int(
            transitions.sum()
        ),
        "switch_rate": float(
            transitions.sum()
            / max(1, len(idx) - 1)
        ),
        "mean_adaptation_delay_packets":
            float(delay_mean),
        "mean_score_evals": float(
            res["evals"].mean()
        ),
        "p95_score_evals": float(
            np.quantile(res["evals"], 0.95)
        ),
        "mean_approx_macs": float(
            res["approx_macs"].mean()
        ),
        "p95_approx_macs": float(
            np.quantile(
                res["approx_macs"], 0.95
            )
        ),
    })

    for t in range(len(idx)):
        s6_packet_rows.append({
            "sequence": "canonical",
            "seed": int(seed),
            "method":
                "S6_confidence_triggered_refinement",
            "packet": int(t),
            "segment": int(segs[t]),
            "snr_db": float(snrs[t]),
            "delay_ns": float(delays[t]),
            "error": int(res["errors"][t]),
            "selected_rank":
                int(res["selected"][t]),
            "selected_rank_transition":
                int(transitions[t]),
            "score_evals":
                float(res["evals"][t]),
            "approx_macs":
                float(res["approx_macs"][t]),
        })

s6_seed_v2 = pd.DataFrame(s6_seed_rows)
s6_packet_v2 = pd.DataFrame(s6_packet_rows)

s6_seed_v2.to_csv(
    V2_DIR / "s6_same_packet_seed_metrics_v2.csv",
    index=False
)
s6_packet_v2.to_csv(
    V2_DIR / "s6_same_packet_packet_results_v2.csv",
    index=False
)

print("\n=== Corrected S6 seed metrics ===")
display(s6_seed_v2)


# ------------------------------------------------------------
# Build corrected canonical Table VII dataset:
# - use reproduced released metrics for unchanged schemes;
# - replace adaptation delay for all adaptive schemes;
# - replace all S6 metrics with same-packet V2 results.
# ------------------------------------------------------------

table7_seed_v2 = (
    released_seed_metrics[
        released_seed_metrics["sequence"].eq(
            "canonical"
        )
    ]
    .copy()
)

# Correct adaptation delay for unchanged adaptive methods.
delay_lookup = delay_v2.set_index(
    ["seed", "method"]
)["adaptation_delay_v2"]

for i, row in table7_seed_v2.iterrows():
    key = (int(row["seed"]), row["method"])

    if key in delay_lookup.index:
        table7_seed_v2.loc[
            i,
            "mean_adaptation_delay_packets"
        ] = float(delay_lookup.loc[key])

# Replace S6 rows fully.
s6_cols = [
    c for c in s6_seed_v2.columns
    if c in table7_seed_v2.columns
]

for _, r in s6_seed_v2.iterrows():
    mask = (
        table7_seed_v2["seed"].eq(r["seed"])
        &
        table7_seed_v2["method"].eq(r["method"])
    )

    for c in s6_cols:
        table7_seed_v2.loc[mask, c] = r[c]


# Three-seed t-based summary.
summary_rows = []

summary_metrics = [
    "PER",
    "mean_selected_rank",
    "controller_switch_rate",
    "switch_rate",
    "mean_adaptation_delay_packets",
    "mean_score_evals",
    "p95_score_evals",
    "mean_approx_macs",
    "p95_approx_macs",
]

for method, g in table7_seed_v2.groupby(
    "method",
    sort=False
):
    row = {
        "method": method,
        "seeds": len(g),
        "blocks_total": int(g["blocks"].sum()),
    }

    for col in summary_metrics:
        vals = (
            g[col]
            .dropna()
            .to_numpy(dtype=float)
        )

        row[col] = (
            float(vals.mean())
            if len(vals)
            else np.nan
        )

        row[col + "_95CI_halfwidth"] = (
            float(
                T_CRIT_DF2
                * vals.std(ddof=1)
                / np.sqrt(len(vals))
            )
            if len(vals) > 1
            else 0.0
        )

    summary_rows.append(row)

table7_summary_v2 = pd.DataFrame(
    summary_rows
)

table7_seed_v2.to_csv(
    V2_DIR / "table_vii_seed_metrics_v2.csv",
    index=False
)
table7_summary_v2.to_csv(
    V2_DIR / "table_vii_summary_v2.csv",
    index=False
)

print("\n=== Corrected Table VII summary ===")
display(table7_summary_v2)

old_s6 = released_seed_metrics[
    released_seed_metrics["sequence"].eq("canonical")
    &
    released_seed_metrics["method"].eq(
        "S6_confidence_triggered_refinement"
    )
]

print("\n=== Released S6 vs same-packet V2 ===")
display(
    pd.concat([
        old_s6.assign(version="released"),
        s6_seed_v2.assign(version="V2 same-packet"),
    ], ignore_index=True)
)


### V2.6 — All-message selective-margin screen (all 255 wrong messages)


In [ ]:

# The released Figure-1 diagnostic sampled a subset of wrong messages.
# V2 adds a stricter screen over ALL 255 wrong messages for each true message.
#
# This does not overwrite the released Figure-1 data.  It provides an
# all-message diagnostic under the same evaluator/CFO grid.

import matplotlib.pyplot as plt
import subprocess
import sys
from tqdm.auto import tqdm

MARGIN_V2 = V2_DIR / "selective_margin_all_wrong"
MARGIN_V2.mkdir(parents=True, exist_ok=True)

margin_all_file = (
    MARGIN_V2
    / "selective_margin_samples_all_255_wrong_v2.csv"
)
margin_log = (
    MARGIN_V2
    / "selective_margin_all_255_wrong_v2.log"
)

cmd = [
    sys.executable,
    "Code/evaluate_selective_margin.py",
    "--cases", "500",
    "--wrong", "255",
    "--ranks", "2,3,4,5,6,8",
    "--seed", "20260918",
    "--output",
    str(margin_all_file.relative_to(ROOT)),
]

print("Running stricter all-message selective-margin screen:")
print(" ".join(cmd))

start = time.time()

with open(margin_log, "w") as log:
    process = subprocess.Popen(
        cmd,
        cwd=ROOT,
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
        env=os.environ.copy(),
    )

    with tqdm(
        desc="All-255-wrong margin screen",
        unit="s",
        bar_format="{desc}: {n:.0f} s elapsed",
    ) as pbar:
        while process.poll() is None:
            time.sleep(1)
            pbar.update(1)

return_code = process.returncode
elapsed = time.time() - start

print("Return code :", return_code)
print(f"Runtime     : {elapsed:.1f} s")
print("Output      :", margin_all_file)
print("Log         :", margin_log)

if return_code != 0:
    print(margin_log.read_text()[-5000:])
    raise RuntimeError(
        "All-message selective-margin screen failed."
    )

margin_all = pd.read_csv(margin_all_file)

# Radius associated with the measured all-message selective margin.
margin_all["selective_margin_radius_norm"] = (
    margin_all["selective_margin_norm"] / 2.0
)

margin_all_summary = (
    margin_all
    .groupby("rank", as_index=False)
    .agg(
        mean_true_mismatch=(
            "true_mismatch_norm", "mean"
        ),
        p90_true_mismatch=(
            "true_mismatch_norm",
            lambda x: x.quantile(0.90)
        ),
        mean_wrong_distance=(
            "nearest_wrong_dist_norm", "mean"
        ),
        p01_wrong_distance=(
            "nearest_wrong_dist_norm",
            lambda x: x.quantile(0.01)
        ),
        mean_selective_margin=(
            "selective_margin_norm", "mean"
        ),
        p01_selective_margin=(
            "selective_margin_norm",
            lambda x: x.quantile(0.01)
        ),
        frac_positive_margin=(
            "selective_margin_norm",
            lambda x: float((x > 0).mean())
        ),
        mean_margin_radius=(
            "selective_margin_radius_norm",
            "mean"
        ),
    )
)

margin_all_summary.to_csv(
    MARGIN_V2
    / "selective_margin_summary_all_255_wrong_v2.csv",
    index=False
)

print("\n=== All-message selective-margin summary ===")
display(margin_all_summary)

# Compare against released sampled-wrong-message summary.
released_margin_summary = pd.read_csv(
    REPRO
    / "Data_Kumar_Original"
    / "selective_margin_summary.csv"
)

margin_summary_compare = (
    released_margin_summary.merge(
        margin_all_summary,
        on="rank",
        suffixes=("_released_sampled", "_all255_v2")
    )
)

print("\n=== Released sampled vs all-255 V2 ===")
display(margin_summary_compare)

# Plot the all-message decomposition.
fig, ax = plt.subplots(figsize=(6.4, 4.0), dpi=220)

ax.plot(
    margin_all_summary["rank"],
    margin_all_summary["mean_true_mismatch"],
    "o-",
    label="Mean true mismatch",
)
ax.plot(
    margin_all_summary["rank"],
    margin_all_summary["mean_wrong_distance"],
    "s--",
    label="Mean nearest-wrong distance",
)
ax.plot(
    margin_all_summary["rank"],
    margin_all_summary["mean_selective_margin"],
    "^-.",
    label="Mean selective margin",
)

ax.axhline(0.0, linewidth=0.8, color="black")
ax.set_xlabel("Retained nuisance rank")
ax.set_ylabel("Normalized distance")
ax.grid(True, alpha=0.20)
ax.legend(frameon=True)
fig.tight_layout()

fig1_all_v2 = (
    MARGIN_V2
    / "fig_01_selective_margin_all_255_wrong_v2.png"
)
fig.savefig(
    fig1_all_v2,
    dpi=300,
    bbox_inches="tight"
)
plt.show()

print("Saved:", fig1_all_v2)


### V2.7 — Final corrected reproducibility ledger


In [ ]:

# Final V2 ledger: distinguish exact reproduction from corrected analysis.

audit_v2 = pd.DataFrame([
    {
        "item": "Main head-to-head / Table III / Fig. 2",
        "released_reproduction": "EXACT",
        "v2_action": "No correction required",
        "comment": "All 15 rows, PER and 95% CI reproduced exactly."
    },
    {
        "item": "Coded OFDM controls / Table IV",
        "released_reproduction": "EXACT",
        "v2_action": "No correction required",
        "comment": "Per-seed, summary and paired comparisons reproduced exactly."
    },
    {
        "item": "Full TDL projector Eq. (15)",
        "released_reproduction": "FORMULATION ISSUE",
        "v2_action": "PSEUDOINVERSE / COMPACT-SVD VERIFIED",
        "comment": "Phi Phi^T is singular for P=23,L=16; V2 verifies pinv and compact-SVD projectors agree."
    },
    {
        "item": "Broad explanation-space dimension",
        "released_reproduction": "FORMULATION ISSUE",
        "v2_action": "RANK SATURATION VERIFIED",
        "comment": "Use rank(Phi) <= min(2P,L-1), not unconditional r=2P."
    },
    {
        "item": "Static I/Q noise covariance",
        "released_reproduction": "MODEL-WORDING ISSUE",
        "v2_action": "COVARIANCE CONVENTION VERIFIED",
        "comment": "Noise added after A remains isotropic; after A^{-1} equalization covariance is A^{-1}A^{-T}."
    },
    {
        "item": "Adaptive released raw outputs",
        "released_reproduction": "EXACT",
        "v2_action": "Preserved as historical baseline",
        "comment": "Packet/seed/segment/transient files reproduce the released implementation exactly."
    },
    {
        "item": "Adaptation delay / Table VII",
        "released_reproduction": "EXACT IMPLEMENTATION, DEFINITION MISMATCH",
        "v2_action": "CORRECTED",
        "comment": "V2 excludes segment 0 and averages only the three true regime boundaries."
    },
    {
        "item": "S6 confidence-triggered refinement",
        "released_reproduction": "EXACT IMPLEMENTATION, SEMANTIC MISMATCH",
        "v2_action": "CORRECTED AND RERUN",
        "comment": "V2 applies the selected adjacent rank to the same packet and uses rank-aware approximate MAC accounting."
    },
    {
        "item": "Fig. 14 transient PER",
        "released_reproduction": "EXACT RELEASED CALCULATION",
        "v2_action": "CORRECTED STATISTICS",
        "comment": "V2 averages transitions within seed, then uses three independent seeds with t(df=2)."
    },
    {
        "item": "Fig. 15 adaptive sensitivity",
        "released_reproduction": "EXACT MEANS",
        "v2_action": "CORRECTED STATISTICS",
        "comment": "V2 replaces 1.96 with t(df=2) and uses corrected boundary-only adaptation delay."
    },
    {
        "item": "Fig. 1 selective-margin sampling",
        "released_reproduction": "NOT EXACTLY REGENERABLE",
        "v2_action": "ALL-255-WRONG SCREEN ADDED",
        "comment": "Released CSV remains reconstructable; V2 adds a stricter all-message diagnostic using the current evaluator."
    },
    {
        "item": "Fig. 4 delay/PDP robustness",
        "released_reproduction": "EXACT",
        "v2_action": "No correction required",
        "comment": "Released sweeps reproduce exactly."
    },
    {
        "item": "Fig. 5 selective-margin radius screen",
        "released_reproduction": "EXACT NUMERIC",
        "v2_action": "Terminology clarified",
        "comment": "Treat sampled-wrong-message result as a diagnostic margin; V2 all-message screen is stricter."
    },
    {
        "item": "Fig. 6 payload scaling",
        "released_reproduction": "EXACT",
        "v2_action": "No correction required",
        "comment": "Payload scaling reproduces exactly."
    },
    {
        "item": "Table V / Fig. 7 BMOCZ",
        "released_reproduction": "EXACT",
        "v2_action": "No correction required",
        "comment": "Summary and per-seed results reproduce exactly."
    },
    {
        "item": "Fig. 8 observable nuisance rank",
        "released_reproduction": "EXACT",
        "v2_action": "No correction required",
        "comment": "Spectrum/effective-rank diagnostics reproduce to floating-point precision."
    },
    {
        "item": "Fig. 9 rank vs SNR",
        "released_reproduction": "RELEASED DATA ONLY",
        "v2_action": "Documentation issue remains",
        "comment": "Figure reconstructs from CSV; exact publication-generation command/seeds remain undocumented."
    },
    {
        "item": "Fig. 10 I/Q sensitivity",
        "released_reproduction": "EXACT",
        "v2_action": "No correction required",
        "comment": "Final 1200-block condition-number sweep reproduces exactly."
    },
    {
        "item": "Fig. 11 CFO-range sensitivity",
        "released_reproduction": "RELEASED DATA ONLY",
        "v2_action": "Documentation issue remains",
        "comment": "Figure reconstructs from released data; exact publication invocation not independently verified."
    },
    {
        "item": "Fig. 12 coarse-to-fine complexity",
        "released_reproduction": "RELEASED DATA ONLY",
        "v2_action": "Documentation issue remains",
        "comment": "Figure reconstructs from released CSV; end-to-end generation is less fully documented."
    },
    {
        "item": "Reference incidence-aware training",
        "released_reproduction": "PIPELINE VERIFIED",
        "v2_action": "No deterministic-equality claim",
        "comment": "Training creates a valid constrained codebook; exact frozen optimized artifact is not claimed."
    },
])

display(audit_v2)

audit_v2.to_csv(
    V2_DIR / "master_reproducibility_audit_v2.csv",
    index=False
)

print("\n=== corrected outputs ===")
for p in sorted(V2_DIR.rglob("*")):
    if p.is_file():
        print(p.relative_to(ROOT))

print("\nV2 COMPLETE")
print("All corrected artifacts are isolated under:")
print(V2_DIR)


# Part III — Manuscript-order inline figures and hybrid result display

This section turns the audit into a **scientific results notebook**.  
When executed, it displays the article figures inline in manuscript order and saves publication-resolution PNG/PDF copies under:

`Reproduction_Work/results/inline_figures/`

Data precedence is:

1. freshly reproduced numerical results where available;
2. corrected V2 outputs where the audit identified a real issue;
3. released CSV reconstruction where the original generation script is unavailable.

The source used for each figure is printed directly below the figure.


In [ ]:
# Common publication-style plotting setup

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

INLINE_DIR = RESULTS / "inline_figures"
INLINE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": [
        "Times New Roman", "Times", "Nimbus Roman",
        "DejaVu Serif"
    ],
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "legend.fontsize": 9,
    "axes.linewidth": 0.8,
    "lines.linewidth": 1.8,
    "figure.dpi": 120,
    "savefig.dpi": 300,
})

def save_inline(fig, stem):
    png = INLINE_DIR / f"{stem}.png"
    pdf = INLINE_DIR / f"{stem}.pdf"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    plt.show()
    print("Saved:", png)
    print("Saved:", pdf)

def first_existing(*paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            return p
    raise FileNotFoundError(
        "None of the candidate files exists:\n" +
        "\n".join(str(Path(p)) for p in paths)
    )

print("Inline figure directory:", INLINE_DIR)


## Figure 1 and Table IX — Selective-margin decomposition


In [ ]:
# Published Figure 1 / Table IX reconstruction from released numerical artifact.
# The current evaluator does not regenerate the exact publication sample set.

fig1_file = first_existing(
    REPRO / "Data_Kumar_Original" / "selective_margin_summary.csv",
    ROOT / "Data" / "selective_margin_summary.csv",
)
f1 = pd.read_csv(fig1_file).sort_values("rank")

display(
    f1[[
        "rank",
        "mean_true_mismatch",
        "mean_wrong_distance",
        "mean_selective_margin",
        "p01_selective_margin",
    ]]
)

fig, ax = plt.subplots(figsize=(5.4, 3.35))
ax.plot(f1["rank"], f1["mean_true_mismatch"], marker="o", label="True-model mismatch")
ax.plot(f1["rank"], f1["mean_wrong_distance"], marker="s", label="Nearest wrong distance")
ax.plot(f1["rank"], f1["mean_selective_margin"], marker="^", label="Selective margin")
ax.set_xlabel("Retained nuisance rank")
ax.set_ylabel("Normalized distance")
ax.grid(True, alpha=0.25)
ax.legend(frameon=False)
fig.tight_layout()
save_inline(fig, "fig01_selective_margin_decomposition")

print("Source:", fig1_file)
print("Status: RELEASED-DATA RECONSTRUCTION (publication sample generation is not exactly recoverable).")

all_wrong = V2_DIR / "selective_margin_all_wrong" / "selective_margin_summary_all_255_wrong_v2.csv"
if all_wrong.exists():
    print("\nV2 all-255-wrong-message diagnostic is also available:")
    display(pd.read_csv(all_wrong))


## Figure 2 and Table III — Main exact fractional-delay TDL-A comparison


In [ ]:
fig2_file = first_existing(
    RESULTS / "main_reproduced_summary.csv",
    REPRO / "Data_Kumar_Original" / "final_rerun_fractional_tdla_headtohead.csv",
)
f2 = pd.read_csv(fig2_file)

display(
    f2[["snr_db", "method", "PER_mean", "PER_95CI_halfwidth"]]
    .sort_values(["snr_db", "method"])
)

label_map = {
    "CP-OFDM TDL-LMMSE": "Practical CP-OFDM",
    "CP-OFDM full oracle ML": "Full-oracle CP-OFDM",
    "Selective SQM": "SQM, rank 4",
}

fig, ax = plt.subplots(figsize=(5.4, 3.5))
for method, g in f2.groupby("method", sort=False):
    g = g.sort_values("snr_db")
    ax.errorbar(
        g["snr_db"],
        g["PER_mean"],
        yerr=g["PER_95CI_halfwidth"],
        marker="o",
        capsize=3,
        label=label_map.get(method, method),
    )
ax.set_yscale("log")
ax.set_xlabel("SNR (dB)")
ax.set_ylabel("Packet error rate")
ax.grid(True, which="both", alpha=0.22)
ax.legend(frameon=False)
fig.tight_layout()
save_inline(fig, "fig02_main_tdl_a_per_vs_snr")

print("Source:", fig2_file)
print("Status: EXACT REPRODUCTION.")


## Figure 3 and Table IV — Coded OFDM controls


In [ ]:
fig3_file = first_existing(
    RESULTS / "table_iv_coded_ofdm" / "coded_reproduced_summary.csv",
    REPRO / "Data_Kumar_Original" / "reviewer_baselines_summary.csv",
)
f3 = pd.read_csv(fig3_file)

ci_col = (
    "PER_95CI_halfwidth_t_df2"
    if "PER_95CI_halfwidth_t_df2" in f3.columns
    else "PER_95CI_halfwidth"
)

display(f3)

name_map = {
    "SQM_rank4": "SQM, rank 4",
    "OFDM_pilot_free_marginal": "Pilot-free coded\n(TDL-A marginal ML)",
    "OFDM_rate_matched_pilot_coded": "Pilot-aided coded\n(8-bit / 16 samples)",
    "OFDM_practical_unknown_IQ_DC": "Practical OFDM\n(unknown I/Q/DC)",
}

p = f3.sort_values("PER_mean").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(6.0, 3.4))
y = np.arange(len(p))
ax.errorbar(
    p["PER_mean"],
    y,
    xerr=p[ci_col],
    fmt="o",
    capsize=3,
)
ax.set_yticks(y)
ax.set_yticklabels([name_map.get(x, x) for x in p["method"]])
ax.set_xlabel("PER at 10 dB")
ax.grid(True, axis="x", alpha=0.25)
fig.tight_layout()
save_inline(fig, "fig03_coded_ofdm_controls")

print("Source:", fig3_file)
print("Status: EXACT REPRODUCTION.")


## Figure 4 — Delay-spread and PDP mismatch robustness


In [ ]:
delay_file = first_existing(
    RESULTS / "figure4_robustness" / "delay_spread_reproduced.csv",
    REPRO / "Data_Kumar_Original" / "delay_spread_sweep.csv",
)
pdp_file = first_existing(
    RESULTS / "figure4_robustness" / "pdp_mismatch_reproduced.csv",
    REPRO / "Data_Kumar_Original" / "pdp_mismatch_sweep.csv",
)

delay = pd.read_csv(delay_file)
pdp = pd.read_csv(pdp_file)

fig, axes = plt.subplots(1, 2, figsize=(8.0, 3.35), sharey=True)

for method, g in delay.groupby("method", sort=False):
    g = g.sort_values("delay_spread_ns")
    axes[0].plot(g["delay_spread_ns"], g["PER"], marker="o", label=method)
axes[0].set_xlabel("True delay scale (ns)")
axes[0].set_ylabel("PER at 10 dB")
axes[0].grid(True, alpha=0.25)

for method, g in pdp.groupby("method", sort=False):
    g = g.sort_values("pdp_power_exponent")
    axes[1].plot(g["pdp_power_exponent"], g["PER"], marker="o", label=method)
axes[1].set_xlabel("PDP power exponent")
axes[1].grid(True, alpha=0.25)
axes[1].legend(frameon=False, fontsize=8)

fig.tight_layout()
save_inline(fig, "fig04_delay_and_pdp_robustness")

print("Sources:", delay_file, "and", pdp_file)
print("Status: EXACT REPRODUCTION.")


## Figure 5 — Selective-margin radius test


In [ ]:
fig5_file = first_existing(
    RESULTS / "figure5_margin_radius" / "theorem3_validation_reproduced.csv",
    REPRO / "Data_Kumar_Original" / "theorem3_validation.csv",
)
f5 = pd.read_csv(fig5_file)

display(f5)

fig, ax = plt.subplots(figsize=(5.2, 3.25))
for rank, g in f5.groupby("rank"):
    g = g.sort_values("noise_fraction_of_certified_radius")
    ax.plot(
        g["noise_fraction_of_certified_radius"],
        g["empirical_error_rate"],
        marker="o",
        label=f"r={int(rank)}",
    )
ax.set_xlabel("Noise norm / selective-margin radius")
ax.set_ylabel("Empirical error rate")
ymax = max(0.0020, 1.15 * f5["empirical_error_rate"].max())
ax.set_ylim(0, ymax)
ax.grid(True, alpha=0.25)
ax.legend(frameon=False)
fig.tight_layout()
save_inline(fig, "fig05_selective_margin_radius")

print("Source:", fig5_file)
print("Status: EXACT REPRODUCTION.")


## Figure 6 — Payload scaling


In [ ]:
fig6_file = first_existing(
    RESULTS / "figure6_scaling" / "scaling_10dB_reproduced.csv",
    REPRO / "Data_Kumar_Original" / "scaling_10dB.csv",
)
f6 = pd.read_csv(fig6_file).sort_values("bits")
display(f6)

fig, axes = plt.subplots(1, 2, figsize=(8.0, 3.2))

axes[0].plot(f6["bits"], f6["PER"], marker="o")
axes[0].set_xlabel("Payload bits")
axes[0].set_ylabel("PER at 10 dB")
axes[0].grid(True, alpha=0.25)

axes[1].plot(f6["bits"], f6["score_evaluations"], marker="o")
axes[1].set_yscale("log")
axes[1].set_xlabel("Payload bits")
axes[1].set_ylabel("Full score evaluations")
axes[1].grid(True, which="both", alpha=0.25)

fig.tight_layout()
save_inline(fig, "fig06_payload_scaling")

print("Source:", fig6_file)
print("Status: EXACT REPRODUCTION.")


## Figure 7 and Table V — BMOCZ comparison


In [ ]:
fig7_file = first_existing(
    RESULTS / "figure7_bmocz" / "dmm_vs_bmocz_reproduced.csv",
    REPRO / "Data_Kumar_Original" / "dmm_vs_bmocz_rerun.csv",
)
f7 = pd.read_csv(fig7_file)
display(f7)

fig, ax = plt.subplots(figsize=(5.4, 3.5))
for method, g in f7.groupby("method", sort=False):
    g = g.sort_values("snr_db")
    ax.errorbar(
        g["snr_db"],
        g["PER_mean"],
        yerr=g["PER_95CI_halfwidth"],
        marker="o",
        capsize=3,
        label=method,
    )
ax.set_yscale("log")
ax.set_xlabel("SNR (dB)")
ax.set_ylabel("Packet error rate")
ax.grid(True, which="both", alpha=0.22)
ax.legend(frameon=False)
fig.tight_layout()
save_inline(fig, "fig07_bmocz_vs_sqm")

print("Source:", fig7_file)
print("Status: EXACT REPRODUCTION.")


## Figure 8 — Observable nuisance rank


In [ ]:
fig8_file = first_existing(
    RESULTS / "figure8_observable_rank" / "mean_singular_spectrum_reproduced.csv",
    REPRO / "Data_Kumar_Original" / "mean_singular_spectrum.csv",
)
f8 = pd.read_csv(fig8_file).sort_values("mode")
display(f8)

fig, ax = plt.subplots(figsize=(5.2, 3.3))
ax.plot(f8["mode"], f8["mean_energy_fraction"], marker="o")
ax.set_yscale("log")
ax.set_xlabel("Fractional-delay nuisance mode")
ax.set_ylabel("Mean normalized singular energy")
ax.grid(True, which="both", alpha=0.22)
fig.tight_layout()
save_inline(fig, "fig08_observable_nuisance_rank")

print("Source:", fig8_file)
print("Status: EXACT REPRODUCTION.")


## Figure 9 — PER versus retained nuisance rank


In [ ]:
fig9_file = first_existing(
    ROOT / "Data" / "rank_vs_snr.csv",
    REPRO / "Data_Kumar_Original" / "rank_vs_snr.csv",
)
f9 = pd.read_csv(fig9_file)
display(f9)

fig, ax = plt.subplots(figsize=(5.4, 3.45))
for snr, g in f9.groupby("snr_db", sort=True):
    g = g.sort_values("rank")
    ax.errorbar(
        g["rank"],
        g["PER_mean"],
        yerr=g["PER_95CI_halfwidth"],
        marker="o",
        capsize=3,
        label=f"{snr:g} dB",
    )
ax.set_yscale("log")
ax.set_xlabel("Retained nuisance modes")
ax.set_ylabel("PER")
ax.grid(True, which="both", alpha=0.22)
ax.legend(frameon=False)
fig.tight_layout()
save_inline(fig, "fig09_rank_vs_snr")

print("Source:", fig9_file)
print("Status: RELEASED-DATA RECONSTRUCTION; independent manuscript-equation rerun is a V3 target.")


## Figure 10 — I/Q condition-number sensitivity


In [ ]:
fig10_file = first_existing(
    RESULTS / "figure10_iq_condition" / "operating_iq_sweep_reproduced.csv",
    REPRO / "Data_Kumar_Original" / "operating_iq_sweep_covaware.csv",
)
raw10 = pd.read_csv(fig10_file)

# The two receiver labels are numerically identical in the released final screen.
# Plot covariance-aware metric when present.
if "receiver" in raw10.columns and (raw10["receiver"] == "covariance-aware metric").any():
    raw10 = raw10[raw10["receiver"] == "covariance-aware metric"].copy()

summary10 = (
    raw10.groupby("condition_number")["PER"]
    .agg(["mean", "std", "count"])
    .reset_index()
)
summary10["ci"] = 4.302652729911275 * summary10["std"] / np.sqrt(summary10["count"])
display(summary10)

fig, ax = plt.subplots(figsize=(5.2, 3.3))
ax.errorbar(
    summary10["condition_number"],
    summary10["mean"],
    yerr=summary10["ci"],
    marker="o",
    capsize=3,
)
ax.set_xlabel("Static I/Q condition number")
ax.set_ylabel("PER at 10 dB")
ax.grid(True, alpha=0.25)
fig.tight_layout()
save_inline(fig, "fig10_iq_condition_number")

print("Source:", fig10_file)
print("Status: EXACT REPRODUCTION.")


## Figure 11 — CFO-range sensitivity


In [ ]:
fig11_file = first_existing(
    ROOT / "Data" / "operating_cfo_sweep.csv",
    REPRO / "Data_Kumar_Original" / "operating_cfo_sweep.csv",
)
f11 = pd.read_csv(fig11_file)
display(f11)

name_map = {
    "Incidence-DMM": "SQM",
    "CP-OFDM practical [12,8]": "Practical CP-OFDM",
    "CP-OFDM + oracle CFO": "CP-OFDM + oracle CFO",
}

fig, ax = plt.subplots(figsize=(5.4, 3.35))
for method, g in f11.groupby("method", sort=False):
    g = g.sort_values("x")
    ax.errorbar(
        g["x"],
        g["PER_mean"],
        yerr=g["PER_95CI_halfwidth"],
        marker="o",
        capsize=3,
        label=name_map.get(method, method),
    )
ax.set_yscale("log")
ax.set_xlabel("CFO range (± degrees/sample)")
ax.set_ylabel("PER at 10 dB")
ax.grid(True, which="both", alpha=0.22)
ax.legend(frameon=False, loc="lower right")
fig.tight_layout()
save_inline(fig, "fig11_cfo_range_sensitivity")

print("Source:", fig11_file)
print("Status: RELEASED-DATA RECONSTRUCTION; independent manuscript-equation rerun is a V3 target.")


## Figure 12 — Coarse-to-fine decoder complexity


In [ ]:
fig12_file = first_existing(
    REPRO / "Data_Kumar_Original" / "fractional_decoder_complexity.csv",
    ROOT / "Data" / "fractional_decoder_complexity.csv",
)
f12 = pd.read_csv(fig12_file)
display(f12)

fig, ax = plt.subplots(figsize=(5.4, 3.25))
for _, row in f12.iterrows():
    ax.errorbar(
        row["relative_score_count"],
        row["PER_mean"],
        yerr=row["PER_95CI_halfwidth"],
        marker="o",
        capsize=3,
        linestyle="none",
        label=str(row["decoder"]),
    )
ax.set_xlabel("Relative candidate-CFO score count")
ax.set_ylabel("PER at 10 dB")
ax.grid(True, alpha=0.25)
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
save_inline(fig, "fig12_coarse_to_fine_complexity")

print("Source:", fig12_file)
print("Status: RELEASED-DATA RECONSTRUCTION; independent manuscript-equation rerun is a V3 target.")


## Figure 13 — Representative adaptive-rank trace


In [ ]:
fig13_file = first_existing(
    ROOT / "Data" / "online_adaptive_packet_results.csv",
    REPRO / "Data_Kumar_Original" / "online_adaptive_packet_results.csv",
)
p13 = pd.read_csv(fig13_file)

print("Columns:", list(p13.columns))

# Canonical trace only when that column exists.
if "sequence" in p13.columns:
    p13 = p13[p13["sequence"].eq("canonical")].copy()

# Pick one representative seed.
seed_col = "seed"
seed_value = sorted(p13[seed_col].dropna().unique())[0]
p13 = p13[p13[seed_col].eq(seed_value)].copy()

x_candidates = ["packet", "packet_index", "t", "index"]
xcol = next((c for c in x_candidates if c in p13.columns), None)
if xcol is None:
    p13 = p13.reset_index().rename(columns={"index": "packet_index"})
    xcol = "packet_index"

method_col = "method"
rank_col = "selected_rank"

preferred = [
    "S0_fixed_r4",
    "S2_snr_lookup",
    "S4_window_gap_residual",
    "S5_ewma_gap_residual",
    "S8_regime_oracle_rank",
]
methods = [m for m in preferred if m in set(p13[method_col])]
if not methods:
    methods = list(p13[method_col].drop_duplicates())[:5]

fig, ax = plt.subplots(figsize=(7.4, 3.2))
for method in methods:
    g = p13[p13[method_col].eq(method)].sort_values(xcol)
    ax.plot(g[xcol], g[rank_col], marker=".", markevery=max(1, len(g)//30), label=method)

if "segment" in p13.columns:
    one = p13[p13[method_col].eq(methods[0])].sort_values(xcol)
    seg = one["segment"].to_numpy()
    xx = one[xcol].to_numpy()
    changes = np.flatnonzero(seg[1:] != seg[:-1]) + 1
    for k in changes:
        ax.axvline(xx[k], linestyle="--", linewidth=0.9, alpha=0.6)

ax.set_xlabel("Packet index")
ax.set_ylabel("Selected rank")
ax.set_yticks([3, 4, 5, 6])
ax.grid(True, alpha=0.20)
ax.legend(frameon=False, fontsize=8, ncol=2)
fig.tight_layout()
save_inline(fig, "fig13_adaptive_rank_trace")

print("Source:", fig13_file)
print("Representative seed:", seed_value)
print("Status: EXACT RELEASED ADAPTIVE OUTPUTS; corrected S6 is reported separately in V2.")


## Figure 14 — Post-transition PER with corrected statistical unit


In [ ]:
# Recompute the corrected Figure 14 directly from the released seed-transition data:
# average the three transitions within each seed, then use the 3 seeds as independent replicates.

fig14_file = first_existing(
    ROOT / "Data" / "online_transient_seed_metrics.csv",
    REPRO / "Data_Kumar_Original" / "online_transient_seed_metrics.csv",
)
tr = pd.read_csv(fig14_file)

if "sequence" in tr.columns:
    tr = tr[tr["sequence"].eq("canonical")].copy()

seed_avg = (
    tr.groupby(["seed", "method", "window"], as_index=False)["PER"]
    .mean()
)

s14 = (
    seed_avg.groupby(["method", "window"])["PER"]
    .agg(["mean", "std", "count"])
    .reset_index()
)
s14["ci"] = 4.302652729911275 * s14["std"] / np.sqrt(s14["count"])
display(s14)

# Preserve natural manuscript window order when possible.
window_order = ["First 25", "First 50", "First 100", "100+"]
present = list(s14["window"].drop_duplicates())
order = [w for w in window_order if w in present] + [w for w in present if w not in window_order]
xmap = {w: i for i, w in enumerate(order)}

fig, ax = plt.subplots(figsize=(6.2, 3.35))
for method, g in s14.groupby("method", sort=False):
    g = g.assign(_x=g["window"].map(xmap)).sort_values("_x")
    ax.errorbar(
        g["_x"],
        g["mean"],
        yerr=g["ci"],
        marker="o",
        capsize=3,
        label=method,
    )
ax.set_xticks(range(len(order)))
ax.set_xticklabels(order)
ax.set_xlabel("Packets after each regime transition")
ax.set_ylabel("PER")
ax.grid(True, alpha=0.22)
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
save_inline(fig, "fig14_post_transition_per_corrected")

print("Source:", fig14_file)
print("Status: CORRECTED ANALYSIS STATISTICS — seed-level aggregation + t(df=2).")


## Figure 15 — Window/EWMA sensitivity


In [ ]:
fig15_v2 = V2_DIR / "online_controller_sensitivity_v2.csv"
fig15_release = first_existing(
    ROOT / "Data" / "online_controller_sensitivity.csv",
    REPRO / "Data_Kumar_Original" / "online_controller_sensitivity.csv",
)

if fig15_v2.exists():
    f15 = pd.read_csv(fig15_v2)
    source15 = fig15_v2
    ci_candidates = ["PER_ci95_t_df2", "PER_ci95"]
    status15 = "CORRECTED ANALYSIS t(df=2) INTERVALS"
else:
    f15 = pd.read_csv(fig15_release)
    source15 = fig15_release
    ci_candidates = ["PER_ci95", "PER_95CI_halfwidth"]
    status15 = "RELEASED SENSITIVITY (V2 corrected file not yet generated)"

ci_col = next(c for c in ci_candidates if c in f15.columns)
display(f15)

fig, axes = plt.subplots(1, 2, figsize=(8.0, 3.25), sharey=True)

w = f15[f15["policy"].eq("S4_window")].sort_values("W")
axes[0].errorbar(w["W"], w["PER_mean"], yerr=w[ci_col], marker="o", capsize=3)
axes[0].set_xlabel("Window length W")
axes[0].set_ylabel("PER")
axes[0].grid(True, alpha=0.22)

e = f15[f15["policy"].eq("S5_EWMA")].sort_values("alpha")
axes[1].errorbar(e["alpha"], e["PER_mean"], yerr=e[ci_col], marker="o", capsize=3)
axes[1].set_xlabel("EWMA factor α")
axes[1].set_xticks(e["alpha"].to_numpy())
axes[1].grid(True, alpha=0.22)

fig.tight_layout()
save_inline(fig, "fig15_controller_sensitivity")

print("Source:", source15)
print("Status:", status15)


## Key manuscript tables — inline audit view


In [ ]:
print("TABLE III — Main TDL-A")
display(
    f2[["snr_db", "method", "PER_mean", "PER_95CI_halfwidth"]]
    .sort_values(["snr_db", "method"])
)

print("\nTABLE IV — Coded OFDM controls")
display(f3)

print("\nTABLE V — BMOCZ")
display(f7)

table_vii_v2 = V2_DIR / "table_vii_summary_v2.csv"
if table_vii_v2.exists():
    print("\nTABLE VII — Corrected V2 adaptive summary")
    display(pd.read_csv(table_vii_v2))
else:
    print("\nTABLE VII — corrected V2 summary has not been generated yet.")
    print("Run Part II (V2.1–V2.7) before this section to populate it.")

print("\nTABLE IX — Published selective-margin decomposition")
display(f1)


## Inline-figure completion ledger

After a complete **Run All**, Part III generates Figs. 1–15 inline and saves matching PNG/PDF files.

Interpretation labels:

- **EXACT REPRODUCTION** — rerun from the released code and matched numerically.
- **RELEASED-DATA RECONSTRUCTION** — the publication CSV exists, but the exact original generating workflow is absent or incompletely documented.
- **CORRECTED ANALYSIS** — the figure/table is intentionally recomputed using the corrected statistical/algorithmic definition.
- **INDEPENDENT VALIDATION** — Part IV re-implements the governing equations/model independently and compares the result with the released artifact.

Part IV independently addresses Fig. 9, Fig. 11, Fig. 12 and Table VIII, and adds adaptation success/censoring analysis.


# Part IV — Mathematical audit and independent equation-based validation

This section is intentionally stricter than a figure reconstruction. It asks two separate questions:

1. **Is the mathematical statement in the manuscript valid as written?**
2. **Can the numerical claim be regenerated from the equations/model rather than merely replotted from a released CSV?**

No manuscript formula is silently repaired. When an equation needs a qualification or correction, the notebook prints that explicitly and uses the corrected form for the independent calculations.

The principal checks are:

- Eq. (15): use `pinv` / compact-SVD projector when the full delayed dictionary is rank deficient.
- Theorem 2 / Eqs. (18)–(20): use the actual explanation rank \(r_{
m full}=\mathrm{rank}(\Phi_c)\le\min(2P,L-1)\); the \(L\ge4P\) statement is only meaningful in the unsaturated full-row-rank regime.
- Post-I/Q noise covariance: distinguish receiver-side noise from noise passed through/equalized by the I/Q transform.
- Grassmann interpretation: qualify the rank-two statement at regular points.
- Appendix inverse expressions: explicitly state the full-row-rank requirement or use a pseudoinverse.
- Adaptive controller reporting: separate success probability, censoring, conditional adaptation time and restricted delay.

The section then independently implements the rank sweep, CFO screen, coarse-to-fine decoder and constructive intersection experiment.


In [ ]:
# independent validation workspace and run controls
from pathlib import Path
import os, math, json, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

IND_DIR = RESULTS / "independent_validation"
IND_FIG_DIR = IND_DIR / "figures"
IND_DIR.mkdir(parents=True, exist_ok=True)
IND_FIG_DIR.mkdir(parents=True, exist_ok=True)

IND_T_CRIT_DF2 = 4.302652729911275
IND_SEEDS = [13, 59, 151]

# Defaults are manuscript-scale or near-manuscript-scale, but can be overridden
# from the shell before launching Jupyter if a faster smoke run is desired.
IND_RANK_BLOCKS_PER_SEED = int(os.getenv("SQM_IND_RANK_BLOCKS", "1200"))
IND_CFO_BLOCKS_PER_SEED = int(os.getenv("SQM_IND_CFO_BLOCKS", "2500"))
IND_CTF_BLOCKS_PER_SEED = int(os.getenv("SQM_IND_CTF_BLOCKS", "900"))


def ind_t_halfwidth(x):
    x = np.asarray(x, dtype=float)
    if len(x) < 2:
        return 0.0
    return float(IND_T_CRIT_DF2 * x.std(ddof=1) / np.sqrt(len(x)))


def ind_savefig(fig, stem):
    png = IND_FIG_DIR / f"{stem}.png"
    pdf = IND_FIG_DIR / f"{stem}.pdf"
    fig.savefig(png, dpi=300, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    print("Saved:", png.relative_to(ROOT))
    print("Saved:", pdf.relative_to(ROOT))
    return png, pdf

print("independent validation output directory:", IND_DIR)
print("Rank sweep blocks/seed:", IND_RANK_BLOCKS_PER_SEED)
print("CFO sweep blocks/seed :", IND_CFO_BLOCKS_PER_SEED)
print("Coarse/fine blocks/seed:", IND_CTF_BLOCKS_PER_SEED)


## IV.1 — Executable mathematical audit of the manuscript equations


In [ ]:
# Equation-level numerical checks.
# These tests do not replace proofs; they catch dimension, rank, covariance,
# and algebra inconsistencies in the actual experimental regime.

from fractional_tdla_core import (
    centering_matrix,
    tdla_operators,
    weighted_nuisance_dictionary,
    exact_fractional_channel,
    random_tdla_coefficients,
)

rng = np.random.default_rng(20261024)
L = 16
H = centering_matrix(L)

# ---------- Eq. (3): centering matrix ----------
err_H_sym = np.linalg.norm(H - H.T)
err_H_idem = np.linalg.norm(H @ H - H)
err_H1 = np.linalg.norm(H @ np.ones(L))

# ---------- Eq. (7) and Theorem 1: exact static affine row-space quotient ----------
def row_projector_fullrow(Z):
    G = Z @ Z.T
    return Z.T @ np.linalg.inv(G) @ Z

Z = rng.normal(size=(2, L))
Z = Z @ H
# Normalize to the paper's centered/equal-row-energy convention.
Gz = Z @ Z.T
w, U = np.linalg.eigh(Gz)
Z = np.sqrt(L / 2.0) * (U @ np.diag(1.0 / np.sqrt(w)) @ U.T) @ Z

A = rng.normal(size=(2, 2))
while abs(np.linalg.det(A)) < 0.2:
    A = rng.normal(size=(2, 2))
d = rng.normal(size=2)
Y_aff = A @ Z + d[:, None]
Qx = row_projector_fullrow(Z @ H)
Qy = row_projector_fullrow(Y_aff @ H)
affine_projector_error = np.max(np.abs(Qx - Qy))

# ---------- Eq. (15): actual 23-path TDL-A full dictionary ----------
X_art = np.load(ROOT / "Artifact" / "fractional_incidence_codebook.npz")["X_opt"]
Xd = X_art[:, 0] + 1j * X_art[:, 1]
delays_tdla, powers_tdla, D_tdla = tdla_operators(L)
Phi = weighted_nuisance_dictionary(Xd[0], 0.0, powers_tdla, D_tdla)
Gram = Phi @ Phi.T
rank_phi = int(np.linalg.matrix_rank(Phi))
rank_gram = int(np.linalg.matrix_rank(Gram))

P_pinv = Phi.T @ np.linalg.pinv(Gram) @ Phi
_, s_phi, Vh_phi = np.linalg.svd(Phi, full_matrices=False)
tol_phi = np.finfo(float).eps * max(Phi.shape) * s_phi[0]
rho = int(np.sum(s_phi > tol_phi))
Vr = Vh_phi[:rho].T
P_svd = Vr @ Vr.T
projector_pinv_svd_error = np.max(np.abs(P_pinv - P_svd))

# ---------- Eq. (17): residual/projection identity ----------
r = 4
Qr = Vh_phi[:r].T
Pr = Qr @ Qr.T
S = rng.normal(size=(2, L)) @ H
rho_fit = np.sum((S @ Pr) ** 2) / np.sum(S ** 2)
residual_ratio = np.linalg.norm(S @ (np.eye(L) - Pr), "fro") / np.linalg.norm(S, "fro")
eq17_error = abs(residual_ratio - np.sqrt(max(0.0, 1.0 - rho_fit)))

# ---------- Theorem 2: rank-saturated lower bound ----------
n = L - 1
r_full = rank_phi
intersection_lower = max(0, 2 * r_full - n)

# ---------- Theorem 3: deterministic triangle-inequality radius sanity ----------
# Construct a true subspace and a separated wrong subspace, then verify that
# random perturbations strictly below Delta/2 do not reverse nearest-subspace order.
Qt = np.linalg.qr(rng.normal(size=(L, 4)))[0]
Qw = np.linalg.qr(rng.normal(size=(L, 4)))[0]
Pt = Qt @ Qt.T
Pw = Qw @ Qw.T
S0 = rng.normal(size=(2, 4)) @ Qt.T

def dist_to_proj(Y, P):
    return float(np.linalg.norm(Y @ (np.eye(L) - P), "fro"))

eps0 = dist_to_proj(S0, Pt)
del0 = dist_to_proj(S0, Pw)
Delta0 = del0 - eps0
radius_check_pass = None
if Delta0 > 1e-10:
    ok = []
    for _ in range(200):
        N = rng.normal(size=S0.shape)
        N /= np.linalg.norm(N, "fro")
        N *= 0.49 * Delta0
        Y = S0 + N
        ok.append(dist_to_proj(Y, Pt) < dist_to_proj(Y, Pw))
    radius_check_pass = bool(np.all(ok))

# ---------- Noise covariance convention after A / after A^{-1} ----------
def normalized_A(kappa=1.5, angle1=0.7, angle2=-0.4):
    c1, s1a = np.cos(angle1), np.sin(angle1)
    c2, s2a = np.cos(angle2), np.sin(angle2)
    R1 = np.array([[c1, -s1a], [s1a, c1]])
    R2 = np.array([[c2, -s2a], [s2a, c2]])
    slo = np.sqrt(2.0 / (1.0 + kappa * kappa))
    shi = kappa * slo
    return R1 @ np.diag([shi, slo]) @ R2

A0 = normalized_A()
sigma2 = 0.05
N = np.sqrt(sigma2) * rng.normal(size=(300000, 2))
cov_received = np.cov(N.T, bias=True)
N_eq = N @ np.linalg.inv(A0).T
cov_equalized = np.cov(N_eq.T, bias=True)
cov_equalized_theory = sigma2 * np.linalg.inv(A0) @ np.linalg.inv(A0).T
noise_received_error = np.max(np.abs(cov_received - sigma2 * np.eye(2)))
noise_equalized_error = np.max(np.abs(cov_equalized - cov_equalized_theory))

# ---------- Eq. (47): widely-linear determinant ----------
alpha = 0.9 + 0.4j
beta = 0.2 - 0.1j
Awl = np.array([
    [np.real(alpha + beta), -np.imag(alpha - beta)],
    [np.imag(alpha + beta),  np.real(alpha - beta)],
])
eq47_error = abs(np.linalg.det(Awl) - (abs(alpha) ** 2 - abs(beta) ** 2))

# ---------- Eqs. (48)-(52): GLRT / projection-energy equivalence ----------
r_glrt = 5
Phi_g = rng.normal(size=(r_glrt, L)) @ H
while np.linalg.matrix_rank(Phi_g) < r_glrt:
    Phi_g = rng.normal(size=(r_glrt, L)) @ H
Yg = rng.normal(size=(2, L)) @ H
Bg = Yg @ Phi_g.T @ np.linalg.inv(Phi_g @ Phi_g.T)
resid_direct = np.linalg.norm(Yg - Bg @ Phi_g, "fro") ** 2
Pg = Phi_g.T @ np.linalg.inv(Phi_g @ Phi_g.T) @ Phi_g
resid_proj = np.linalg.norm(Yg, "fro") ** 2 - np.trace(Yg @ Pg @ Yg.T)
glrt_error = abs(resid_direct - resid_proj)

# ---------- Eq. (36): arithmetic and optional score reuse ----------
M = 256
Omega_full = 11
Omega_coarse = 5
K = 8
scores_eq36 = M * Omega_coarse + K * Omega_full
scores_reuse = M * Omega_coarse + K * (Omega_full - Omega_coarse)

math_audit_ind = pd.DataFrame([
    ["Eq. (3)", "PASS", f"sym={err_H_sym:.2e}, idem={err_H_idem:.2e}, H1={err_H1:.2e}", "No change."],
    ["Eqs. (7)-(10) / Theorem 1", "PASS WITH STATED RANK-2 ASSUMPTION", f"affine projector max error={affine_projector_error:.2e}", "Ordinary inverse is valid for normalized rank-2 codewords; pinv is a safe generic form."],
    ["Eq. (15) full multipath projector", "MANUSCRIPT CORRECTION REQUIRED", f"Phi={Phi.shape}, rank={rank_phi}; Gram rank={rank_gram}/{Gram.shape[0]}; pinv-vs-SVD={projector_pinv_svd_error:.2e}", "Replace inverse by Moore-Penrose pseudoinverse or compact-SVD projector."],
    ["Eqs. (18)-(20) over-invariance barrier", "PASS AFTER RANK-SATURATION QUALIFICATION", f"n={n}, actual r_full={r_full}, lower bound=max(0,2r-n)={intersection_lower}", "State r_full=rank(Phi_c)<=min(2P,L-1). L>=4P is the necessary unsaturated/full-row-rank condition; saturation is even more over-invariant."],
    ["Eq. (17) projection-distance identity", "PASS", f"absolute error={eq17_error:.2e}", "No change."],
    ["Eq. (26) deterministic radius", "PASS", f"random below-radius trials pass={radius_check_pass}", "The theorem is correct for exact true/wrong explanation sets in the stated metric."],
    ["Eq. (27) chi-square tail", "PASS WITH NOISE-CONVENTION QUALIFICATION", "2(L-1) centered real degrees of freedom is consistent with isotropic Gaussian noise on the centered subspace.", "Keep sigma convention explicit (per-real-dimension variance)."],
    ["Eq. (28) diagnostic + following covariance paragraph", "FORMULA DIAGNOSTIC OK; COVARIANCE WORDING REQUIRES CORRECTION", f"post-A isotropic cov error={noise_received_error:.2e}; equalized cov error={noise_equalized_error:.2e}", "With Y=A S0+d1^T+N and N added after A: Cov(N)=sigma^2 I. After A^{-1}: sigma^2 A^{-1}A^{-T}."],
    ["Eqs. (29)-(35)", "PASS", "Projection/incidence and nested-rank decoder identities are dimensionally consistent.", "Rename overloaded Q symbols for readability, not correctness."],
    ["Eq. (36) complexity", "PASS AS IMPLEMENTATION COUNT", f"reported={scores_eq36}; if coarse scores are cached/reused={scores_reuse}", "1368 is correct if all 11 shortlisted scores are recomputed; it is not the minimum possible count."],
    ["Eqs. (37)-(39) adaptive proxies/controller", "PASS AS DEFINITIONS", "gap/residual and hysteresis equations are internally consistent.", "Implementation semantics still require same-packet vs next-packet wording to match code."],
    ["Eq. (41) Grassmann manifold", "QUALIFICATION REQUIRED", "Broad B can be rank-deficient.", "Restrict E_m to regular points with rank(B Phi_m(theta))=2, or describe a stratified union rather than all of E_m subset Gr(2,L-1)."],
    ["Eq. (42) local-overlap heuristic", "PASS AS HEURISTIC", "dim Gr(2,15)=26 and d_i+d_j-26 is the generic transverse-intersection dimension.", "Keep the existing regularity/transversality caveat."],
    ["Eq. (44) constructive B_i/B_j", "PASS WHEN Phi_i,Phi_j HAVE FULL ROW RANK", "P=3 screened construction has rank 6 in the independent independent validation test below.", "State the full-row-rank assumption or use pinv."],
    ["Eqs. (45)-(47) widely-linear real form", "PASS", f"determinant identity error={eq47_error:.2e}", "No change."],
    ["Eqs. (48)-(52) GLRT", "PASS UNDER EXPLICIT FULL-ROW-RANK ASSUMPTION", f"residual/projection identity error={glrt_error:.2e}", "Appendix D already states full row rank; pinv generalization is optional."],
    ["Fig. 5 / Appendix B empirical 'certified radius' wording", "TERMINOLOGY CORRECTION REQUIRED", "Original Appendix B searches only 64 wrong messages.", "Call it a sampled/diagnostic radius; the V2 all-255 discrete-grid screen is stricter but still grid-relative."],
], columns=["item", "status", "numerical_check", "recommended_manuscript_action"])

display(math_audit_ind)
math_audit_ind.to_csv(IND_DIR / "mathematical_audit_ind.csv", index=False)

assert err_H_sym < 1e-12 and err_H_idem < 1e-12 and err_H1 < 1e-12
assert affine_projector_error < 1e-10
assert rank_phi == 15 and rank_gram == 15
assert projector_pinv_svd_error < 1e-5
assert eq17_error < 1e-10
assert radius_check_pass is True
assert noise_received_error < 2e-3
assert noise_equalized_error < 2e-3
assert eq47_error < 1e-12
assert glrt_error < 1e-10

print("\nIND mathematical audit: executable checks completed successfully.")


### Corrected mathematical forms to carry into the manuscript

The notebook uses the following corrected/qualified forms:

\[
P_m^{(P)}(\omega)
=\Phi_{m,c}(\omega)^T
\big(\Phi_{m,c}(\omega)\Phi_{m,c}(\omega)^T\big)^{\dagger}
\Phi_{m,c}(\omega)
=V_\rho V_\rho^T,
\]
where \(\rho=\mathrm{rank}(\Phi_{m,c})\le\min(2P,L-1)\).

For two \(r\)-dimensional explanation subspaces in an \(n\)-dimensional centered space,
\[
\dim(\mathcal S_i\cap\mathcal S_j)\ge \max(0,2r-n).
\]
For the broad full-row-rank unsaturated model \(r=2P\), avoiding a *dimensionally forced* rank-two intersection requires \(L\ge4P\). If \(2P>L-1\), the explanation rank saturates at the centered dimension and the over-invariance problem is stronger, not weaker.

Under the stated model \(Y=A S_0+d1^T+N\) with additive noise injected after \(A\),
\[
\mathrm{Cov}(N)=\sigma^2 I,
\qquad
\mathrm{Cov}(A^{-1}N)=\sigma^2 A^{-1}A^{-T}.
\]

For Eq. (41), interpret \(\mathcal E_m\) on the regular rank-two stratum
\(\{\mathrm{row}(B\Phi_m(\theta)): \mathrm{rank}(B\Phi_m(\theta))=2\}\subset\mathrm{Gr}(2,L-1)\).


## IV.2 — Adaptation success and censoring analysis


In [ ]:
# The old mean delay encoded failures as a full-segment delay, hiding whether
# the controller ever reached the target. independent validation separates event occurrence from time.

released_packet = pd.read_csv(ROOT / "Data" / "online_adaptive_packet_results.csv")
packet_ind = released_packet[released_packet["sequence"].eq("canonical")].copy()

# Prefer the corrected same-packet S6 generated in Part II when available.
s6_v2_path = V2_DIR / "s6_same_packet_packet_results_v2.csv"
if s6_v2_path.exists():
    s6_corr = pd.read_csv(s6_v2_path)
    s6_corr = s6_corr[s6_corr["sequence"].eq("canonical")].copy()
    packet_ind = packet_ind[~packet_ind["method"].eq("S6_confidence_triggered_refinement")]
    # Align the released packet schema; missing controller_switch is safely filled.
    if "controller_switch" not in s6_corr.columns:
        s6_corr["controller_switch"] = s6_corr.get("selected_rank_transition", 0)
    packet_ind = pd.concat([packet_ind, s6_corr[packet_ind.columns]], ignore_index=True)
    s6_source = "corrected V2 same-packet S6"
else:
    s6_source = "released S6 fallback (run Part II first for corrected S6)"

oracle = (
    packet_ind[packet_ind["method"].eq("S8_regime_oracle_rank")]
    .groupby(["seed", "segment"])["selected_rank"]
    .first()
)


def ind_adaptation_events(df, window=10):
    rows = []
    for (seed, method), g in df.groupby(["seed", "method"], sort=False):
        g = g.sort_values("packet")
        segments = sorted(g["segment"].unique())
        for seg in segments[1:]:  # true regime boundaries only
            q = g[g["segment"].eq(seg)].sort_values("packet")
            selected = q["selected_rank"].to_numpy(dtype=int)
            target = int(oracle.loc[(seed, seg)])
            hit = None
            for k in range(0, len(selected) - window + 1):
                if np.all(selected[k:k+window] == target):
                    hit = k
                    break
            success = hit is not None
            seg_len = len(selected)
            rows.append({
                "seed": int(seed),
                "method": method,
                "segment": int(seg),
                "target_rank": target,
                "segment_packets": int(seg_len),
                "success": bool(success),
                "censored": bool(not success),
                "delay_to_run_start": float(hit) if success else np.nan,
                "delay_to_run_confirmation": float(hit + window - 1) if success else np.nan,
                # Restricted time-to-adaptation: failures contribute the observation horizon.
                "restricted_delay": float(hit) if success else float(seg_len),
                "censor_time": float(seg_len),
            })
    return pd.DataFrame(rows)

adapt_events_ind = ind_adaptation_events(packet_ind, window=10)
adapt_events_ind.to_csv(IND_DIR / "adaptation_events_ind.csv", index=False)

# Primary adaptive methods; fixed ranks are retained in the event CSV as negative controls.
adaptive_methods = [
    "S2_SNR_class_lookup",
    "S3_window_confidence",
    "S4_window_confidence_residual",
    "S5_EWMA_confidence_residual",
    "S6_confidence_triggered_refinement",
    "S7_ACK_UCB_optional",
    "S8_regime_oracle_rank",
]

seed_summary_rows = []
for (seed, method), g in adapt_events_ind[adapt_events_ind["method"].isin(adaptive_methods)].groupby(["seed", "method"], sort=False):
    succ = g[g["success"]]
    seed_summary_rows.append({
        "seed": int(seed),
        "method": method,
        "boundaries": len(g),
        "success_rate": float(g["success"].mean()),
        "successful_boundaries": int(g["success"].sum()),
        "censored_boundaries": int(g["censored"].sum()),
        "conditional_delay_successes": float(succ["delay_to_run_start"].mean()) if len(succ) else np.nan,
        "restricted_delay": float(g["restricted_delay"].mean()),
    })
seed_adapt_ind = pd.DataFrame(seed_summary_rows)

summary_rows = []
for method, g in seed_adapt_ind.groupby("method", sort=False):
    raw = adapt_events_ind[adapt_events_ind["method"].eq(method)]
    success_seed = g["success_rate"].to_numpy(float)
    restricted_seed = g["restricted_delay"].to_numpy(float)
    cond_seed = g["conditional_delay_successes"].dropna().to_numpy(float)
    summary_rows.append({
        "method": method,
        "boundary_events": int(len(raw)),
        "successes": int(raw["success"].sum()),
        "censored": int(raw["censored"].sum()),
        "pooled_success_rate": float(raw["success"].mean()),
        "seed_mean_success_rate": float(success_seed.mean()),
        "success_rate_95CI_halfwidth_t_df2": ind_t_halfwidth(success_seed),
        "mean_delay_given_success": float(raw.loc[raw["success"], "delay_to_run_start"].mean()) if raw["success"].any() else np.nan,
        "seed_mean_conditional_delay": float(cond_seed.mean()) if len(cond_seed) else np.nan,
        "conditional_delay_95CI_halfwidth": ind_t_halfwidth(cond_seed) if len(cond_seed) > 1 else np.nan,
        "seed_mean_restricted_delay": float(restricted_seed.mean()),
        "restricted_delay_95CI_halfwidth_t_df2": ind_t_halfwidth(restricted_seed),
    })

adapt_summary_ind = pd.DataFrame(summary_rows)
adapt_summary_ind.to_csv(IND_DIR / "adaptation_success_censoring_summary_ind.csv", index=False)
seed_adapt_ind.to_csv(IND_DIR / "adaptation_success_censoring_per_seed_ind.csv", index=False)

print("S6 source used:", s6_source)
print("\nPer-boundary event table:")
display(adapt_events_ind[adapt_events_ind["method"].isin(adaptive_methods)])
print("\nIND adaptation success/censoring summary:")
display(adapt_summary_ind)

# Plot success probability and restricted delay. Restricted delay is deliberately
# shown alongside success so a censored failure is never mistaken for a measured delay.
plot_methods = [m for m in adaptive_methods if m in set(adapt_summary_ind["method"])]
labels = {
    "S2_SNR_class_lookup": "S2 lookup",
    "S3_window_confidence": "S3 window",
    "S4_window_confidence_residual": "S4 window+residual",
    "S5_EWMA_confidence_residual": "S5 EWMA+residual",
    "S6_confidence_triggered_refinement": "S6 refinement",
    "S7_ACK_UCB_optional": "S7 ACK-UCB",
    "S8_regime_oracle_rank": "S8 oracle",
}
q = adapt_summary_ind.set_index("method").loc[plot_methods]
x = np.arange(len(plot_methods))
fig, axes = plt.subplots(2, 1, figsize=(8.2, 6.0), sharex=True)
axes[0].errorbar(x, q["seed_mean_success_rate"], yerr=q["success_rate_95CI_halfwidth_t_df2"], fmt="o", capsize=3)
axes[0].set_ylabel("Adaptation success rate")
axes[0].set_ylim(-0.05, 1.05)
axes[0].grid(True, alpha=0.22)
axes[1].errorbar(x, q["seed_mean_restricted_delay"], yerr=q["restricted_delay_95CI_halfwidth_t_df2"], fmt="o", capsize=3)
axes[1].set_ylabel("Restricted delay (packets)")
axes[1].set_xticks(x)
axes[1].set_xticklabels([labels[m] for m in plot_methods], rotation=25, ha="right")
axes[1].grid(True, alpha=0.22)
fig.tight_layout()
ind_savefig(fig, "fig_ind_adaptation_success_censoring")
plt.show()

print("\nInterpretation: report success/censoring first; quote a delay only with its success definition.")


## IV.3 — Shared independent TDL-A simulation helpers for Figs. 9 and 12


In [ ]:
# Self-contained implementation of Eqs. (21), (22), (34), and (35) on the
# released frozen codebook. The core fractional-delay operator is imported,
# but rank selection, basis bank construction, channel draws, and decoding are
# explicitly implemented here rather than delegated to a publication CSV.

M_IND = 256
L_IND = 16
IND_GRID_DEG = np.arange(-5.0, 6.0, 1.0)
IND_GRID_RAD = np.deg2rad(IND_GRID_DEG)
IND_RANKS = [2, 3, 4, 5, 6, 8]

X_ind = np.load(ROOT / "Artifact" / "fractional_incidence_codebook.npz")["X_opt"]
Xd_ind = X_ind[:, 0] + 1j * X_ind[:, 1]
delays_ind, powers_ind, D_ind = tdla_operators(L_IND)


def ind_rot(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])


def ind_random_A(rng, N, kappa=1.5):
    slo = np.sqrt(2.0 / (1.0 + kappa * kappa))
    shi = kappa * slo
    out = np.empty((N, 2, 2), dtype=float)
    a = rng.uniform(0.0, 2 * np.pi, N)
    b = rng.uniform(0.0, 2 * np.pi, N)
    for i in range(N):
        out[i] = ind_rot(a[i]) @ np.diag([shi, slo]) @ ind_rot(b[i])
    return out


def ind_build_tdla_basis_bank(rmax=8):
    bank = np.empty((M_IND, len(IND_GRID_RAD), L_IND, rmax), dtype=np.float32)
    for m in range(M_IND):
        for j, omega in enumerate(IND_GRID_RAD):
            F = weighted_nuisance_dictionary(Xd_ind[m], omega, powers_ind, D_ind)
            _, _, Vh = np.linalg.svd(F, full_matrices=False)
            bank[m, j] = Vh[:rmax].T.astype(np.float32)
    return bank


def ind_generate_tdla_trace(snr_db, blocks, seed, kappa=1.5):
    rng = np.random.default_rng(seed)
    idx = rng.integers(M_IND, size=blocks)
    h = random_tdla_coefficients(rng, blocks, powers_ind)
    omega = rng.uniform(-np.deg2rad(4.0), np.deg2rad(4.0), blocks)
    A = ind_random_A(rng, blocks, kappa)
    dc_angle = rng.uniform(0.0, 2 * np.pi, blocks)
    d = np.stack([np.cos(dc_angle), np.sin(dc_angle)], axis=1)
    N0 = 10.0 ** (-snr_db / 10.0)
    noise = np.sqrt(N0 / 2.0) * rng.normal(size=(blocks, 2, L_IND))
    s = exact_fractional_channel(Xd_ind[idx], h, D_ind)
    s *= np.exp(1j * omega[:, None] * np.arange(L_IND)[None, :])
    sr = np.stack([s.real, s.imag], axis=1)
    Y = np.einsum("nij,njk->nik", A, sr, optimize=True) + d[:, :, None] + noise
    return Y, idx


def ind_score_cube(Y, bank, rank=4, batch=32):
    Yc = Y - Y.mean(axis=2, keepdims=True)
    out = np.empty((len(Y), M_IND, len(IND_GRID_RAD)), dtype=np.float32)
    B = bank[..., :rank]
    for s in range(0, len(Y), batch):
        e = min(len(Y), s + batch)
        Z = np.einsum("bal,mwlr->bmwar", Yc[s:e], B, optimize=True)
        out[s:e] = np.sum(Z * Z, axis=(3, 4), dtype=np.float64).astype(np.float32)
    return out


def ind_decode_from_scores(scores):
    return np.argmax(scores.max(axis=2), axis=1)

print("Building shared independent validation rank-8 TDL-A basis bank ...")
t0 = time.time()
ind_tdla_bank = ind_build_tdla_basis_bank(rmax=max(IND_RANKS))
print(f"Bank shape: {ind_tdla_bank.shape}; built in {time.time()-t0:.1f} s")


## IV.4 — Figure 9: independent PER-versus-rank simulation


In [ ]:
# Independent paired rank sweep: every rank sees the same packet/channel/noise
# realization within a seed, making the rank comparison cleaner than separate draws.

fig9_rows = []
fig9_seed_rows = []
for snr_db in [5.0, 10.0, 15.0]:
    per_rank_seed = {r: [] for r in IND_RANKS}
    for seed in IND_SEEDS:
        sim_seed = 2026102400 + int(100 * snr_db) + seed
        Y, idx = ind_generate_tdla_trace(snr_db, IND_RANK_BLOCKS_PER_SEED, sim_seed)
        for r in IND_RANKS:
            scores = ind_score_cube(Y, ind_tdla_bank, rank=r)
            pred = ind_decode_from_scores(scores)
            per = float(np.mean(pred != idx))
            per_rank_seed[r].append(per)
            fig9_seed_rows.append({"snr_db": snr_db, "rank": r, "seed": seed, "PER": per, "blocks": len(idx)})
    for r in IND_RANKS:
        vals = np.asarray(per_rank_seed[r], float)
        fig9_rows.append({
            "snr_db": snr_db,
            "rank": r,
            "PER_mean": float(vals.mean()),
            "PER_95CI_halfwidth": ind_t_halfwidth(vals),
            "blocks": int(IND_RANK_BLOCKS_PER_SEED * len(vals)),
        })

fig9_ind = pd.DataFrame(fig9_rows)
fig9_seed_ind = pd.DataFrame(fig9_seed_rows)
fig9_ind.to_csv(IND_DIR / "fig9_rank_vs_snr_equation_ind.csv", index=False)
fig9_seed_ind.to_csv(IND_DIR / "fig9_rank_vs_snr_equation_per_seed_ind.csv", index=False)

released_fig9 = pd.read_csv(ROOT / "Data" / "rank_vs_snr.csv")
print("independent validation independent Fig. 9:")
display(fig9_ind)
print("Released Fig. 9 data:")
display(released_fig9)

fig, ax = plt.subplots(figsize=(6.6, 4.2))
for snr_db, g in fig9_ind.groupby("snr_db"):
    g = g.sort_values("rank")
    ax.errorbar(g["rank"], g["PER_mean"], yerr=g["PER_95CI_halfwidth"], marker="o", capsize=3, label=f"independent validation {snr_db:g} dB")
for snr_db, g in released_fig9.groupby("snr_db"):
    g = g.sort_values("rank")
    ax.plot(g["rank"], g["PER_mean"], linestyle="--", marker=".", alpha=0.65, label=f"released {snr_db:g} dB")
ax.set_yscale("log")
ax.set_xlabel("Retained nuisance modes")
ax.set_ylabel("Packet error rate")
ax.grid(True, which="both", alpha=0.22)
ax.legend(frameon=False, ncol=2, fontsize=8)
fig.tight_layout()
ind_savefig(fig, "fig09_rank_vs_snr_independent_ind")
plt.show()

best_ind = fig9_ind.loc[fig9_ind.groupby("snr_db")["PER_mean"].idxmin(), ["snr_db", "rank", "PER_mean"]].sort_values("snr_db")
print("Best screened independent validation rank at each SNR:")
display(best_ind)
print("Status: INDEPENDENT EQUATION-BASED RE-SIMULATION. Exact equality is not required because independent validation uses a new paired draw design.")


## IV.5 — Figure 11: independent equation-based CFO-screen reconstruction

The released repository contains the final Fig. 11 CSV but not the legacy generator that produced it. independent validation therefore performs a **transparent reconstruction attempt from the manuscript model and the closest matching released operating codebook artifact**, `Artifact/stage3_codebooks.npz::X_dmm` (16 complex samples), instead of silently treating the CSV as independently reproduced.

The independent validation reconstruction uses:

- the released 8-bit `X_dmm` operating codebook;
- a physical two-tap PDP \([0.8,0.2]\);
- full rank-four delayed I/Q explanation spaces;
- physical CFO uniform over the requested support;
- a one-degree guarded CFO search;
- static invertible I/Q mixing with condition number 2 and fixed Frobenius energy, as described in the manuscript;
- DC offset and AWGN;
- Eq. (34) projection-energy decoding.

The resulting SQM curve is compared directly with the released `Incidence-DMM` Fig. 11 curve and the released OFDM reference curves. If the independent SQM levels do not match within Monte-Carlo uncertainty, independent validation reports that mismatch as a **remaining provenance/specification gap** rather than tuning hidden assumptions until the curves agree.


In [ ]:
# Closest released operating-screen codebook to the legacy Fig. 11 experiment.
stage3 = np.load(ROOT / "Artifact" / "stage3_codebooks.npz")
X2 = stage3["X_dmm"]
XD2 = X2[:, 0] + 1j * X2[:, 1]
L2 = XD2.shape[1]
PDP2 = np.array([0.8, 0.2], dtype=float)


def ind_shift1(x):
    y = np.zeros_like(x)
    y[..., 1:] = x[..., :-1]
    return y


def ind_two_tap_basis_bank(cfo_range_deg, rank=4):
    if float(cfo_range_deg) == 0.0:
        grid_deg = np.array([0.0])
    else:
        grid_deg = np.linspace(
            -cfo_range_deg - 1.0,
            cfo_range_deg + 1.0,
            int(round(2 * cfo_range_deg + 2)) + 1,
        )
    grid = np.deg2rad(grid_deg)
    bank = np.empty((256, len(grid), L2, rank), np.float32)
    H2 = np.eye(L2) - np.ones((L2, L2)) / L2
    t = np.arange(L2)
    for m in range(256):
        for j, w in enumerate(grid):
            # Physical two-tap feature family. A constant phase factor on a delayed
            # tap is absorbed by the arbitrary real 2x2 coefficient block for that tap.
            u = XD2[m] * np.exp(1j * w * t)
            v = ind_shift1(u)
            F = np.stack([u.real, u.imag, v.real, v.imag]) @ H2
            _, _, Vh = np.linalg.svd(F, full_matrices=False)
            bank[m, j] = Vh[:rank].T.astype(np.float32)
    return grid_deg, bank


def ind_decode_variable_grid(Y, bank, rank=4, batch=48):
    Yc = Y - Y.mean(axis=2, keepdims=True)
    pred = np.empty(len(Y), int)
    B = bank[..., :rank]
    for s in range(0, len(Y), batch):
        e = min(len(Y), s + batch)
        Z = np.einsum("bal,mwlr->bmwar", Yc[s:e], B, optimize=True)
        sc = np.sum(Z * Z, axis=(3, 4))
        pred[s:e] = np.argmax(sc.max(axis=2), axis=1)
    return pred


def ind_two_tap_run(cfo_range_deg, blocks, seed, snr_db=10.0, kappa=2.0):
    rng = np.random.default_rng(seed)
    idx = rng.integers(256, size=blocks)
    h = (
        (rng.normal(size=(blocks, 2)) + 1j * rng.normal(size=(blocks, 2)))
        / np.sqrt(2.0)
    ) * np.sqrt(PDP2)[None, :]

    x = XD2[idx]
    y = np.zeros_like(x, dtype=complex)
    y[:, 0] = h[:, 0] * x[:, 0]
    y[:, 1:] = h[:, 0, None] * x[:, 1:] + h[:, 1, None] * x[:, :-1]

    if cfo_range_deg == 0:
        omega = np.zeros(blocks)
    else:
        omega = rng.uniform(
            -np.deg2rad(cfo_range_deg),
            np.deg2rad(cfo_range_deg),
            blocks,
        )
    y *= np.exp(1j * omega[:, None] * np.arange(L2)[None, :])

    A = ind_random_A(rng, blocks, kappa=kappa)
    ang = rng.uniform(0.0, 2 * np.pi, blocks)
    d = np.stack([np.cos(ang), np.sin(ang)], axis=1)
    N0 = 10.0 ** (-snr_db / 10.0)
    noise = np.sqrt(N0 / 2.0) * rng.normal(size=(blocks, 2, L2))
    yr = np.stack([y.real, y.imag], axis=1)
    Y = np.einsum("nij,njk->nik", A, yr, optimize=True) + d[:, :, None] + noise

    _, bank = ind_two_tap_basis_bank(cfo_range_deg, rank=4)
    pred = ind_decode_variable_grid(Y, bank, rank=4)
    return float(np.mean(pred != idx))


cfo_rows = []
for cfo in [0.0, 2.0, 4.0, 8.0, 12.0]:
    vals = []
    for seed in IND_SEEDS:
        per = ind_two_tap_run(
            cfo,
            IND_CFO_BLOCKS_PER_SEED,
            2026102500 + int(100 * cfo) + seed,
        )
        vals.append(per)
    vals = np.asarray(vals)
    cfo_rows.append({
        "cfo_range_deg_per_sample": cfo,
        "method": "independent validation equation-based SQM reconstruction",
        "PER_mean": float(vals.mean()),
        "PER_95CI_halfwidth": ind_t_halfwidth(vals),
        "blocks": int(IND_CFO_BLOCKS_PER_SEED * len(vals)),
    })

fig11_ind = pd.DataFrame(cfo_rows)
fig11_ind.to_csv(IND_DIR / "fig11_cfo_sqm_equation_ind.csv", index=False)
released_cfo = pd.read_csv(ROOT / "Data" / "operating_cfo_sweep.csv")
released_sqm = (
    released_cfo[released_cfo["method"].eq("Incidence-DMM")]
    [["x", "PER_mean", "PER_95CI_halfwidth"]]
    .rename(columns={
        "x": "cfo_range_deg_per_sample",
        "PER_mean": "released_PER_mean",
        "PER_95CI_halfwidth": "released_95CI_halfwidth",
    })
)
fig11_compare_ind = fig11_ind.merge(released_sqm, on="cfo_range_deg_per_sample", how="left")
fig11_compare_ind["ind_minus_released"] = fig11_compare_ind["PER_mean"] - fig11_compare_ind["released_PER_mean"]
fig11_compare_ind["abs_difference"] = fig11_compare_ind["ind_minus_released"].abs()
fig11_compare_ind.to_csv(IND_DIR / "fig11_cfo_sqm_released_comparison_ind.csv", index=False)

print("Independent independent validation SQM Fig. 11 reconstruction:")
display(fig11_ind)
print("Direct comparison with the released SQM curve:")
display(fig11_compare_ind)
print("Released full Fig. 11 data:")
display(released_cfo)

fig, ax = plt.subplots(figsize=(6.7, 4.1))
ax.errorbar(
    fig11_ind["cfo_range_deg_per_sample"],
    fig11_ind["PER_mean"],
    yerr=fig11_ind["PER_95CI_halfwidth"],
    marker="o", capsize=3,
    label="independent validation equation-based SQM reconstruction",
)
for method in ["Incidence-DMM", "CP-OFDM practical [12,8]", "CP-OFDM + oracle CFO"]:
    g = released_cfo[released_cfo["method"].eq(method)].sort_values("x")
    if len(g):
        ax.plot(
            g["x"], g["PER_mean"],
            linestyle="--", marker=".", alpha=0.75,
            label=f"released: {method}",
        )
ax.set_yscale("log")
ax.set_xlabel("CFO range (± degrees/sample)")
ax.set_ylabel("PER at 10 dB")
ax.grid(True, which="both", alpha=0.22)
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
ind_savefig(fig, "fig11_cfo_hybrid_independent_ind")
plt.show()

mean_abs_gap = float(fig11_compare_ind["abs_difference"].mean())
print(f"Mean absolute independent validation-vs-released SQM PER gap: {mean_abs_gap:.6f}")
if mean_abs_gap > 0.01:
    print("PROVENANCE WARNING: the independent reconstruction does not reproduce the released Fig. 11 SQM levels closely.")
    print("This indicates that at least one legacy codebook/model/noise/RNG convention used for operating_cfo_sweep.csv is not fully specified by the current manuscript/repository.")
else:
    print("The independent reconstruction is numerically close to the released SQM curve.")
print("Status: equation-based reconstruction attempted and quantified; released OFDM curves remain reference data, not independently regenerated.")


## IV.6 — Figure 12: independent coarse-to-fine decoder

The manuscript specifies five coarse CFO points but does not enumerate them. independent validation makes the missing convention explicit and uses the symmetric interior subset
\([-4,-2,0,2,4]^\circ/\mathrm{sample}\) of the guarded \([-5,5]\) one-degree grid.

For each packet:

1. evaluate every message on the five coarse points;
2. shortlist the top \(K\) messages by maximum coarse score;
3. evaluate those messages on all eleven guarded-grid points;
4. choose the largest refined score.

The reported score count follows Eq. (36), i.e. it **recomputes** all eleven shortlisted scores. The notebook also prints the lower count available if the five already-computed coarse scores are cached and reused.


In [ ]:
coarse_deg = np.array([-4.0, -2.0, 0.0, 2.0, 4.0])
coarse_idx = np.array([int(np.where(IND_GRID_DEG == d)[0][0]) for d in coarse_deg])
K_LIST = [4, 8, 16]


def ind_ctf_predictions(scores, K):
    coarse_msg = scores[:, :, coarse_idx].max(axis=2)
    shortlist = np.argpartition(coarse_msg, -K, axis=1)[:, -K:]
    pred = np.empty(len(scores), int)
    for b in range(len(scores)):
        mids = shortlist[b]
        refined = scores[b, mids, :]
        flat = int(np.argmax(refined))
        pred[b] = int(mids[flat // refined.shape[1]])
    return pred

ctf_seed_rows = []
for seed in IND_SEEDS:
    Y, idx = ind_generate_tdla_trace(10.0, IND_CTF_BLOCKS_PER_SEED, 2026102600 + seed)
    scores = ind_score_cube(Y, ind_tdla_bank, rank=4, batch=24)
    pred_full = ind_decode_from_scores(scores)
    per_full = float(np.mean(pred_full != idx))
    ctf_seed_rows.append({
        "decoder": "full search", "seed": seed, "PER": per_full,
        "disagreement_with_full": 0.0, "candidate_CFO_scores": 256 * 11,
    })
    for K in K_LIST:
        pred = ind_ctf_predictions(scores, K)
        ctf_seed_rows.append({
            "decoder": f"coarse-to-fine top-{K}",
            "seed": seed,
            "PER": float(np.mean(pred != idx)),
            "disagreement_with_full": float(np.mean(pred != pred_full)),
            "candidate_CFO_scores": int(256 * 5 + K * 11),
        })

ctf_seed_ind = pd.DataFrame(ctf_seed_rows)
ctf_rows = []
for decoder, g in ctf_seed_ind.groupby("decoder", sort=False):
    p = g["PER"].to_numpy(float)
    d = g["disagreement_with_full"].to_numpy(float)
    count = int(g["candidate_CFO_scores"].iloc[0])
    K = None if decoder == "full search" else int(decoder.rsplit("-", 1)[1])
    reuse_count = count if K is None else int(256 * 5 + K * (11 - 5))
    ctf_rows.append({
        "decoder": decoder,
        "PER_mean": float(p.mean()),
        "PER_95CI_halfwidth": ind_t_halfwidth(p),
        "disagreement_with_full": float(d.mean()),
        "candidate_CFO_scores": count,
        "relative_score_count": count / (256 * 11),
        "scores_if_coarse_reused": reuse_count,
        "relative_if_coarse_reused": reuse_count / (256 * 11),
        "blocks": int(IND_CTF_BLOCKS_PER_SEED * len(g)),
    })

fig12_ind = pd.DataFrame(ctf_rows)
fig12_ind.to_csv(IND_DIR / "fig12_coarse_to_fine_equation_ind.csv", index=False)
ctf_seed_ind.to_csv(IND_DIR / "fig12_coarse_to_fine_equation_per_seed_ind.csv", index=False)
released_ctf = pd.read_csv(ROOT / "Data" / "fractional_decoder_complexity.csv")

print("independent validation independent coarse-to-fine results:")
display(fig12_ind)
print("Released Fig. 12 data:")
display(released_ctf)

fig, ax = plt.subplots(figsize=(6.5, 4.0))
ax.errorbar(fig12_ind["relative_score_count"], fig12_ind["PER_mean"], yerr=fig12_ind["PER_95CI_halfwidth"], fmt="o", capsize=3, label="independent validation independent")
for _, row in fig12_ind.iterrows():
    ax.annotate(row["decoder"].replace("coarse-to-fine ", ""), (row["relative_score_count"], row["PER_mean"]), xytext=(5, 5), textcoords="offset points", fontsize=8)
ax.scatter(released_ctf["relative_score_count"], released_ctf["PER_mean"], marker="x", label="released")
ax.set_xlabel("Relative candidate–CFO score count")
ax.set_ylabel("PER at 10 dB")
ax.grid(True, alpha=0.22)
ax.legend(frameon=False)
fig.tight_layout()
ind_savefig(fig, "fig12_coarse_to_fine_independent_ind")
plt.show()

print("Top-8 Eq. (36) count:", 256*5 + 8*11, "=", (256*5+8*11)/(256*11)*100, "% of full")
print("Top-8 with cached coarse scores:", 256*5 + 8*(11-5), "=", (256*5+8*(11-5))/(256*11)*100, "% of full")


## IV.7 — Table VIII: independent structured three-tap constructive intersection check


In [ ]:
# Independent implementation of the geometry behind Appendix A / Table VIII.
# We use a structured P=3 delayed-feature family (zero-padded delays 0,1,2)
# built from random complex codewords, center every feature row, and verify the
# generic intersection dimension and constructive rank-two collision.


def ind_random_centered_complex_codeword(L, rng):
    x = rng.normal(size=L) + 1j * rng.normal(size=L)
    x = x - x.mean()
    x = x * np.sqrt(L / np.sum(np.abs(x) ** 2))
    return x


def ind_zero_shift(x, d):
    y = np.zeros_like(x)
    if d == 0:
        y[:] = x
    else:
        y[d:] = x[:-d]
    return y


def ind_phi_p3(x):
    L = len(x)
    H = np.eye(L) - np.ones((L, L)) / L
    rows = []
    for d in [0, 1, 2]:
        v = ind_zero_shift(x, d)
        rows.extend([v.real, v.imag])
    return np.stack(rows) @ H


def ind_row_basis(Phi, tol=1e-10):
    _, s, Vh = np.linalg.svd(Phi, full_matrices=False)
    r = int(np.sum(s > tol * s[0]))
    return Vh[:r], r


def ind_intersection_basis(Qi, Qj, tol=1e-8):
    U, s, Vh = np.linalg.svd(Qi @ Qj.T, full_matrices=False)
    hit = np.where(s > 1.0 - tol)[0]
    return U[:, hit].T @ Qi, s

rng8 = np.random.default_rng(20261027)
rows8 = []
TRIALS8 = 100
for L8 in range(9, 15):
    dims = []
    collision_residuals = []
    for trial in range(TRIALS8):
        Phi_i = ind_phi_p3(ind_random_centered_complex_codeword(L8, rng8))
        Phi_j = ind_phi_p3(ind_random_centered_complex_codeword(L8, rng8))
        ri = int(np.linalg.matrix_rank(Phi_i))
        rj = int(np.linalg.matrix_rank(Phi_j))
        stacked_rank = int(np.linalg.matrix_rank(np.vstack([Phi_i, Phi_j])))
        dint = ri + rj - stacked_rank
        dims.append(dint)

        if dint >= 2:
            Qi, _ = ind_row_basis(Phi_i)
            Qj, _ = ind_row_basis(Phi_j)
            Ibas, sing = ind_intersection_basis(Qi, Qj)
            if Ibas.shape[0] >= 2:
                Y = Ibas[:2]
                # Eq. (44) is valid here because rank(Phi_i)=rank(Phi_j)=6.
                Bi = Y @ Phi_i.T @ np.linalg.inv(Phi_i @ Phi_i.T)
                Bj = Y @ Phi_j.T @ np.linalg.inv(Phi_j @ Phi_j.T)
                rel_i = np.linalg.norm(Y - Bi @ Phi_i, "fro") / np.linalg.norm(Y, "fro")
                rel_j = np.linalg.norm(Y - Bj @ Phi_j, "fro") / np.linalg.norm(Y, "fro")
                collision_residuals.append(max(rel_i, rel_j))

    n8 = L8 - 1
    r8 = 6
    lower = max(0, 2 * r8 - n8)
    rows8.append({
        "P": 3,
        "L": L8,
        "centered_dimension_n": n8,
        "feature_rank_r": r8,
        "dimension_lower_bound_max0_2r_minus_n": lower,
        "median_observed_intersection_dim": float(np.median(dims)),
        "min_observed_intersection_dim": int(np.min(dims)),
        "max_observed_intersection_dim": int(np.max(dims)),
        "rank_two_collision_constructed": bool(len(collision_residuals) > 0),
        "max_relative_collision_residual": float(max(collision_residuals)) if collision_residuals else np.nan,
    })

table8_ind = pd.DataFrame(rows8)
table8_ind.to_csv(IND_DIR / "table_viii_intersection_equation_ind.csv", index=False)
released_table8 = pd.read_csv(ROOT / "Data" / "P3_intersection_barrier.csv")

print("independent validation independent Table VIII:")
display(table8_ind)
print("Released Table VIII artifact:")
display(released_table8)

assert np.all(table8_ind["feature_rank_r"].eq(6))
assert np.all(table8_ind["median_observed_intersection_dim"].to_numpy() == np.array([4,3,2,1,0,0], float))
assert table8_ind.loc[table8_ind["L"] <= 11, "rank_two_collision_constructed"].all()
assert (~table8_ind.loc[table8_ind["L"] >= 12, "rank_two_collision_constructed"]).all()

print("Table VIII theorem check: exact generic intersection dimensions reproduced; rank-two collisions constructed exactly where the lower bound is at least two.")


## IV.8 — Final independent validation validation ledger


In [ ]:
ind_ledger = pd.DataFrame([
    ["Manuscript equations", "MATH-AUDITED", "Most equations pass; Eq. (15), Eq. (28) covariance prose, Eq. (41), and rank assumptions require correction/qualification."],
    ["Adaptation delay", "UPGRADED", "Success/censoring, conditional delay and restricted delay now reported separately; corrected same-packet S6 is used when Part II has run."],
    ["Fig. 9 rank vs SNR", "INDEPENDENT EQUATION-BASED", "Fresh paired TDL-A simulation from Eqs. (21),(22),(34),(35); released CSV retained only for comparison."],
    ["Fig. 11 CFO sensitivity", "INDEPENDENT RECONSTRUCTION + PROVENANCE TEST", "independent validation re-simulates the SQM branch from the closest released operating codebook/model and quantitatively compares it with the released curve. Any residual mismatch is reported as a provenance/specification gap; released OFDM curves remain reference data."],
    ["Fig. 12 coarse-to-fine", "INDEPENDENT EQUATION-BASED", "Explicit five-point coarse grid, top-K shortlist, full guarded-grid refinement; Eq. (36) score counts and score-reuse lower bound both reported."],
    ["Table VIII 4P barrier", "INDEPENDENT EQUATION-BASED", "Structured P=3 delayed-feature experiment reproduces generic intersection dimensions 4,3,2,1,0,0 and constructs rank-two collisions for L<=11."],
    ["Fig. 5 radius terminology", "CORRECTED INTERPRETATION", "Original 64-wrong-message result is diagnostic, not a global certificate; V2 all-255 discrete-grid diagnostic is stricter."],
], columns=["item", "ind_status", "comment"])

display(ind_ledger)
ind_ledger.to_csv(IND_DIR / "master_validation_ledger_ind.csv", index=False)

print("\nIND COMPLETE")
print("Outputs:", IND_DIR)
for p in sorted(IND_DIR.rglob("*")):
    if p.is_file():
        print(p.relative_to(ROOT))



# Part V — Comparative baselines and factorial SQM ablation

comparative study extends the independent validation math-audited notebook in two directions requested after the literature cross-check:

1. **Direct competitor studies** against four relevant noncoherent short-packet / Grassmannian families:
   - *Hardware Impairments-Aware Design of Noncoherent Grassmannian Constellations* (Cuevas et al., ICASSP 2024),
   - *Cube-Split: A Structured Grassmannian Constellation for Non-Coherent SIMO Communications* (Ngo et al.),
   - *Spectrally-Efficient Modulation on Conjugate-Reciprocal Zeros (SE-MOCZ) for Non-Coherent Short Packet Communications* (Siddiqui et al.),
   - *Index Modulation for Modulation on Conjugate-Reciprocal Zeros (IM-MOCZ)* (Corbett and Bedeer).

2. **A 2×2 SQM ablation** that separately removes:
   - incidence-aware codebook training, and
   - selective rank truncation.

## Scientific-comparison rule

The four competitors do **not** have the same native system model. comparative study therefore does **not** collapse them into one misleading universal curve.

- **Flat/HWI family:** SQM vs. a single-stream adaptation of the hardware-aware Grassmannian design, with Cube-Split as an HWI-unaware Grassmannian control.
- **Multipath short-packet family:** SQM vs. BMOCZ, SE-MOCZ source-model implementation, and IM-MOCZ.
- **Cube-Split native check:** the exact CS(4,1) 256-symbol construction is reproduced independently and also embedded isometrically into 16 samples for an equal-blocklength flat-fading diagnostic.

Every adapted competitor is labelled explicitly as an **independent matched implementation**, not as an exact reproduction of the numerical curve in the cited paper.


In [ ]:

from pathlib import Path
import os, sys, math, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import norm
from scipy.linalg import toeplitz
from IPython.display import display

# Comparative section is self-contained with respect to the released repository.
ROOT = Path.cwd()
CODE_DIR = ROOT / "Code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from fractional_tdla_core import (
    tdla_operators,
    weighted_nuisance_dictionary,
    exact_fractional_channel,
    random_tdla_coefficients,
)

COMP_DIR = ROOT / "Reproduction_Work" / "results" / "comparative_baselines_ablation"
COMP_DIR.mkdir(parents=True, exist_ok=True)

# "manuscript" is the intended final run. Change to "smoke" only for a quick test.
COMP_MODE = os.environ.get("SQM_COMPARISON_MODE", "manuscript").strip().lower()
COMP_FORCE = False

if COMP_MODE == "manuscript":
    COMP_SEEDS = [13, 59, 151]
    COMP_SNRS = [5, 7, 10, 12, 15]
    COMP_ABLATION_BLOCKS = 1400
    COMP_FLAT_BLOCKS = 1000
    COMP_MOCZ_BLOCKS = 600
    COMP_RADIUS_BLOCKS = 180
    COMP_UNIFIED_BLOCKS = 700
    HWI_NITER_CFO = 50
    HWI_NCFO = 20
    HWI_NITER_IQI = 20
else:
    COMP_SEEDS = [13]
    COMP_SNRS = [10]
    COMP_ABLATION_BLOCKS = 100
    COMP_FLAT_BLOCKS = 100
    COMP_MOCZ_BLOCKS = 80
    COMP_RADIUS_BLOCKS = 50
    COMP_UNIFIED_BLOCKS = 80
    HWI_NITER_CFO = 4
    HWI_NCFO = 6
    HWI_NITER_IQI = 4

T_CRIT_DF2 = 4.302652729911275

def t_halfwidth(x):
    x = np.asarray(x, float)
    if len(x) < 2:
        return 0.0
    # Manuscript mode uses 3 independent seeds -> df=2.
    if len(x) == 3:
        crit = T_CRIT_DF2
    else:
        from scipy.stats import t
        crit = float(t.ppf(0.975, len(x)-1))
    return crit * x.std(ddof=1) / np.sqrt(len(x))

def summarize_per_seed(df, group_cols=("snr_db", "method")):
    rows = []
    for key, g in df.groupby(list(group_cols), sort=True):
        if not isinstance(key, tuple):
            key = (key,)
        row = {c: v for c, v in zip(group_cols, key)}
        x = g["PER"].to_numpy(float)
        row.update(
            PER_mean=float(x.mean()),
            PER_95CI_halfwidth=float(t_halfwidth(x)),
            seeds=int(g["seed"].nunique()),
            blocks=int(g["blocks"].sum()),
        )
        rows.append(row)
    return pd.DataFrame(rows)

def save_current_figure(stem):
    png = COMP_DIR / f"{stem}.png"
    pdf = COMP_DIR / f"{stem}.pdf"
    plt.savefig(png, dpi=220, bbox_inches="tight")
    plt.savefig(pdf, bbox_inches="tight")
    print("Saved:", png.relative_to(ROOT))
    print("Saved:", pdf.relative_to(ROOT))

source_scope = pd.DataFrame([
    ["SQM", "8 bits / 16 complex samples", "TDL-A + CFO + static I/Q + DC", "Released implementation"],
    ["Cube-Split", "CS(4,1): 256 symbols in T=4", "Flat block fading, SIMO; N=1 used here", "Exact source construction; optional 16-sample isometric embedding"],
    ["HWI-aware Grassmannian", "Source paper: K=64, M=2, T=4/8", "Flat block fading + CFO + TX I/Q imbalance", "Single-stream M=1, T=16, K=256 adaptation"],
    ["SE-MOCZ", "Source examples use K=7/40/70", "Frequency-selective multipath + FTN pulse model", "8-bit SE-BMOCZ source-model implementation, tau=0.9, beta=0.3"],
    ["IM-MOCZ", "Source equal-bit example N=10, K=8/6/4", "Frequency-selective multipath + AWGN", "Matched payload N=8, K=6 using source DiZeT/majority-vote equations"],
], columns=["scheme", "source/native setting", "native nuisance model", "comparative study role"])

print("=== Comparative benchmark configuration ===")
print("Run mode:", COMP_MODE)
print("Seeds:", COMP_SEEDS)
print("SNRs :", COMP_SNRS)
print("Output:", COMP_DIR)
display(source_scope)
source_scope.to_csv(COMP_DIR / "source_scope_and_fairness.csv", index=False)



## V.1 — Factorial ablation: selective rank × incidence-aware training

This is the clean ablation that the manuscript was previously missing.

The experiment uses exactly the same TDL-A nuisance/noise draws for all four cells of a 2×2 design:

| Training | Rank | Meaning |
|---|---:|---|
| incidence-trained | 4 | full proposed SQM |
| incidence-trained | 15 | remove selective truncation |
| pre-incidence stage-3 | 4 | remove incidence-aware training |
| pre-incidence stage-3 | 15 | remove both |

For the actual \(L=16\) centered block, the saturated full explanation rank is at most \(L-1=15\), not \(2P=46\).


The released `train_selective_incidence.py` initializes incidence-aware training from `Artifact/stage3_codebooks.npz::X_dmm`, so the stage-3 codebook is the natural released **pre-incidence** control.


In [ ]:

# ---------- Shared TDL-A ablation helpers ----------

M5, L5 = 256, 16
GRID5_DEG = np.arange(-5.0, 6.0, 1.0)
GRID5_RAD = np.deg2rad(GRID5_DEG)
R_SEL, R_FULL = 4, 15

X_inc_r = np.load(ROOT / "Artifact" / "fractional_incidence_codebook.npz")["X_opt"]
X_pre_r = np.load(ROOT / "Artifact" / "stage3_codebooks.npz")["X_dmm"]
X_inc = X_inc_r[:, 0] + 1j * X_inc_r[:, 1]
X_pre = X_pre_r[:, 0] + 1j * X_pre_r[:, 1]

delays5, powers5, D5 = tdla_operators(L5)

def comp_rot(th):
    c, s = np.cos(th), np.sin(th)
    return np.array([[c, -s], [s, c]], dtype=float)

def comp_random_A(rng, N, kappa=1.5):
    slo = np.sqrt(2.0 / (1.0 + kappa*kappa))
    shi = kappa * slo
    a = rng.uniform(0, 2*np.pi, N)
    b = rng.uniform(0, 2*np.pi, N)
    A = np.empty((N, 2, 2), float)
    for i in range(N):
        A[i] = comp_rot(a[i]) @ np.diag([shi, slo]) @ comp_rot(b[i])
    return A

def build_tdla_bank_for_codebook(Xd, rmax=15):
    bank = np.empty((M5, len(GRID5_RAD), L5, rmax), dtype=np.float32)
    for m in range(M5):
        for j, omega in enumerate(GRID5_RAD):
            F = weighted_nuisance_dictionary(Xd[m], omega, powers5, D5)
            _, _, Vh = np.linalg.svd(F, full_matrices=False)
            rr = min(rmax, Vh.shape[0])
            bank[m, j, :, :rr] = Vh[:rr].T.astype(np.float32)
            if rr < rmax:
                bank[m, j, :, rr:] = 0.0
    return bank

def draw_common_tdla_nuisance(snr_db, blocks, seed):
    rng = np.random.default_rng(seed)
    idx = rng.integers(M5, size=blocks)
    h = random_tdla_coefficients(rng, blocks, powers5)
    omega = rng.uniform(-np.deg2rad(4.0), np.deg2rad(4.0), blocks)
    A = comp_random_A(rng, blocks, 1.5)
    dcang = rng.uniform(0, 2*np.pi, blocks)
    d = np.stack([np.cos(dcang), np.sin(dcang)], axis=1)
    N0 = 10.0**(-snr_db/10.0)
    noise = np.sqrt(N0/2.0) * rng.normal(size=(blocks, 2, L5))
    return idx, h, omega, A, d, noise

def apply_tdla_nuisance(Xd, nuisance):
    idx, h, omega, A, d, noise = nuisance
    s = exact_fractional_channel(Xd[idx], h, D5)
    s *= np.exp(1j * omega[:, None] * np.arange(L5)[None, :])
    sr = np.stack([s.real, s.imag], axis=1)
    return np.einsum("nij,njk->nik", A, sr, optimize=True) + d[:, :, None] + noise

def decode_tdla_bank(Y, bank, rank, batch=32):
    Yc = Y - Y.mean(axis=2, keepdims=True)
    B = bank[..., :rank]
    pred = np.empty(len(Y), int)
    for s in range(0, len(Y), batch):
        e = min(len(Y), s+batch)
        Z = np.einsum("bal,mwlr->bmwar", Yc[s:e], B, optimize=True)
        score = np.sum(Z*Z, axis=(3,4))
        pred[s:e] = np.argmax(score.max(axis=2), axis=1)
    return pred

bank_inc_path = COMP_DIR / "ablation_bank_incidence_r15.npz"
bank_pre_path = COMP_DIR / "ablation_bank_preincidence_r15.npz"

if bank_inc_path.exists() and bank_pre_path.exists() and not COMP_FORCE:
    bank_inc = np.load(bank_inc_path)["bank"]
    bank_pre = np.load(bank_pre_path)["bank"]
    print("Loaded cached comparative study ablation basis banks.")
else:
    print("Building incidence-trained rank-15 bank ...")
    bank_inc = build_tdla_bank_for_codebook(X_inc, R_FULL)
    np.savez_compressed(bank_inc_path, bank=bank_inc)
    print("Building pre-incidence rank-15 bank ...")
    bank_pre = build_tdla_bank_for_codebook(X_pre, R_FULL)
    np.savez_compressed(bank_pre_path, bank=bank_pre)

ablation_ps_path = COMP_DIR / "sqm_factorial_ablation_per_seed.csv"

if ablation_ps_path.exists() and not COMP_FORCE:
    ablation_ps = pd.read_csv(ablation_ps_path)
    print("Loaded cached factorial ablation.")
else:
    rows = []
    configs = [
        ("SQM: incidence + rank 4", X_inc, bank_inc, R_SEL),
        ("Ablation: incidence + full rank", X_inc, bank_inc, R_FULL),
        ("Ablation: pre-incidence + rank 4", X_pre, bank_pre, R_SEL),
        ("Ablation: pre-incidence + full rank", X_pre, bank_pre, R_FULL),
    ]
    for snr in COMP_SNRS:
        for seed in COMP_SEEDS:
            nuisance = draw_common_tdla_nuisance(
                snr, COMP_ABLATION_BLOCKS, 20261025 + 1000*int(snr) + seed
            )
            idx = nuisance[0]
            for name, Xd, bank, rank in configs:
                Y = apply_tdla_nuisance(Xd, nuisance)
                pred = decode_tdla_bank(Y, bank, rank)
                rows.append({
                    "snr_db": snr,
                    "seed": seed,
                    "method": name,
                    "rank": rank,
                    "PER": float(np.mean(pred != idx)),
                    "blocks": COMP_ABLATION_BLOCKS,
                })
                print(f"Ablation SNR={snr:>4} seed={seed:>3} {name:38s} PER={rows[-1]['PER']:.5f}")
    ablation_ps = pd.DataFrame(rows)
    ablation_ps.to_csv(ablation_ps_path, index=False)

ablation_summary = summarize_per_seed(ablation_ps)
ablation_summary.to_csv(COMP_DIR / "sqm_factorial_ablation_summary.csv", index=False)
display(ablation_summary)

# Paired factorial effects: positive values mean the named SQM component reduces PER.
effect_rows = []
for snr, g in ablation_ps.groupby("snr_db"):
    p = g.pivot(index="seed", columns="method", values="PER")
    effects = {
        "incidence benefit at rank 4":
            p["Ablation: pre-incidence + rank 4"] - p["SQM: incidence + rank 4"],
        "incidence benefit at full rank":
            p["Ablation: pre-incidence + full rank"] - p["Ablation: incidence + full rank"],
        "selective-rank benefit with incidence":
            p["Ablation: incidence + full rank"] - p["SQM: incidence + rank 4"],
        "selective-rank benefit pre-incidence":
            p["Ablation: pre-incidence + full rank"] - p["Ablation: pre-incidence + rank 4"],
    }
    effects["interaction (incidence changes rank benefit)"] = (
        effects["selective-rank benefit with incidence"] -
        effects["selective-rank benefit pre-incidence"]
    )
    for name, x in effects.items():
        effect_rows.append({
            "snr_db": snr,
            "effect": name,
            "mean_delta_PER": float(x.mean()),
            "paired_95CI_halfwidth": float(t_halfwidth(x.to_numpy())),
            "seeds": len(x),
        })

ablation_effects = pd.DataFrame(effect_rows)
ablation_effects.to_csv(COMP_DIR / "sqm_factorial_ablation_effects.csv", index=False)
display(ablation_effects)

plt.figure(figsize=(7.0, 4.5))
markers = ["o", "s", "^", "D"]
for marker, (name, g) in zip(markers, ablation_summary.groupby("method", sort=False)):
    g = g.sort_values("snr_db")
    plt.errorbar(g["snr_db"], g["PER_mean"], yerr=g["PER_95CI_halfwidth"],
                 marker=marker, capsize=3, label=name)
plt.yscale("log")
plt.xlabel("SNR (dB)")
plt.ylabel("Packet error rate")
plt.title("SQM factorial ablation")
plt.grid(True, which="both", alpha=0.25)
plt.legend(fontsize=8)
plt.tight_layout()
save_current_figure("sqm_factorial_ablation")
plt.show()



## V.2 — Cube-Split: exact CS(4,1) construction and equal-blocklength flat-fading diagnostic

For Cube-Split, comparative study independently implements the source construction:

- \(T=4\),
- \(B_0=1\),
- \(4\times2^{2(T-1)B_0}=256\) Grassmannian lines,
- the source mappings in Eqs. (9)–(15),
- exact noncoherent ML line detection.

The source paper itself uses **CS(4,1), 256 symbols** in its single-receive-antenna BER example. comparative study checks that construction directly.

Because SQM uses 16 complex samples, comparative study also creates an **isometric 4→16 embedding** of CS(4,1). This preserves all pairwise Grassmannian inner products but is explicitly an equal-blocklength diagnostic, not the native Cube-Split waveform.


In [ ]:

def cube_split_cs4_b1():
    T = 4
    # Eq. (9), B_j = 1 -> {1/4, 3/4} for each of 2(T-1)=6 real coordinates.
    vals = np.array([0.25, 0.75])
    coords = np.array(np.meshgrid(*([vals]*6), indexing="ij")).reshape(6, -1).T
    code = []
    labels = []
    for cell in range(T):
        for q, a in enumerate(coords):
            tcoords = []
            for j in range(T-1):
                w = norm.ppf(a[2*j]) + 1j*norm.ppf(a[2*j+1])
                r2 = abs(w)**2
                # Eq. (12)
                amp = np.sqrt((1-np.exp(-r2/2)) / (1+np.exp(-r2/2))) if r2 > 1e-14 else 0.0
                tj = amp * w/abs(w) if abs(w) > 1e-14 else 0.0j
                tcoords.append(tj)
            # Eq. (11): insert "1" in the cell coordinate and normalize.
            x = np.empty(T, complex)
            x[cell] = 1.0
            k = 0
            for i in range(T):
                if i != cell:
                    x[i] = tcoords[k]
                    k += 1
            x /= np.linalg.norm(x)
            code.append(x)
            labels.append((cell, q))
    return np.asarray(code), labels

CUBE4, cube_labels = cube_split_cs4_b1()
assert CUBE4.shape == (256, 4)
assert np.allclose(np.linalg.norm(CUBE4, axis=1), 1.0)

# Pairwise-distance sanity check.
cube_gram = np.abs(CUBE4 @ CUBE4.conj().T)**2
np.fill_diagonal(cube_gram, 0.0)
cube_max_coherence = float(cube_gram.max())
cube_min_chordal = float(np.sqrt(1.0 - cube_max_coherence))

# Fixed semi-unitary 4->16 DFT embedding: preserves all pairwise inner products.
n = np.arange(16)[:, None]
k = np.arange(4)[None, :]
U16x4 = np.exp(2j*np.pi*n*k/16) / 4.0
assert np.allclose(U16x4.conj().T @ U16x4, np.eye(4), atol=1e-12)
CUBE16 = CUBE4 @ U16x4.conj().T
CUBE16 *= 4.0  # equal energy 16 for the 16-sample diagnostic

np.savez_compressed(
    COMP_DIR / "cube_split_cs4_b1_and_embedding.npz",
    cube4=CUBE4,
    cube16=CUBE16,
    max_coherence=cube_max_coherence,
    min_chordal=cube_min_chordal,
)

print("Cube-Split CS(4,1) size:", CUBE4.shape)
print("Minimum chordal distance:", cube_min_chordal)
print("16-sample embedding energy:", np.mean(np.sum(np.abs(CUBE16)**2, axis=1)))

def cube_ml_decode(y, C):
    # Noncoherent line ML: maximize |y^H c|^2.
    return np.argmax(np.abs(y @ C.conj().T)**2, axis=1)

def flat_sqm_basis_bank(Xd, cfo_grid_deg=(0.0,)):
    Xd = np.asarray(Xd)
    L = Xd.shape[1]
    H = np.eye(L) - np.ones((L,L))/L
    grid = np.asarray(cfo_grid_deg, float)
    bank = np.empty((len(Xd), len(grid), L, 2), np.float32)
    t = np.arange(L)
    for m, x in enumerate(Xd):
        for j, deg in enumerate(grid):
            v = x * np.exp(1j*np.deg2rad(deg)*t)
            F = np.stack([v.real, v.imag]) @ H
            _, _, Vh = np.linalg.svd(F, full_matrices=False)
            bank[m,j] = Vh[:2].T.astype(np.float32)
    return bank

SQM_FLAT0_BANK = flat_sqm_basis_bank(X_inc, [0.0])

def decode_sqm_flat(y_complex, bank, batch=64):
    Y = np.stack([y_complex.real, y_complex.imag], axis=1)
    Y -= Y.mean(axis=2, keepdims=True)
    pred = np.empty(len(Y), int)
    for s in range(0, len(Y), batch):
        e = min(len(Y), s+batch)
        Z = np.einsum("bal,mwlr->bmwar", Y[s:e], bank, optimize=True)
        sc = np.sum(Z*Z, axis=(3,4))
        pred[s:e] = np.argmax(sc.max(axis=2), axis=1)
    return pred

cube_ps_path = COMP_DIR / "cube_split_flat_comparison_per_seed.csv"
if cube_ps_path.exists() and not COMP_FORCE:
    cube_ps = pd.read_csv(cube_ps_path)
else:
    rows = []
    Xsqm = X_inc.copy()  # already energy 16
    for snr in COMP_SNRS:
        for seed in COMP_SEEDS:
            rng = np.random.default_rng(20262000 + int(snr)*1000 + seed)
            idx = rng.integers(256, size=COMP_FLAT_BLOCKS)
            h = (rng.normal(size=COMP_FLAT_BLOCKS) + 1j*rng.normal(size=COMP_FLAT_BLOCKS))/np.sqrt(2)
            N0 = 10.0**(-snr/10.0)

            # Equal 16-sample / equal-energy diagnostic.
            noise16a = np.sqrt(N0/2)*(rng.normal(size=(COMP_FLAT_BLOCKS,16)) + 1j*rng.normal(size=(COMP_FLAT_BLOCKS,16)))
            noise16b = np.sqrt(N0/2)*(rng.normal(size=(COMP_FLAT_BLOCKS,16)) + 1j*rng.normal(size=(COMP_FLAT_BLOCKS,16)))
            y_sqm = h[:,None]*Xsqm[idx] + noise16a
            y_cube16 = h[:,None]*CUBE16[idx] + noise16b

            p_sqm = decode_sqm_flat(y_sqm, SQM_FLAT0_BANK)
            p_cube16 = cube_ml_decode(y_cube16, CUBE16)

            # Native CS(4,1): four channel uses, unit-energy source codeword.
            noise4 = np.sqrt(N0/2)*(rng.normal(size=(COMP_FLAT_BLOCKS,4)) + 1j*rng.normal(size=(COMP_FLAT_BLOCKS,4)))
            y_cube4 = h[:,None]*(2.0*CUBE4[idx]) + noise4  # energy T=4, matching unit average power per channel use
            p_cube4 = cube_ml_decode(y_cube4, CUBE4)

            for method, per, uses in [
                ("SQM flat quotient, L=16", np.mean(p_sqm != idx), 16),
                ("Cube-Split CS(4,1), 16-sample isometric embedding", np.mean(p_cube16 != idx), 16),
                ("Cube-Split CS(4,1), native T=4", np.mean(p_cube4 != idx), 4),
            ]:
                rows.append({
                    "snr_db": snr, "seed": seed, "method": method,
                    "PER": float(per), "blocks": COMP_FLAT_BLOCKS,
                    "channel_uses": uses, "payload_bits": 8,
                    "bits_per_channel_use": 8/uses,
                })
    cube_ps = pd.DataFrame(rows)
    cube_ps.to_csv(cube_ps_path, index=False)

cube_summary = summarize_per_seed(cube_ps)
cube_meta = cube_ps.groupby("method", as_index=False)[["channel_uses","payload_bits","bits_per_channel_use"]].first()
cube_summary = cube_summary.merge(cube_meta, on="method", how="left")
cube_summary.to_csv(COMP_DIR / "cube_split_flat_comparison_summary.csv", index=False)
display(cube_summary)

plt.figure(figsize=(7.0, 4.5))
for marker, (name, g) in zip(["o","s","^"], cube_summary.groupby("method", sort=False)):
    g = g.sort_values("snr_db")
    plt.errorbar(g.snr_db, g.PER_mean, yerr=g.PER_95CI_halfwidth,
                 marker=marker, capsize=3, label=name)
plt.yscale("log")
plt.xlabel("SNR (dB)")
plt.ylabel("Packet error rate")
plt.title("Cube-Split source construction and equal-blocklength flat-fading diagnostic")
plt.grid(True, which="both", alpha=0.25)
plt.legend(fontsize=8)
plt.tight_layout()
save_current_figure("comp_cube_split_flat_comparison")
plt.show()



## V.3 — Hardware-aware Grassmannian baseline under matched flat-fading hardware uncertainty

The ICASSP 2024 paper designs Grassmannian constellations robust to bounded CFO and transmitter I/Q imbalance. Its published numerical setup is MIMO (\(M=2\), \(K=64\)); Kumar's tested link is one complex stream with 256 messages.

comparative study therefore implements a **single-stream \(M=1\) adaptation** of the paper's Algorithm 1:

- 256 Grassmannian lines,
- \(T=16\),
- CFO uncertainty \(\pm10^\circ\)/sample,
- amplitude imbalance up to 3 dB,
- phase imbalance up to \(15^\circ\),
- source algorithm parameters \(\mu=0.1\), \(N_{\rm CFO}=20\), \(N_{\rm iter,CFO}=50\), \(N_{\rm iter,IQI}=20\), \(q=5\) in manuscript mode,
- CFO-centroid update using the dominant eigenvector of the mean projection matrix,
- I/Q perturbation/retraction,
- line-packing repulsion using the \(M=1\) minimum-diversity-product equivalent (minimise worst pairwise coherence).

This is a deliberately strong, source-inspired matched baseline, but it is **not** claimed to reproduce the paper's \(M=2\) numerical curves.


In [ ]:

def line_repel_step(C, mu=0.1):
    C = C.copy()
    Gram = C @ C.conj().T
    np.fill_diagonal(Gram, 0.0)
    partner = np.argmax(np.abs(Gram)**2, axis=1)
    inner = np.sum(C[partner].conj()*C, axis=1)  # c_j^H c_k
    g = C[partner] * inner[:,None]
    # Riemannian projection to the tangent space of the complex unit sphere.
    g -= C * np.sum(C.conj()*g, axis=1)[:,None]
    C -= mu*g
    C /= np.linalg.norm(C, axis=1, keepdims=True)
    return C

def iq_perturb_lines(C, g_db, theta_rad):
    G = 10.0**(g_db/20.0)
    a = (1 + G*np.exp(1j*theta_rad))/2
    b = (1 - G*np.exp(-1j*theta_rad))/2
    Z = a*C + b*C.conj()
    return Z / np.linalg.norm(Z, axis=1, keepdims=True)

def cfo_centroid_update(C, rng, cfo_max_deg=10.0, n_cfo=20):
    T = C.shape[1]
    t = np.arange(T)
    out = np.empty_like(C)
    for m, x in enumerate(C):
        om = np.deg2rad(rng.uniform(-cfo_max_deg, cfo_max_deg, n_cfo))
        F = np.exp(1j*om[:,None]*t[None,:]) * x[None,:]
        Pbar = (F.T @ F.conj()) / n_cfo
        _, V = np.linalg.eigh(Pbar)
        out[m] = V[:, -1]
    out /= np.linalg.norm(out, axis=1, keepdims=True)
    return out

def train_hwi_aware_lines(
    C0, seed=20261025, cfo_max_deg=10.0, gmax_db=3.0, theta_max_deg=15.0,
    mu=0.1, niter_cfo=50, ncfo=20, niter_iqi=20, q=5
):
    rng = np.random.default_rng(seed)
    C = C0.copy()
    for outer in range(niter_cfo):
        C = cfo_centroid_update(C, rng, cfo_max_deg, ncfo)
        for it in range(1, niter_iqi+1):
            if it % q == 0:
                gd = float(rng.uniform(0.0, gmax_db))
                th = float(np.deg2rad(rng.uniform(0.0, theta_max_deg)))
                C = iq_perturb_lines(C, gd, th)
            else:
                C = line_repel_step(C, mu)
        if (outer+1) % max(1, niter_cfo//5) == 0:
            Gm = np.abs(C @ C.conj().T)**2
            np.fill_diagonal(Gm, 0.0)
            print(f"HWI design {outer+1:3d}/{niter_cfo}: worst coherence={np.sqrt(Gm.max()):.4f}")
    return C

hwi_code_path = COMP_DIR / "hwi_aware_grassmann_single_stream_codebook.npz"
if hwi_code_path.exists() and not COMP_FORCE:
    HWI16 = np.load(hwi_code_path)["codebook"]
    print("Loaded cached single-stream HWI-aware Grassmannian codebook.")
else:
    print("Training single-stream HWI-aware Grassmannian codebook ...")
    t0 = time.time()
    # Source paper starts from an HWI-unaware Grassmannian codebook.
    # CS(4,1) embedded in C^16 is a deterministic, structured HWI-unaware initializer.
    HWI16 = train_hwi_aware_lines(
        CUBE16 / 4.0,
        niter_cfo=HWI_NITER_CFO,
        ncfo=HWI_NCFO,
        niter_iqi=HWI_NITER_IQI,
        q=5,
    )
    HWI16 *= 4.0
    np.savez_compressed(hwi_code_path, codebook=HWI16)
    print(f"Training completed in {time.time()-t0:.1f} s")

# SQM bank covers the same guarded CFO support used for the matched HWI experiment.
HWI_GRID_DEG = np.arange(-11.0, 12.0, 1.0)
SQM_HWI_BANK = flat_sqm_basis_bank(X_inc, HWI_GRID_DEG)

def apply_hwi_channel(C, idx, h, omega, gdb, theta, noise):
    x = C[idx]
    G = 10.0**(gdb/20.0)
    a = (1 + G*np.exp(1j*theta))/2
    b = (1 - G*np.exp(-1j*theta))/2
    z = a[:,None]*x + b[:,None]*x.conj()
    z *= np.exp(1j*omega[:,None]*np.arange(C.shape[1])[None,:])
    return h[:,None]*z + noise

hwi_ps_path = COMP_DIR / "flat_hwi_comparison_per_seed.csv"
if hwi_ps_path.exists() and not COMP_FORCE:
    hwi_ps = pd.read_csv(hwi_ps_path)
else:
    rows = []
    for snr in COMP_SNRS:
        for seed in COMP_SEEDS:
            rng = np.random.default_rng(20263000 + int(snr)*1000 + seed)
            B = COMP_FLAT_BLOCKS
            idx = rng.integers(256, size=B)
            h = (rng.normal(size=B)+1j*rng.normal(size=B))/np.sqrt(2)
            omega = np.deg2rad(rng.uniform(-10.0, 10.0, B))
            gdb = rng.uniform(0.0, 3.0, B)
            theta = np.deg2rad(rng.uniform(0.0, 15.0, B))
            N0 = 10.0**(-snr/10.0)

            # Independent receiver noises, same nuisance draws.
            noises = [
                np.sqrt(N0/2)*(rng.normal(size=(B,16))+1j*rng.normal(size=(B,16)))
                for _ in range(3)
            ]

            Ysqm = apply_hwi_channel(X_inc, idx, h, omega, gdb, theta, noises[0])
            Yhwi = apply_hwi_channel(HWI16, idx, h, omega, gdb, theta, noises[1])
            Ycube = apply_hwi_channel(CUBE16, idx, h, omega, gdb, theta, noises[2])

            p_sqm = decode_sqm_flat(Ysqm, SQM_HWI_BANK)
            p_hwi = cube_ml_decode(Yhwi, HWI16)
            p_cube = cube_ml_decode(Ycube, CUBE16)

            for method, pred in [
                ("SQM quotient + CFO search", p_sqm),
                ("HWI-aware Grassmannian, single-stream adaptation", p_hwi),
                ("HWI-unaware Cube-Split embedding", p_cube),
            ]:
                rows.append({
                    "snr_db": snr, "seed": seed, "method": method,
                    "PER": float(np.mean(pred != idx)),
                    "blocks": B,
                    "payload_bits": 8,
                    "blocklength": 16,
                    "cfo_max_deg_per_sample": 10.0,
                    "iq_amp_max_db": 3.0,
                    "iq_phase_max_deg": 15.0,
                })
                print(f"HWI SNR={snr:>4} seed={seed:>3} {method:48s} PER={rows[-1]['PER']:.5f}")
    hwi_ps = pd.DataFrame(rows)
    hwi_ps.to_csv(hwi_ps_path, index=False)

hwi_summary = summarize_per_seed(hwi_ps)
hwi_summary.to_csv(COMP_DIR / "flat_hwi_comparison_summary.csv", index=False)
display(hwi_summary)

plt.figure(figsize=(7.0, 4.5))
for marker, (name, g) in zip(["o","s","^"], hwi_summary.groupby("method", sort=False)):
    g = g.sort_values("snr_db")
    plt.errorbar(g.snr_db, g.PER_mean, yerr=g.PER_95CI_halfwidth,
                 marker=marker, capsize=3, label=name)
plt.yscale("log")
plt.xlabel("SNR (dB)")
plt.ylabel("Packet error rate")
plt.title("Matched flat-fading hardware-uncertainty comparison")
plt.grid(True, which="both", alpha=0.25)
plt.legend(fontsize=8)
plt.tight_layout()
save_current_figure("flat_hardware_uncertainty_comparison")
plt.show()



## V.4 — Multipath short-packet family: SQM vs BMOCZ, SE-MOCZ and IM-MOCZ

This section uses an 8-bit payload and a common two-tap Rayleigh multipath law with PDP \([0.8,0.2]\).

### What is exact and what is adapted

- **SQM:** released 10-sample two-tap incidence-trained codebook and rank-4 quotient decoder.
- **BMOCZ:** standard conjugate-reciprocal-zero construction with Gaussian true-PDP marginal ML.
- **SE-MOCZ:** independent implementation of the paper's linear model \(y=x*g*h+w\), with \(\tau=0.9\), SRRC roll-off \(\beta=0.3\), and \(2L_g+1=17\). comparative study uses an exact Gaussian marginal-ML detector under that source model. This is a strong source-model receiver, not a reproduction of the paper's RFMD curve or its partial-zero-removal filter.
- **IM-MOCZ:** matched 8-bit adaptation with \(N=8,K=6\); four phase-rotated BMOCZ codebooks and source DiZeT penalty / majority-vote logic.

Because the native waveform durations differ, comparative study reports **PER and nominal information efficiency separately** instead of hiding the mismatch.


In [ ]:

# ---------- MOCZ family helpers ----------

PDP2 = np.array([0.8, 0.2], float)
E_MATCH = 10.0

# Released 10-sample SQM comparison artifact.
X_2TAP_R = np.load(ROOT / "Artifact" / "two_tap_incidence_codebook.npz")["X_inc"]
X_2TAP = X_2TAP_R[:,0] + 1j*X_2TAP_R[:,1]

def bits_lsb(M=256, K=8):
    return np.array([[(m>>k)&1 for k in range(K)] for m in range(M)], dtype=np.int8)

BITS8_LSB = bits_lsb()

def bmocz_codebook(K=8, R=None, total_energy=E_MATCH):
    if R is None:
        R = np.sqrt(1 + np.sin(np.pi/K))
    M = 2**K
    B = bits_lsb(M, K)
    ang = 2*np.pi*np.arange(K)/K
    X = np.empty((M, K+1), complex)
    for m in range(M):
        roots = np.where(B[m] == 1, R, 1/R) * np.exp(1j*ang)
        x = np.poly(roots)[::-1]
        X[m] = x / np.linalg.norm(x) * np.sqrt(total_energy)
    return X, R

def im_mocz_codebook(N=8, K=6, R=None, total_energy=E_MATCH):
    if R is None:
        R = np.sqrt(1 + np.sin(np.pi/K))
    M = 2**N
    nc = 2**(N-K)
    X = np.empty((M, K+1), complex)
    for m in range(M):
        bits = np.array([(m>>(N-1-j))&1 for j in range(N)], dtype=int)  # MSB first
        ci = 0
        for b in bits[:N-K]:
            ci = (ci << 1) | int(b)
        theta_i = 2*np.pi*ci/(K*nc)
        radial = bits[N-K:]
        roots = np.where(radial == 1, R, 1/R) * np.exp(
            1j*(2*np.pi*np.arange(K)/K + theta_i)
        )
        x = np.poly(roots)[::-1]
        X[m] = x / np.linalg.norm(x) * np.sqrt(total_energy)
    return X, R

def conv_design(x, g, Lh):
    s = np.convolve(x, g)
    U = np.zeros((len(s)+Lh-1, Lh), complex)
    for p in range(Lh):
        U[p:p+len(s), p] = s
    return U

def raised_cosine_samples(integer_lags, tau=0.9, beta=0.3):
    # Matched SRRC Tx/Rx gives a raised-cosine correlation pulse.
    t = np.asarray(integer_lags, float) * tau
    den = 1 - (2*beta*t)**2
    out = np.empty_like(t)
    regular = np.abs(den) > 1e-7
    out[regular] = np.sinc(t[regular]) * np.cos(np.pi*beta*t[regular]) / den[regular]
    # Stable numerical limit at the removable singularity.
    for q in np.where(~regular)[0]:
        tt = t[q] + 1e-6
        out[q] = np.sinc(tt) * np.cos(np.pi*beta*tt) / (1-(2*beta*tt)**2)
    return out

def build_gaussian_marginal_bank(X, g, pdp, N0, tau_noise=None, beta=0.3):
    pdp = np.asarray(pdp, float)
    Lh = len(pdp)
    U = np.stack([conv_design(x, g, Lh) for x in X])  # M x N x Lh
    N = U.shape[1]

    if tau_noise is None:
        Rw = np.eye(N)
    else:
        first = raised_cosine_samples(np.arange(N), tau=tau_noise, beta=beta)
        Rw = toeplitz(first)
        lam_min = np.linalg.eigvalsh(Rw).min()
        if lam_min < 1e-8:
            Rw += np.eye(N)*(1e-8-lam_min+1e-10)

    Winv = np.linalg.inv(N0*Rw)
    WU = np.einsum("ab,mbj->maj", Winv, U, optimize=True)
    G = np.einsum("mai,maj->mij", U.conj(), WU, optimize=True)

    Dinv = np.diag(1/pdp)
    Binv = np.linalg.inv(Dinv[None,:,:] + G)
    sqrtD = np.diag(np.sqrt(pdp))
    DG = np.einsum("ij,mjk,kl->mil", sqrtD, G, sqrtD, optimize=True)
    logterm = np.array([
        np.linalg.slogdet(np.eye(Lh) + DG[m])[1].real for m in range(len(X))
    ])
    return dict(U=U, Rw=Rw, Winv=Winv, Binv=Binv, logterm=logterm)

def decode_gaussian_marginal(y, bank, batch=96):
    U, Winv, Binv, logterm = bank["U"], bank["Winv"], bank["Binv"], bank["logterm"]
    pred = np.empty(len(y), int)
    for s in range(0, len(y), batch):
        yy = y[s:s+batch]
        Wy = yy @ Winv.T
        z = np.einsum("mni,bn->bmi", U.conj(), Wy, optimize=True)
        gain = np.einsum("bmi,mij,bmj->bm", z.conj(), Binv, z, optimize=True).real
        metric = logterm[None,:] - gain
        pred[s:s+len(yy)] = np.argmin(metric, axis=1)
    return pred

def simulate_from_marginal_bank(idx, h, bank, N0, rng):
    U = bank["U"]
    y = np.einsum("bni,bi->bn", U[idx], h, optimize=True)
    Lc = np.linalg.cholesky(bank["Rw"])
    z = (rng.normal(size=y.shape) + 1j*rng.normal(size=y.shape))/np.sqrt(2)
    y += np.sqrt(N0) * (z @ Lc.T)
    return y

def conv_channel_batch(Xsel, h):
    B, Lx = Xsel.shape
    Lh = h.shape[1]
    y = np.zeros((B, Lx+Lh-1), complex)
    for p in range(Lh):
        y[:, p:p+Lx] += h[:,p,None]*Xsel
    return y

def im_dizet_decode(y, N=8, K=6, R=1.2):
    nc = 2**(N-K)
    theta_i = 2*np.pi*np.arange(nc)/(K*nc)
    base = 2*np.pi*np.arange(K)/K
    outer = R*np.exp(1j*(theta_i[:,None] + base[None,:]))
    inner = (1/R)*np.exp(1j*(theta_i[:,None] + base[None,:]))
    pred = np.empty(len(y), int)

    for b, yy in enumerate(y):
        roots = np.roots(yy[::-1])
        Mroot = len(roots)
        pout = np.empty((nc,K), float)
        pin = np.empty((nc,K), float)
        for i in range(nc):
            for k in range(K):
                pout[i,k] = np.prod(np.abs(outer[i,k] - roots))
                pin[i,k] = (R**Mroot)*np.prod(np.abs(inner[i,k] - roots))
        p = np.minimum(pout, pin)
        sector_vote = np.argmin(p, axis=0)
        counts = np.bincount(sector_vote, minlength=nc)
        ci = int(np.argmax(counts))  # deterministic tie handling
        radial = (pout[ci] < pin[ci]).astype(int)  # outer=1, inner=0

        bits = [(ci>>(N-K-1-j))&1 for j in range(N-K)] + radial.tolist()
        m = 0
        for bit in bits:
            m = (m << 1) | int(bit)
        pred[b] = m
    return pred

# Released SQM two-tap rank-4 projector decoder, written locally for comparative study.
def two_tap_projector_bank():
    M, L = 256, 10
    arr = np.empty((M, L, L), np.float32)
    for m in range(M):
        U = X_2TAP_R[m].copy()
        V = np.zeros_like(U)
        V[:,1:] = U[:,:-1]
        F = np.concatenate([U, V], axis=0)
        F -= F.mean(axis=-1, keepdims=True)
        Q = np.linalg.qr(F.T)[0][:,:4]
        arr[m] = (Q @ Q.T).astype(np.float32)
    return arr.reshape(256, 100)

P2T = two_tap_projector_bank()


def finite_block_two_tap(Xsel, h):
    """Released SQM two-tap convention: fixed 10-sample observation block."""
    y = np.zeros_like(Xsel, dtype=complex)
    y[:,0] = h[:,0] * Xsel[:,0]
    y[:,1:] = h[:,0,None]*Xsel[:,1:] + h[:,1,None]*Xsel[:,:-1]
    return y

def decode_sqm_two_tap(y):
    Y = np.stack([y.real, y.imag], axis=1)
    Y -= Y.mean(axis=2, keepdims=True)
    C = np.einsum("bil,bik->blk", Y, Y).reshape(len(Y), -1).astype(np.float32)
    return np.argmax(C @ P2T.T, axis=1)

# SE-MOCZ paper source parameters used in the matched adaptation.
SE_TAU = 0.9
SE_BETA = 0.3
SE_LG = 8  # source reports 2*Lg+1 = 17 at tau=0.9
SE_G = raised_cosine_samples(np.arange(-SE_LG, SE_LG+1), tau=SE_TAU, beta=SE_BETA)

print("SE pulse-model taps:", len(SE_G), "center=", SE_G[SE_LG])


In [ ]:

# ---------- Held-out radius validation ----------
# Radius selection is isolated from the evaluation seeds.
radius_path = COMP_DIR / "mocz_family_radius_validation.csv"

if radius_path.exists() and not COMP_FORCE:
    radius_df = pd.read_csv(radius_path)
else:
    radius_grid = np.round(np.arange(1.10, 1.71, 0.10), 2)
    rows = []
    snr_val = 10.0
    seed_val = 20269991
    B = COMP_RADIUS_BLOCKS

    for R in radius_grid:
        rng = np.random.default_rng(seed_val)
        idx = rng.integers(256, size=B)
        h = (rng.normal(size=(B,2)) + 1j*rng.normal(size=(B,2)))/np.sqrt(2)
        h *= np.sqrt(PDP2)[None,:]
        N0 = 10.0**(-snr_val/10.0)

        # BMOCZ
        Xb, _ = bmocz_codebook(8, R)
        bb = build_gaussian_marginal_bank(Xb, np.array([1.0]), PDP2, N0)
        yb = simulate_from_marginal_bank(idx, h, bb, N0, rng)
        pb = decode_gaussian_marginal(yb, bb)
        rows.append({"scheme":"BMOCZ marginal ML", "R":R, "PER":np.mean(pb!=idx)})

        # SE-BMOCZ source model
        Xs, _ = bmocz_codebook(8, R)
        bs = build_gaussian_marginal_bank(
            Xs, SE_G, PDP2, N0, tau_noise=SE_TAU, beta=SE_BETA
        )
        ys = simulate_from_marginal_bank(idx, h, bs, N0, rng)
        ps = decode_gaussian_marginal(ys, bs)
        rows.append({"scheme":"SE-BMOCZ source-model marginal ML", "R":R, "PER":np.mean(ps!=idx)})

        # IM-MOCZ DiZeT, N=8, K=6
        Xi, _ = im_mocz_codebook(8, 6, R)
        yi = conv_channel_batch(Xi[idx], h)
        ni = np.sqrt(N0/2)*(rng.normal(size=yi.shape)+1j*rng.normal(size=yi.shape))
        pi = im_dizet_decode(yi+ni, 8, 6, R)
        rows.append({"scheme":"IM-MOCZ N=8,K=6 DiZeT", "R":R, "PER":np.mean(pi!=idx)})

        print(f"Radius R={R:.2f} completed.")

    radius_df = pd.DataFrame(rows)
    radius_df.to_csv(radius_path, index=False)

best_radius = (
    radius_df.sort_values(["scheme","PER","R"])
    .groupby("scheme", as_index=False)
    .first()[["scheme","R","PER"]]
    .rename(columns={"PER":"validation_PER"})
)
display(best_radius)

R_BM = float(best_radius.loc[best_radius.scheme.eq("BMOCZ marginal ML"), "R"].iloc[0])
R_SE = float(best_radius.loc[best_radius.scheme.eq("SE-BMOCZ source-model marginal ML"), "R"].iloc[0])
R_IM = float(best_radius.loc[best_radius.scheme.eq("IM-MOCZ N=8,K=6 DiZeT"), "R"].iloc[0])

plt.figure(figsize=(7.0,4.5))
for marker, (name, g) in zip(["o","s","^"], radius_df.groupby("scheme", sort=False)):
    plt.plot(g.R, g.PER, marker=marker, label=name)
plt.yscale("log")
plt.xlabel("Conjugate-reciprocal radius R")
plt.ylabel("Held-out packet error rate at 10 dB")
plt.title("Held-out MOCZ-family radius validation")
plt.grid(True, which="both", alpha=0.25)
plt.legend(fontsize=8)
plt.tight_layout()
save_current_figure("mocz_radius_validation")
plt.show()


In [ ]:

# ---------- Main matched multipath comparison ----------
mocz_ps_path = COMP_DIR / "mocz_family_comparison_per_seed.csv"

if mocz_ps_path.exists() and not COMP_FORCE:
    mocz_ps = pd.read_csv(mocz_ps_path)
else:
    X_BM, _ = bmocz_codebook(8, R_BM)
    X_SE, _ = bmocz_codebook(8, R_SE)
    X_IM, _ = im_mocz_codebook(8, 6, R_IM)

    rows = []
    for snr in COMP_SNRS:
        N0 = 10.0**(-snr/10.0)
        bank_bm = build_gaussian_marginal_bank(X_BM, np.array([1.0]), PDP2, N0)
        bank_se = build_gaussian_marginal_bank(
            X_SE, SE_G, PDP2, N0, tau_noise=SE_TAU, beta=SE_BETA
        )

        for seed in COMP_SEEDS:
            rng = np.random.default_rng(20264000 + int(snr)*1000 + seed)
            B = COMP_MOCZ_BLOCKS
            idx = rng.integers(256, size=B)
            h = (rng.normal(size=(B,2))+1j*rng.normal(size=(B,2)))/np.sqrt(2)
            h *= np.sqrt(PDP2)[None,:]

            # SQM, released 10-sample two-tap construction.
            yq = finite_block_two_tap(X_2TAP[idx], h)
            nq = np.sqrt(N0/2)*(rng.normal(size=yq.shape)+1j*rng.normal(size=yq.shape))
            pq = decode_sqm_two_tap(yq+nq)

            # BMOCZ true-PDP Gaussian marginal ML.
            yb = simulate_from_marginal_bank(idx, h, bank_bm, N0, rng)
            pb = decode_gaussian_marginal(yb, bank_bm)

            # SE-BMOCZ source-model Gaussian marginal ML.
            ys = simulate_from_marginal_bank(idx, h, bank_se, N0, rng)
            ps = decode_gaussian_marginal(ys, bank_se)

            # IM-MOCZ source DiZeT / majority vote.
            yi = conv_channel_batch(X_IM[idx], h)
            ni = np.sqrt(N0/2)*(rng.normal(size=yi.shape)+1j*rng.normal(size=yi.shape))
            pi = im_dizet_decode(yi+ni, 8, 6, R_IM)

            methods = [
                ("SQM two-tap, rank 4", pq, 10, 8/10, np.nan),
                ("BMOCZ marginal ML", pb, 10, 8/10,
                 8/((8+len(PDP2))*(1+SE_BETA))),
                ("SE-BMOCZ source-model marginal ML", ps, len(ys[0]), 8/len(ys[0]),
                 8/(SE_TAU*(8+len(PDP2))*(1+SE_BETA))),
                ("IM-MOCZ N=8,K=6 DiZeT", pi, len(yi[0]), 8/len(yi[0]),
                 8/(6+len(PDP2))),
            ]
            for method, pred, obs_len, bits_per_obs, source_se in methods:
                rows.append({
                    "snr_db": snr, "seed": seed, "method": method,
                    "PER": float(np.mean(pred != idx)),
                    "blocks": B,
                    "payload_bits": 8,
                    "observation_samples": int(obs_len),
                    "bits_per_observation_sample": float(bits_per_obs),
                    "source_nominal_SE": float(source_se) if np.isfinite(source_se) else np.nan,
                })
                print(f"MOCZ family SNR={snr:>4} seed={seed:>3} {method:38s} PER={rows[-1]['PER']:.5f}")

    mocz_ps = pd.DataFrame(rows)
    mocz_ps.to_csv(mocz_ps_path, index=False)

mocz_summary = summarize_per_seed(mocz_ps)
mocz_meta = mocz_ps.groupby("method", as_index=False)[
    ["payload_bits","observation_samples","bits_per_observation_sample","source_nominal_SE"]
].first()
mocz_summary = mocz_summary.merge(mocz_meta, on="method", how="left")
mocz_summary.to_csv(COMP_DIR / "mocz_family_comparison_summary.csv", index=False)
display(mocz_summary)

plt.figure(figsize=(7.0, 4.5))
for marker, (name, g) in zip(["o","s","^","D"], mocz_summary.groupby("method", sort=False)):
    g = g.sort_values("snr_db")
    plt.errorbar(g.snr_db, g.PER_mean, yerr=g.PER_95CI_halfwidth,
                 marker=marker, capsize=3, label=name)
plt.yscale("log")
plt.xlabel("SNR (dB)")
plt.ylabel("Packet error rate")
plt.title("Multipath short-packet comparison")
plt.grid(True, which="both", alpha=0.25)
plt.legend(fontsize=8)
plt.tight_layout()
save_current_figure("mocz_family_comparison")
plt.show()

eff_table = mocz_summary.groupby("method", as_index=False)[
    ["payload_bits","observation_samples","bits_per_observation_sample","source_nominal_SE"]
].first()
eff_table["note"] = [
    "10-sample released two-tap SQM diagnostic" if "SQM" in m else
    "BMOCZ source SE includes the same beta=0.3 roll-off convention used for the SE-MOCZ comparison" if m.startswith("BMOCZ") else
    "Source SE includes roll-off beta=0.3; observation vector includes matched-filter memory" if m.startswith("SE-") else
    "Matched N=8,K=6 adaptation; shorter native polynomial output"
    for m in eff_table["method"]
]
display(eff_table)
eff_table.to_csv(COMP_DIR / "mocz_family_efficiency_metadata.csv", index=False)


## V.5 — Unified five-scheme comparison on a common flat-fading benchmark

This is the requested **single-figure comparison of all five schemes**:

1. SQM,
2. hardware-aware Grassmannian,
3. Cube-Split,
4. SE-MOCZ,
5. IM-MOCZ.

### Common operating model

To make one cross-scheme curve scientifically interpretable, this benchmark uses a deliberately simple common denominator:

- **8 information bits / 256 messages** for every scheme;
- **equal total discrete waveform energy \(E=8\)**, so \(E_b=1\);
- one-tap **Rayleigh block fading**, unknown at the receiver;
- additive noise parameterized directly by \(E_b/N_0\);
- **no CFO, I/Q imbalance or DC offset** in this particular combined benchmark;
- each scheme retains its own waveform construction and receiver logic;
- native waveform lengths are retained and reported in the metadata table.

For SE-MOCZ, scalar rescaling is applied after constructing the source FTN-shaped waveform so that every message has equal energy; scalar rescaling does not alter the MOCZ zeros. The source colored matched-filter noise model and Gaussian marginal-ML receiver are retained. IM-MOCZ retains the DiZeT/majority-vote receiver. Cube-Split and the hardware-aware Grassmannian scheme use noncoherent line ML. SQM uses its quotient decoder.

This plot is a **controlled common-channel benchmark**, not a replacement for the hardware-uncertainty and multipath comparisons above.


In [ ]:
# ---------- Unified five-scheme common flat-fading benchmark ----------

def normalize_rows_energy(X, energy=8.0):
    X = np.asarray(X, complex).copy()
    nrm = np.linalg.norm(X, axis=1, keepdims=True)
    return X * (np.sqrt(energy) / np.maximum(nrm, 1e-15))

UNIFIED_BITS = 8
UNIFIED_ENERGY = float(UNIFIED_BITS)  # Eb = 1 by construction.

# SQM: energy normalization does not change its quotient subspaces.
X_SQM_UNIFIED = normalize_rows_energy(X_inc, UNIFIED_ENERGY)

# Cube-Split: retain the native CS(4,1) four-sample waveform.
X_CUBE_UNIFIED = normalize_rows_energy(CUBE4, UNIFIED_ENERGY)

# Hardware-aware Grassmannian: retain the 16-sample single-stream adaptation.
X_HWI_UNIFIED = normalize_rows_energy(HWI16, UNIFIED_ENERGY)

# SE-MOCZ: construct the source FTN-shaped observation waveform and then
# scalar-normalize each message. The polynomial zeros are unchanged by scaling.
X_SE_RAW, _ = bmocz_codebook(8, R_SE, total_energy=1.0)
S_SE_RAW = np.stack([np.convolve(x, SE_G) for x in X_SE_RAW])
se_scale = np.sqrt(UNIFIED_ENERGY) / np.maximum(np.linalg.norm(S_SE_RAW, axis=1), 1e-15)
X_SE_UNIFIED = X_SE_RAW * se_scale[:, None]

# IM-MOCZ: scalar normalization likewise preserves all polynomial roots.
X_IM_UNIFIED, _ = im_mocz_codebook(8, 6, R_IM, total_energy=UNIFIED_ENERGY)
X_IM_UNIFIED = normalize_rows_energy(X_IM_UNIFIED, UNIFIED_ENERGY)

unified_ps_path = COMP_DIR / "five_scheme_common_flat_fading_per_seed.csv"

if unified_ps_path.exists() and not COMP_FORCE:
    unified_ps = pd.read_csv(unified_ps_path)
else:
    rows = []
    for ebn0_db in COMP_SNRS:
        N0 = 10.0**(-float(ebn0_db)/10.0)  # Eb=1.

        # Source-model SE-MOCZ bank for a one-tap Rayleigh channel.
        se_bank = build_gaussian_marginal_bank(
            X_SE_UNIFIED,
            SE_G,
            np.array([1.0]),
            N0,
            tau_noise=SE_TAU,
            beta=SE_BETA,
        )

        for seed in COMP_SEEDS:
            rng = np.random.default_rng(20265000 + int(ebn0_db)*1000 + seed)
            B = COMP_UNIFIED_BLOCKS
            idx = rng.integers(256, size=B)
            h = (rng.normal(size=B) + 1j*rng.normal(size=B))/np.sqrt(2)

            # SQM: same unknown flat scalar fading, quotient decoder.
            n_sqm = np.sqrt(N0/2) * (
                rng.normal(size=(B, X_SQM_UNIFIED.shape[1])) +
                1j*rng.normal(size=(B, X_SQM_UNIFIED.shape[1]))
            )
            y_sqm = h[:,None] * X_SQM_UNIFIED[idx] + n_sqm
            p_sqm = decode_sqm_flat(y_sqm, SQM_FLAT0_BANK)

            # HWI-aware Grassmannian codebook evaluated here in the common clean
            # flat-fading regime, using its noncoherent line detector.
            n_hwi = np.sqrt(N0/2) * (
                rng.normal(size=(B, X_HWI_UNIFIED.shape[1])) +
                1j*rng.normal(size=(B, X_HWI_UNIFIED.shape[1]))
            )
            y_hwi = h[:,None] * X_HWI_UNIFIED[idx] + n_hwi
            p_hwi = cube_ml_decode(y_hwi, X_HWI_UNIFIED)

            # Native CS(4,1) Cube-Split.
            n_cube = np.sqrt(N0/2) * (
                rng.normal(size=(B, X_CUBE_UNIFIED.shape[1])) +
                1j*rng.normal(size=(B, X_CUBE_UNIFIED.shape[1]))
            )
            y_cube = h[:,None] * X_CUBE_UNIFIED[idx] + n_cube
            p_cube = cube_ml_decode(y_cube, X_CUBE_UNIFIED)

            # SE-MOCZ: source FTN pulse/noise model with one-tap fading.
            h1 = h[:,None]
            y_se = simulate_from_marginal_bank(idx, h1, se_bank, N0, rng)
            p_se = decode_gaussian_marginal(y_se, se_bank)

            # IM-MOCZ: native polynomial output under one-tap fading, with
            # source DiZeT/majority-vote receiver.
            y_im = h[:,None] * X_IM_UNIFIED[idx]
            n_im = np.sqrt(N0/2) * (
                rng.normal(size=y_im.shape) + 1j*rng.normal(size=y_im.shape)
            )
            p_im = im_dizet_decode(y_im + n_im, 8, 6, R_IM)

            methods = [
                ("SQM", p_sqm, X_SQM_UNIFIED.shape[1], "quotient decoder"),
                ("HWI-aware Grassmannian", p_hwi, X_HWI_UNIFIED.shape[1], "noncoherent line ML"),
                ("Cube-Split", p_cube, X_CUBE_UNIFIED.shape[1], "noncoherent line ML"),
                ("SE-MOCZ", p_se, se_bank["U"].shape[1], "source-model Gaussian marginal ML"),
                ("IM-MOCZ", p_im, X_IM_UNIFIED.shape[1], "DiZeT + majority vote"),
            ]
            for method, pred, obs_len, receiver in methods:
                rows.append({
                    "ebn0_db": float(ebn0_db),
                    "snr_db": float(ebn0_db),  # compatibility with summary helper
                    "seed": int(seed),
                    "method": method,
                    "PER": float(np.mean(pred != idx)),
                    "blocks": int(B),
                    "payload_bits": UNIFIED_BITS,
                    "waveform_or_observation_samples": int(obs_len),
                    "bits_per_observation_sample": float(UNIFIED_BITS/obs_len),
                    "receiver": receiver,
                })
                print(
                    f"Unified Eb/N0={ebn0_db:>4} seed={seed:>3} "
                    f"{method:26s} PER={rows[-1]['PER']:.5f}"
                )

    unified_ps = pd.DataFrame(rows)
    unified_ps.to_csv(unified_ps_path, index=False)

unified_summary = summarize_per_seed(unified_ps)
unified_summary = unified_summary.rename(columns={"snr_db":"ebn0_db"})
unified_meta = unified_ps.groupby("method", as_index=False)[
    ["payload_bits", "waveform_or_observation_samples", "bits_per_observation_sample", "receiver"]
].first()
unified_summary = unified_summary.merge(unified_meta, on="method", how="left")
unified_summary.to_csv(COMP_DIR / "five_scheme_common_flat_fading_summary.csv", index=False)

display(unified_summary)

# Main requested figure: all five schemes on one set of axes.
plt.figure(figsize=(7.4, 4.8))
markers = ["o", "s", "^", "D", "v"]
order = ["SQM", "HWI-aware Grassmannian", "Cube-Split", "SE-MOCZ", "IM-MOCZ"]
for marker, name in zip(markers, order):
    g = unified_summary[unified_summary.method.eq(name)].sort_values("ebn0_db")
    plt.errorbar(
        g.ebn0_db, g.PER_mean, yerr=g.PER_95CI_halfwidth,
        marker=marker, capsize=3, label=name
    )
plt.yscale("log")
plt.xlabel(r"$E_b/N_0$ (dB)")
plt.ylabel("Packet error rate")
plt.title("Common flat-fading benchmark: SQM and four noncoherent alternatives")
plt.grid(True, which="both", alpha=0.25)
plt.legend(fontsize=8)
plt.tight_layout()
save_current_figure("five_scheme_common_flat_fading_per")
plt.show()

# Companion snapshot at 10 dB (or nearest available point).
target = min(COMP_SNRS, key=lambda x: abs(float(x)-10.0))
bar = unified_summary[np.isclose(unified_summary.ebn0_db, float(target))].copy()
bar["method"] = pd.Categorical(bar["method"], categories=order, ordered=True)
bar = bar.sort_values("method")

plt.figure(figsize=(7.4, 4.5))
plt.bar(bar["method"].astype(str), bar["PER_mean"], yerr=bar["PER_95CI_halfwidth"], capsize=4)
plt.ylabel("Packet error rate")
plt.xlabel("Scheme")
plt.title(f"Five-scheme comparison at {float(target):g} dB $E_b/N_0$")
plt.xticks(rotation=20, ha="right")
plt.grid(True, axis="y", alpha=0.25)
plt.tight_layout()
save_current_figure("five_scheme_common_flat_fading_snapshot")
plt.show()

unified_metadata = unified_meta.copy()
unified_metadata["benchmark"] = (
    "8-bit payload; Eb=1; one-tap Rayleigh block fading; no CFO/IQ/DC; "
    "native waveform lengths and receiver structures retained"
)
unified_metadata.to_csv(COMP_DIR / "five_scheme_common_flat_fading_metadata.csv", index=False)
display(unified_metadata)


## V.6 — Decisive matched multipath + hardware-uncertainty test: SQM vs HWI-Grassmannian

This is the decisive experiment motivated by the earlier comparison: **HWI-aware Grassmannian signaling was better than SQM in the flat-fading hardware-impairment benchmark**, so the important question is whether that advantage survives once both schemes are placed in SQM's native difficult regime.

### Primary matched environment

The main experiment uses the **same physical nuisance realization for both codebooks**:

- 8 information bits / 256 messages;
- 16 complex transmitted samples;
- equal codeword energy \(E=16\);
- the released fractional **TDL-A multipath** model;
- physical CFO uniformly distributed over \([-4^\circ,4^\circ]\) per sample;
- the same static invertible real \(2\times2\) I/Q mixing model with condition number 1.5;
- the same random DC-offset model;
- the same AWGN realization, SNR grid, independent seeds, and packet count.

### Two HWI-Grassmannian receivers are tested

1. **Centered CFO-search line ML** — the closest extension of the HWI-Grassmannian line detector. Centering removes DC and a CFO grid is searched, but multipath is not explicitly modeled.

2. **HWI codebook + matched selective-quotient receiver** — a deliberately generous stress-test adaptation. The HWI-designed codebook is retained, but it is given the same TDL-A explanation machinery used by SQM. Its retained rank is selected using **held-out seeds that are never used for evaluation**.

The second comparator is important scientifically: if SQM only beats the first receiver, the result could simply mean that ordinary line ML is mismatched to multipath. If SQM also beats the second receiver, the evidence for the end-to-end SQM design is much stronger.

### Decision rule

- If both HWI variants beat SQM, SQM's **performance novelty is weak** in this regime.
- If SQM beats native HWI line ML but not HWI + matched selective quotient, the main value is likely the **selective-quotient receiver**, not the SQM waveform/codebook alone.
- If SQM beats both, there is strong evidence that the **end-to-end SQM design adds value under combined multipath and hardware uncertainty**.

Every HWI result in this section remains an **independent single-stream adaptation**, not an exact reproduction of the source paper's numerical curves.


In [ ]:
# ---------- Decisive matched TDL-A + hardware-uncertainty comparison ----------

DECISIVE_DIR = COMP_DIR / "decisive_hwi_vs_sqm"
DECISIVE_DIR.mkdir(parents=True, exist_ok=True)

if COMP_MODE == "manuscript":
    DECISIVE_BLOCKS = 800
    DECISIVE_TUNE_BLOCKS = 220
    DECISIVE_TWO_TAP_BLOCKS = 700
else:
    DECISIVE_BLOCKS = 80
    DECISIVE_TUNE_BLOCKS = 40
    DECISIVE_TWO_TAP_BLOCKS = 70

DECISIVE_RANKS = [2, 3, 4, 5, 6, 8]
DECISIVE_TUNE_SEEDS = [271, 863]  # held out from COMP_SEEDS
DECISIVE_TUNE_SNR_DB = 10

def save_decisive_figure(stem):
    png = DECISIVE_DIR / f"{stem}.png"
    pdf = DECISIVE_DIR / f"{stem}.pdf"
    plt.savefig(png, dpi=220, bbox_inches="tight")
    plt.savefig(pdf, bbox_inches="tight")
    print("Saved:", png.relative_to(ROOT))
    print("Saved:", pdf.relative_to(ROOT))

# Fairness audit.
fairness_audit = pd.DataFrame([
    ["payload bits", 8, 8],
    ["messages", 256, 256],
    ["complex samples", 16, 16],
    ["mean codeword energy", float(np.mean(np.sum(np.abs(X_inc)**2, axis=1))),
     float(np.mean(np.sum(np.abs(HWI16)**2, axis=1)))],
    ["multipath", "fractional TDL-A", "fractional TDL-A"],
    ["CFO support", "[-4,4] deg/sample", "[-4,4] deg/sample"],
    ["I/Q model", "static real 2x2, cond=1.5", "static real 2x2, cond=1.5"],
    ["DC model", "same random draw", "same random draw"],
    ["AWGN", "same random draw", "same random draw"],
], columns=["quantity", "SQM", "HWI-Grassmannian"])
display(fairness_audit)
fairness_audit.to_csv(DECISIVE_DIR / "fairness_audit.csv", index=False)

def build_centered_cfo_line_bank(C, grid_deg=GRID5_DEG):
    """Centered unit-norm line templates for a CFO-search Grassmannian receiver."""
    C = np.asarray(C, complex)
    grid_deg = np.asarray(grid_deg, float)
    t = np.arange(C.shape[1])
    B = np.empty((len(C), len(grid_deg), C.shape[1]), complex)
    for m, x in enumerate(C):
        for j, deg in enumerate(grid_deg):
            v = x * np.exp(1j*np.deg2rad(deg)*t)
            v = v - v.mean()
            nv = np.linalg.norm(v)
            B[m, j] = v / max(nv, 1e-15)
    return B

def decode_centered_cfo_line(Y_real, bank, batch=64):
    """Closest native HWI extension: center, search CFO, then noncoherent line ML."""
    yc = Y_real[:, 0] + 1j*Y_real[:, 1]
    yc = yc - yc.mean(axis=1, keepdims=True)
    pred = np.empty(len(yc), int)
    for s in range(0, len(yc), batch):
        e = min(len(yc), s + batch)
        z = np.einsum("bl,mwl->bmw", yc[s:e], bank.conj(), optimize=True)
        score = np.abs(z)**2
        pred[s:e] = np.argmax(score.max(axis=2), axis=1)
    return pred

HWI_NATIVE_CFO_BANK = build_centered_cfo_line_bank(HWI16, GRID5_DEG)

# Generous HWI comparator: HWI codebook, but with the same TDL-A selective
# explanation machinery as SQM. This is explicitly a stress-test adaptation,
# not the source-paper HWI receiver.
hwi_tdla_bank_path = DECISIVE_DIR / "hwi_codebook_tdla_selective_bank_r8.npz"
if hwi_tdla_bank_path.exists() and not COMP_FORCE:
    HWI_TDLA_BANK = np.load(hwi_tdla_bank_path)["bank"]
    print("Loaded cached HWI TDL-A selective bank.")
else:
    print("Building HWI-codebook TDL-A selective basis bank ...")
    HWI_TDLA_BANK = build_tdla_bank_for_codebook(HWI16, max(DECISIVE_RANKS))
    np.savez_compressed(hwi_tdla_bank_path, bank=HWI_TDLA_BANK)

# Held-out rank selection for the generous HWI multipath-aware receiver.
rank_tune_path = DECISIVE_DIR / "hwi_tdla_rank_tuning_heldout.csv"
if rank_tune_path.exists() and not COMP_FORCE:
    hwi_rank_tune = pd.read_csv(rank_tune_path)
else:
    tune_rows = []
    for seed in DECISIVE_TUNE_SEEDS:
        nuisance = draw_common_tdla_nuisance(
            DECISIVE_TUNE_SNR_DB,
            DECISIVE_TUNE_BLOCKS,
            20267100 + seed,
        )
        idx = nuisance[0]
        Yhwi = apply_tdla_nuisance(HWI16, nuisance)
        for rank in DECISIVE_RANKS:
            pred = decode_tdla_bank(Yhwi, HWI_TDLA_BANK, rank)
            tune_rows.append({
                "snr_db": DECISIVE_TUNE_SNR_DB,
                "seed": seed,
                "rank": rank,
                "PER": float(np.mean(pred != idx)),
                "blocks": DECISIVE_TUNE_BLOCKS,
            })
    hwi_rank_tune = pd.DataFrame(tune_rows)
    hwi_rank_tune.to_csv(rank_tune_path, index=False)

hwi_rank_tune_summary = (
    hwi_rank_tune.groupby("rank", as_index=False)
    .agg(PER_mean=("PER", "mean"), seeds=("seed", "nunique"), blocks=("blocks", "sum"))
    .sort_values(["PER_mean", "rank"])
)
HWI_TDLA_SELECTED_RANK = int(hwi_rank_tune_summary.iloc[0]["rank"])
hwi_rank_tune_summary.to_csv(
    DECISIVE_DIR / "hwi_tdla_rank_tuning_summary.csv", index=False
)
print("Held-out HWI selective rank:", HWI_TDLA_SELECTED_RANK)
display(hwi_rank_tune_summary)

# Main decisive full TDL-A + CFO + static I/Q + DC experiment.
decisive_ps_path = DECISIVE_DIR / "tdla_hwi_dc_matched_per_seed.csv"
if decisive_ps_path.exists() and not COMP_FORCE:
    decisive_ps = pd.read_csv(decisive_ps_path)
else:
    rows = []
    for snr in COMP_SNRS:
        for seed in COMP_SEEDS:
            nuisance = draw_common_tdla_nuisance(
                snr, DECISIVE_BLOCKS, 20267200 + 1000*int(snr) + seed
            )
            idx = nuisance[0]

            # Same nuisance and AWGN realization for both codebooks.
            Ysqm = apply_tdla_nuisance(X_inc, nuisance)
            Yhwi = apply_tdla_nuisance(HWI16, nuisance)

            p_sqm = decode_tdla_bank(Ysqm, bank_inc, R_SEL)
            p_hwi_native = decode_centered_cfo_line(Yhwi, HWI_NATIVE_CFO_BANK)
            p_hwi_sq = decode_tdla_bank(
                Yhwi, HWI_TDLA_BANK, HWI_TDLA_SELECTED_RANK
            )

            for method, pred, receiver in [
                ("SQM released codebook + selective quotient r=4", p_sqm,
                 "released SQM selective quotient"),
                ("HWI-Grassmannian + centered CFO-search line ML", p_hwi_native,
                 "centered line ML with CFO search; no multipath model"),
                (f"HWI-Grassmannian + matched selective quotient r={HWI_TDLA_SELECTED_RANK}", p_hwi_sq,
                 "HWI codebook with held-out-tuned TDL-A selective quotient"),
            ]:
                rows.append({
                    "snr_db": snr,
                    "seed": seed,
                    "method": method,
                    "PER": float(np.mean(pred != idx)),
                    "blocks": DECISIVE_BLOCKS,
                    "payload_bits": 8,
                    "blocklength": 16,
                    "multipath": "released fractional TDL-A",
                    "cfo_support_deg_per_sample": "[-4,4]",
                    "iq_model": "static invertible real 2x2, condition number 1.5",
                    "dc_model": "unit-magnitude random direction",
                    "receiver": receiver,
                })
                print(
                    f"Decisive TDL-A SNR={snr:>4} seed={seed:>3} "
                    f"{method:58s} PER={rows[-1]['PER']:.5f}"
                )
    decisive_ps = pd.DataFrame(rows)
    decisive_ps.to_csv(decisive_ps_path, index=False)

decisive_summary = summarize_per_seed(decisive_ps)
decisive_summary.to_csv(
    DECISIVE_DIR / "tdla_hwi_dc_matched_summary.csv", index=False
)
display(decisive_summary)

plt.figure(figsize=(7.7, 4.8))
method_order = [
    "SQM released codebook + selective quotient r=4",
    "HWI-Grassmannian + centered CFO-search line ML",
    f"HWI-Grassmannian + matched selective quotient r={HWI_TDLA_SELECTED_RANK}",
]
for marker, name in zip(["o", "s", "^"], method_order):
    g = decisive_summary[decisive_summary.method.eq(name)].sort_values("snr_db")
    plt.errorbar(
        g.snr_db, g.PER_mean, yerr=g.PER_95CI_halfwidth,
        marker=marker, capsize=3, label=name
    )
plt.yscale("log")
plt.xlabel("SNR (dB)")
plt.ylabel("Packet error rate")
plt.title("Decisive matched test: TDL-A + CFO + I/Q + DC")
plt.grid(True, which="both", alpha=0.25)
plt.legend(fontsize=7.5)
plt.tight_layout()
save_decisive_figure("tdla_hwi_dc_matched_comparison")
plt.show()

# Paired deltas: positive means SQM has lower PER.
delta_rows = []
for snr, g in decisive_ps.groupby("snr_db"):
    p = g.pivot(index="seed", columns="method", values="PER")
    sqm_name = method_order[0]
    for competitor in method_order[1:]:
        d = p[competitor] - p[sqm_name]
        delta_rows.append({
            "snr_db": snr,
            "competitor": competitor,
            "mean_delta_PER_competitor_minus_SQM": float(d.mean()),
            "paired_95CI_halfwidth": float(t_halfwidth(d.to_numpy())),
            "seeds": len(d),
        })

decisive_deltas = pd.DataFrame(delta_rows)
decisive_deltas.to_csv(
    DECISIVE_DIR / "tdla_hwi_dc_paired_deltas.csv", index=False
)
display(decisive_deltas)


### V.6.2 — Secondary two-tap + CFO + I/Q + DC check

To separate the effect of a very rich TDL-A model from the more basic question of **multipath itself**, the notebook also repeats the comparison under a common two-tap Rayleigh channel with PDP \([0.8,0.2]\).

Both SQM and HWI-Grassmannian use:

- 8 bits / 256 messages;
- 16 complex samples and equal energy;
- the same two channel taps;
- the same CFO, I/Q transform, DC offset, AWGN, SNR grid, seeds, and packet draws.

The HWI codebook is again evaluated both with centered CFO-search line ML and with a matched two-tap selective-quotient receiver.


In [ ]:
# ---------- Secondary matched two-tap + CFO + I/Q + DC check ----------

def draw_common_two_tap_hwi_nuisance(snr_db, blocks, seed):
    rng = np.random.default_rng(seed)
    idx = rng.integers(256, size=blocks)
    z = (
        rng.normal(size=(blocks,2)) + 1j*rng.normal(size=(blocks,2))
    ) / np.sqrt(2)
    h = z * np.sqrt(np.array([0.8, 0.2]))[None,:]
    omega = rng.uniform(-np.deg2rad(4.0), np.deg2rad(4.0), blocks)
    A = comp_random_A(rng, blocks, 1.5)
    dcang = rng.uniform(0, 2*np.pi, blocks)
    d = np.stack([np.cos(dcang), np.sin(dcang)], axis=1)
    N0 = 10.0**(-snr_db/10.0)
    noise = np.sqrt(N0/2.0) * rng.normal(size=(blocks, 2, 16))
    return idx, h, omega, A, d, noise

def apply_two_tap_hwi_nuisance(C, nuisance):
    idx, h, omega, A, d, noise = nuisance
    x = C[idx]
    s = np.empty_like(x, dtype=complex)
    s[:,0] = h[:,0] * x[:,0]
    s[:,1:] = h[:,0,None]*x[:,1:] + h[:,1,None]*x[:,:-1]
    s *= np.exp(1j*omega[:,None]*np.arange(16)[None,:])
    sr = np.stack([s.real, s.imag], axis=1)
    return (
        np.einsum("nij,njk->nik", A, sr, optimize=True)
        + d[:,:,None] + noise
    )

def build_two_tap_selective_bank(C, rmax=4):
    C = np.asarray(C, complex)
    bank = np.zeros((256, len(GRID5_RAD), 16, rmax), np.float32)
    t = np.arange(16)
    H = np.eye(16) - np.ones((16,16))/16
    for m, x in enumerate(C):
        for j, omega in enumerate(GRID5_RAD):
            rot = np.exp(1j*omega*t)
            v0 = x * rot
            v1 = np.zeros(16, complex)
            v1[1:] = x[:-1] * rot[1:]
            F = np.stack([v0.real, v0.imag, v1.real, v1.imag]) @ H
            _, _, Vh = np.linalg.svd(F, full_matrices=False)
            rr = min(rmax, Vh.shape[0])
            bank[m,j,:,:rr] = Vh[:rr].T.astype(np.float32)
    return bank

two_tap_sqm_bank_path = DECISIVE_DIR / "two_tap_sqm_bank_r4.npz"
two_tap_hwi_bank_path = DECISIVE_DIR / "two_tap_hwi_bank_r4.npz"

if (
    two_tap_sqm_bank_path.exists()
    and two_tap_hwi_bank_path.exists()
    and not COMP_FORCE
):
    TWO_TAP_SQM_BANK = np.load(two_tap_sqm_bank_path)["bank"]
    TWO_TAP_HWI_BANK = np.load(two_tap_hwi_bank_path)["bank"]
else:
    print("Building matched 16-sample two-tap selective banks ...")
    TWO_TAP_SQM_BANK = build_two_tap_selective_bank(X_inc, 4)
    TWO_TAP_HWI_BANK = build_two_tap_selective_bank(HWI16, 4)
    np.savez_compressed(two_tap_sqm_bank_path, bank=TWO_TAP_SQM_BANK)
    np.savez_compressed(two_tap_hwi_bank_path, bank=TWO_TAP_HWI_BANK)

two_tap_ps_path = DECISIVE_DIR / "two_tap_hwi_dc_matched_per_seed.csv"

if two_tap_ps_path.exists() and not COMP_FORCE:
    two_tap_ps = pd.read_csv(two_tap_ps_path)
else:
    rows = []
    for snr in COMP_SNRS:
        for seed in COMP_SEEDS:
            nuisance = draw_common_two_tap_hwi_nuisance(
                snr,
                DECISIVE_TWO_TAP_BLOCKS,
                20267300 + 1000*int(snr) + seed,
            )
            idx = nuisance[0]
            Ysqm = apply_two_tap_hwi_nuisance(X_inc, nuisance)
            Yhwi = apply_two_tap_hwi_nuisance(HWI16, nuisance)

            p_sqm = decode_tdla_bank(Ysqm, TWO_TAP_SQM_BANK, 4)
            p_hwi_native = decode_centered_cfo_line(
                Yhwi, HWI_NATIVE_CFO_BANK
            )
            p_hwi_sq = decode_tdla_bank(
                Yhwi, TWO_TAP_HWI_BANK, 4
            )

            for method, pred in [
                ("SQM selective quotient r=4", p_sqm),
                ("HWI-Grassmannian centered CFO-search line ML", p_hwi_native),
                ("HWI-Grassmannian + matched two-tap quotient r=4", p_hwi_sq),
            ]:
                rows.append({
                    "snr_db": snr,
                    "seed": seed,
                    "method": method,
                    "PER": float(np.mean(pred != idx)),
                    "blocks": DECISIVE_TWO_TAP_BLOCKS,
                })

    two_tap_ps = pd.DataFrame(rows)
    two_tap_ps.to_csv(two_tap_ps_path, index=False)

two_tap_summary = summarize_per_seed(two_tap_ps)
two_tap_summary.to_csv(
    DECISIVE_DIR / "two_tap_hwi_dc_matched_summary.csv", index=False
)
display(two_tap_summary)

plt.figure(figsize=(7.7, 4.8))
for marker, (name, g) in zip(
    ["o", "s", "^"], two_tap_summary.groupby("method", sort=False)
):
    g = g.sort_values("snr_db")
    plt.errorbar(
        g.snr_db, g.PER_mean, yerr=g.PER_95CI_halfwidth,
        marker=marker, capsize=3, label=name
    )
plt.yscale("log")
plt.xlabel("SNR (dB)")
plt.ylabel("Packet error rate")
plt.title("Matched two-tap + CFO + I/Q + DC stress test")
plt.grid(True, which="both", alpha=0.25)
plt.legend(fontsize=7.5)
plt.tight_layout()
save_decisive_figure("two_tap_hwi_dc_matched_comparison")
plt.show()


### V.6.3 — Machine-readable decision table

The table below does **not** hard-code a preferred outcome. It records the paired PER difference

\[
\Delta_{\mathrm{HWI-SQM}} =
\mathrm{PER}_{\mathrm{HWI}}-\mathrm{PER}_{\mathrm{SQM}},
\]

so that a **positive** value means SQM has the lower PER under the fully matched TDL-A + hardware-uncertainty experiment.


In [ ]:
# ---------- Machine-readable decision table ----------
decision_rows = []
for _, row in decisive_deltas.iterrows():
    decision_rows.append({
        "environment": "TDL-A + CFO + I/Q + DC",
        "snr_db": row["snr_db"],
        "competitor": row["competitor"],
        "SQM_lower_PER_if_positive":
            row["mean_delta_PER_competitor_minus_SQM"],
        "paired_95CI_halfwidth": row["paired_95CI_halfwidth"],
    })

decision_table = pd.DataFrame(decision_rows)
decision_table.to_csv(
    DECISIVE_DIR / "decisive_interpretation_table.csv", index=False
)
display(decision_table)

print("\nInterpretation rule:")
print("- If HWI line ML and HWI+matched-quotient both beat SQM, SQM's performance novelty is weak in this regime.")
print("- If SQM beats HWI line ML but not HWI+matched-quotient, the selective-quotient receiver is the main value rather than the waveform alone.")
print("- If SQM beats both, the end-to-end SQM design has strong evidence of added value under combined multipath/hardware uncertainty.")


# Part VI — Complete 1+4 comparison matrix

The earlier sections answer source-specific questions. This section answers the presentation question directly: **how does SQM compare with all four selected alternatives on the same figures?**

The five curves are always:

1. **SQM**
2. **HWI-Grassmannian**
3. **Cube-Split**
4. **SE-MOCZ**
5. **IM-MOCZ**

## Four operating conditions

The comparison matrix progresses from easy to hard:

- **A — clean flat Rayleigh fading**: common 8-bit payload and equal energy per bit; native waveform lengths retained.
- **B — flat fading + CFO + I/Q + DC**: SQM remains calibration-light; HWI/Cube use centered CFO-search line detection; SE-MOCZ and IM-MOCZ are given an **oracle hardware-cleaned upper reference** because their native receivers do not model the SQM affine I/Q/DC nuisance.
- **C — clean two-tap Rayleigh multipath**: SQM, HWI and Cube use matched rank-4 two-tap selective-quotient decoding; SE-MOCZ and IM-MOCZ keep their multipath-native receivers.
- **D — two-tap multipath + CFO + I/Q + DC**: SQM, HWI and Cube receive the same physical nuisance model and use matched selective-quotient decoding. SE-MOCZ and IM-MOCZ again receive an **oracle front-end-cleaned upper reference** before their native multipath receivers. This is intentionally conservative against SQM.

The purpose is not to claim that the original papers used identical models. It is to provide a transparent **1+4 stress-test matrix** while preserving a separate metadata table that records where each competitor is source-native, adapted, or oracle-aided.

The fully matched TDL-A SQM-vs-HWI experiment from Part V remains the stronger pairwise test for the closest competitor; this section complements it by putting **all five schemes on common plots**.


In [ ]:
# ---------- V8 complete 1+4 comparison matrix: configuration and helpers ----------

V8_DIR = COMP_DIR / "complete_1plus4_matrix"
V8_DIR.mkdir(parents=True, exist_ok=True)

V8_SCALE = os.environ.get("SQM_V8_SCALE", "manuscript").strip().lower()
V8_FORCE = False

if V8_SCALE == "publication":
    # High-confidence final run: 10 independent seeds x 1000 packets/SNR.
    V8_SEEDS = [13, 59, 151, 233, 347, 431, 587, 719, 863, 997]
    V8_BLOCKS = 1000
elif V8_SCALE == "manuscript":
    # Practical manuscript run with more independent seeds than V7.
    V8_SEEDS = [13, 59, 151, 233, 347]
    V8_BLOCKS = 600
else:
    V8_SEEDS = [13]
    V8_BLOCKS = 60

V8_SNRS = list(COMP_SNRS)
V8_METHODS = ["SQM", "HWI-Grassmannian", "Cube-Split", "SE-MOCZ", "IM-MOCZ"]
V8_MARKERS = ["o", "s", "^", "D", "v"]
V8_BITS = 8
V8_ENERGY = float(V8_BITS)  # Eb = 1.
V8_CFO_MAX_DEG = 4.0

print("V8 scale:", V8_SCALE)
print("V8 seeds:", V8_SEEDS)
print("V8 packets per seed/SNR:", V8_BLOCKS)

# Equal-energy waveform representatives. Native lengths are retained for MOCZ.
V8_X_SQM = normalize_rows_energy(X_inc, V8_ENERGY)
V8_X_HWI = normalize_rows_energy(HWI16, V8_ENERGY)
V8_X_CUBE = normalize_rows_energy(CUBE16, V8_ENERGY)

# Reuse held-out MOCZ radii from the earlier validation.
V8_X_SE_POLY, _ = bmocz_codebook(8, R_SE, total_energy=1.0)
V8_SE_SHAPED = np.stack([np.convolve(x, SE_G) for x in V8_X_SE_POLY])
V8_SE_SCALE = np.sqrt(V8_ENERGY) / np.maximum(np.linalg.norm(V8_SE_SHAPED, axis=1), 1e-15)
V8_X_SE = V8_X_SE_POLY * V8_SE_SCALE[:, None]
V8_X_IM, _ = im_mocz_codebook(8, 6, R_IM, total_energy=V8_ENERGY)
V8_X_IM = normalize_rows_energy(V8_X_IM, V8_ENERGY)

# CFO-search banks for the flat HWI panel.
V8_CFO_GRID_DEG = GRID5_DEG.copy()
V8_SQM_FLAT_HWI_BANK = flat_sqm_basis_bank(V8_X_SQM, V8_CFO_GRID_DEG)
V8_HWI_LINE_BANK = build_centered_cfo_line_bank(V8_X_HWI, V8_CFO_GRID_DEG)
V8_CUBE_LINE_BANK = build_centered_cfo_line_bank(V8_X_CUBE, V8_CFO_GRID_DEG)

# Matched 16-sample two-tap rank-4 banks for SQM, HWI and Cube.
def v8_load_or_build_two_tap_bank(name, C):
    p = V8_DIR / f"two_tap_bank_{name}_r4.npz"
    if p.exists() and not V8_FORCE:
        return np.load(p)["bank"]
    b = build_two_tap_selective_bank(C, 4)
    np.savez_compressed(p, bank=b)
    return b

V8_SQM_2T_BANK = v8_load_or_build_two_tap_bank("sqm", V8_X_SQM)
V8_HWI_2T_BANK = v8_load_or_build_two_tap_bank("hwi", V8_X_HWI)
V8_CUBE_2T_BANK = v8_load_or_build_two_tap_bank("cube", V8_X_CUBE)


def v8_summary(df):
    return summarize_per_seed(df, group_cols=("panel", "snr_db", "method"))


def v8_draw_flat_common(snr_db, blocks, seed):
    rng = np.random.default_rng(seed)
    idx = rng.integers(256, size=blocks)
    h = (rng.normal(size=blocks) + 1j*rng.normal(size=blocks))/np.sqrt(2)
    N0 = 10.0**(-float(snr_db)/10.0)
    return rng, idx, h, N0


def v8_real_affine_observation(C, idx, s_complex, omega, A, d, N0, rng):
    """Apply CFO -> static real 2x2 I/Q mixing -> DC -> post-front-end AWGN."""
    L = s_complex.shape[1]
    z = s_complex * np.exp(1j*omega[:,None]*np.arange(L)[None,:])
    zr = np.stack([z.real, z.imag], axis=1)
    noise = np.sqrt(N0/2.0) * rng.normal(size=(len(idx), 2, L))
    return np.einsum("nij,njk->nik", A, zr, optimize=True) + d[:,:,None] + noise


def v8_two_tap_signal(C, idx, h2):
    x = C[idx]
    y = np.zeros_like(x, complex)
    y[:,0] = h2[:,0] * x[:,0]
    y[:,1:] = h2[:,0,None]*x[:,1:] + h2[:,1,None]*x[:,:-1]
    return y


def v8_complex_to_real(Y):
    return np.stack([Y.real, Y.imag], axis=1)


def v8_append(rows, panel, snr, seed, method, pred, idx, receiver, calibration, obs_len):
    rows.append({
        "panel": panel,
        "snr_db": float(snr),
        "seed": int(seed),
        "method": method,
        "PER": float(np.mean(pred != idx)),
        "blocks": int(len(idx)),
        "payload_bits": V8_BITS,
        "observation_samples": int(obs_len),
        "bits_per_observation_sample": float(V8_BITS/obs_len),
        "receiver": receiver,
        "calibration_assumption": calibration,
    })

comparison_rules = pd.DataFrame([
    ["A: flat clean", "SQM", "quotient decoder", "none"],
    ["A: flat clean", "HWI-Grassmannian", "noncoherent line ML", "none"],
    ["A: flat clean", "Cube-Split", "noncoherent line ML", "none"],
    ["A: flat clean", "SE-MOCZ", "source-model Gaussian marginal ML", "none"],
    ["A: flat clean", "IM-MOCZ", "DiZeT + majority vote", "none"],
    ["B: flat + HWI", "SQM", "centered quotient + CFO search", "none"],
    ["B: flat + HWI", "HWI-Grassmannian", "centered CFO-search line ML", "none"],
    ["B: flat + HWI", "Cube-Split", "centered CFO-search line ML", "none"],
    ["B: flat + HWI", "SE-MOCZ", "source-model Gaussian marginal ML", "oracle front-end cleanup"],
    ["B: flat + HWI", "IM-MOCZ", "DiZeT + majority vote", "oracle front-end cleanup"],
    ["C: two-tap clean", "SQM", "matched rank-4 two-tap quotient", "none"],
    ["C: two-tap clean", "HWI-Grassmannian", "matched rank-4 two-tap quotient", "none"],
    ["C: two-tap clean", "Cube-Split", "matched rank-4 two-tap quotient", "none"],
    ["C: two-tap clean", "SE-MOCZ", "two-tap Gaussian marginal ML", "none"],
    ["C: two-tap clean", "IM-MOCZ", "two-tap DiZeT + majority vote", "none"],
    ["D: two-tap + HWI", "SQM", "matched rank-4 two-tap quotient + CFO search", "none"],
    ["D: two-tap + HWI", "HWI-Grassmannian", "matched rank-4 two-tap quotient + CFO search", "none"],
    ["D: two-tap + HWI", "Cube-Split", "matched rank-4 two-tap quotient + CFO search", "none"],
    ["D: two-tap + HWI", "SE-MOCZ", "two-tap Gaussian marginal ML", "oracle front-end cleanup"],
    ["D: two-tap + HWI", "IM-MOCZ", "two-tap DiZeT + majority vote", "oracle front-end cleanup"],
], columns=["panel", "scheme", "receiver", "hardware_calibration"])
comparison_rules.to_csv(V8_DIR / "comparison_rules.csv", index=False)
display(comparison_rules)


## VI.1 — Run the complete five-scheme matrix

Two design choices make this comparison conservative rather than promotional:

- In multipath, **HWI-Grassmannian and Cube-Split are given the same matched rank-4 quotient machinery** instead of being left with a flat-fading line detector that is obviously mismatched.
- In hardware-uncertainty panels, **SE-MOCZ and IM-MOCZ are given oracle front-end cleanup** before their native receivers. Their plotted PER is therefore an upper-reference for what those waveforms could achieve with perfect hardware calibration; SQM receives no such oracle information.

Thus, the combined-stress plot should not be interpreted as an exact reproduction of the source papers. It is a deliberately documented, competitor-favorable stress test.


In [ ]:
# ---------- Run / load the complete 1+4 matrix ----------

v8_ps_path = V8_DIR / "complete_1plus4_per_seed.csv"

if v8_ps_path.exists() and not V8_FORCE:
    v8_ps = pd.read_csv(v8_ps_path)
    print("Loaded cached V8 complete 1+4 results.")
else:
    rows = []

    for snr in V8_SNRS:
        N0 = 10.0**(-float(snr)/10.0)

        # MOCZ receiver banks for the current Eb/N0.
        se_flat_bank = build_gaussian_marginal_bank(
            V8_X_SE, SE_G, np.array([1.0]), N0,
            tau_noise=SE_TAU, beta=SE_BETA,
        )
        se_two_bank = build_gaussian_marginal_bank(
            V8_X_SE, SE_G, PDP2, N0,
            tau_noise=SE_TAU, beta=SE_BETA,
        )

        for seed in V8_SEEDS:
            B = V8_BLOCKS

            # ================================================================
            # A — clean flat Rayleigh fading
            # ================================================================
            rng, idx, h, N0 = v8_draw_flat_common(
                snr, B, 20268000 + int(snr)*1000 + seed
            )

            def cnoise(shape):
                return np.sqrt(N0/2)*(rng.normal(size=shape)+1j*rng.normal(size=shape))

            y = h[:,None]*V8_X_SQM[idx] + cnoise((B, V8_X_SQM.shape[1]))
            p = decode_sqm_flat(y, SQM_FLAT0_BANK)
            v8_append(rows, "A: flat clean", snr, seed, "SQM", p, idx,
                      "quotient decoder", "none", V8_X_SQM.shape[1])

            y = h[:,None]*V8_X_HWI[idx] + cnoise((B, V8_X_HWI.shape[1]))
            p = cube_ml_decode(y, V8_X_HWI)
            v8_append(rows, "A: flat clean", snr, seed, "HWI-Grassmannian", p, idx,
                      "noncoherent line ML", "none", V8_X_HWI.shape[1])

            y = h[:,None]*V8_X_CUBE[idx] + cnoise((B, V8_X_CUBE.shape[1]))
            p = cube_ml_decode(y, V8_X_CUBE)
            v8_append(rows, "A: flat clean", snr, seed, "Cube-Split", p, idx,
                      "noncoherent line ML", "none", V8_X_CUBE.shape[1])

            y = simulate_from_marginal_bank(idx, h[:,None], se_flat_bank, N0, rng)
            p = decode_gaussian_marginal(y, se_flat_bank)
            v8_append(rows, "A: flat clean", snr, seed, "SE-MOCZ", p, idx,
                      "source-model Gaussian marginal ML", "none", y.shape[1])

            y = h[:,None]*V8_X_IM[idx] + cnoise((B, V8_X_IM.shape[1]))
            p = im_dizet_decode(y, 8, 6, R_IM)
            v8_append(rows, "A: flat clean", snr, seed, "IM-MOCZ", p, idx,
                      "DiZeT + majority vote", "none", y.shape[1])

            # ================================================================
            # B — flat fading + CFO + I/Q + DC
            # ================================================================
            rng = np.random.default_rng(20268100 + int(snr)*1000 + seed)
            idx = rng.integers(256, size=B)
            h = (rng.normal(size=B)+1j*rng.normal(size=B))/np.sqrt(2)
            omega = rng.uniform(-np.deg2rad(V8_CFO_MAX_DEG), np.deg2rad(V8_CFO_MAX_DEG), B)
            A = comp_random_A(rng, B, 1.5)
            dcang = rng.uniform(0, 2*np.pi, B)
            d = np.stack([np.cos(dcang), np.sin(dcang)], axis=1)

            s = h[:,None]*V8_X_SQM[idx]
            Y = v8_real_affine_observation(V8_X_SQM, idx, s, omega, A, d, N0, rng)
            p = decode_sqm_flat(Y[:,0] + 1j*Y[:,1], V8_SQM_FLAT_HWI_BANK)
            v8_append(rows, "B: flat + HWI", snr, seed, "SQM", p, idx,
                      "centered quotient + CFO search", "none", V8_X_SQM.shape[1])

            s = h[:,None]*V8_X_HWI[idx]
            Y = v8_real_affine_observation(V8_X_HWI, idx, s, omega, A, d, N0, rng)
            p = decode_centered_cfo_line(Y, V8_HWI_LINE_BANK)
            v8_append(rows, "B: flat + HWI", snr, seed, "HWI-Grassmannian", p, idx,
                      "centered CFO-search line ML", "none", V8_X_HWI.shape[1])

            s = h[:,None]*V8_X_CUBE[idx]
            Y = v8_real_affine_observation(V8_X_CUBE, idx, s, omega, A, d, N0, rng)
            p = decode_centered_cfo_line(Y, V8_CUBE_LINE_BANK)
            v8_append(rows, "B: flat + HWI", snr, seed, "Cube-Split", p, idx,
                      "centered CFO-search line ML", "none", V8_X_CUBE.shape[1])

            # MOCZ family gets an intentionally generous oracle front-end cleanup:
            # same payload/fading/EbN0, but hardware nuisance is perfectly removed.
            y = simulate_from_marginal_bank(idx, h[:,None], se_flat_bank, N0, rng)
            p = decode_gaussian_marginal(y, se_flat_bank)
            v8_append(rows, "B: flat + HWI", snr, seed, "SE-MOCZ", p, idx,
                      "Gaussian marginal ML after oracle HWI cleanup",
                      "oracle front-end cleanup", y.shape[1])

            y = h[:,None]*V8_X_IM[idx] + cnoise((B, V8_X_IM.shape[1]))
            p = im_dizet_decode(y, 8, 6, R_IM)
            v8_append(rows, "B: flat + HWI", snr, seed, "IM-MOCZ", p, idx,
                      "DiZeT after oracle HWI cleanup",
                      "oracle front-end cleanup", y.shape[1])

            # ================================================================
            # C — clean two-tap multipath
            # ================================================================
            rng = np.random.default_rng(20268200 + int(snr)*1000 + seed)
            idx = rng.integers(256, size=B)
            h2 = (rng.normal(size=(B,2))+1j*rng.normal(size=(B,2)))/np.sqrt(2)
            h2 *= np.sqrt(PDP2)[None,:]

            def cnoise2(shape):
                return np.sqrt(N0/2)*(rng.normal(size=shape)+1j*rng.normal(size=shape))

            y = v8_two_tap_signal(V8_X_SQM, idx, h2) + cnoise2((B,16))
            p = decode_tdla_bank(v8_complex_to_real(y), V8_SQM_2T_BANK, 4)
            v8_append(rows, "C: two-tap clean", snr, seed, "SQM", p, idx,
                      "matched rank-4 two-tap quotient", "none", 16)

            y = v8_two_tap_signal(V8_X_HWI, idx, h2) + cnoise2((B,16))
            p = decode_tdla_bank(v8_complex_to_real(y), V8_HWI_2T_BANK, 4)
            v8_append(rows, "C: two-tap clean", snr, seed, "HWI-Grassmannian", p, idx,
                      "matched rank-4 two-tap quotient", "none", 16)

            y = v8_two_tap_signal(V8_X_CUBE, idx, h2) + cnoise2((B,16))
            p = decode_tdla_bank(v8_complex_to_real(y), V8_CUBE_2T_BANK, 4)
            v8_append(rows, "C: two-tap clean", snr, seed, "Cube-Split", p, idx,
                      "matched rank-4 two-tap quotient", "none", 16)

            y = simulate_from_marginal_bank(idx, h2, se_two_bank, N0, rng)
            p = decode_gaussian_marginal(y, se_two_bank)
            v8_append(rows, "C: two-tap clean", snr, seed, "SE-MOCZ", p, idx,
                      "two-tap Gaussian marginal ML", "none", y.shape[1])

            y = conv_channel_batch(V8_X_IM[idx], h2) + cnoise2((B, V8_X_IM.shape[1]+1))
            p = im_dizet_decode(y, 8, 6, R_IM)
            v8_append(rows, "C: two-tap clean", snr, seed, "IM-MOCZ", p, idx,
                      "two-tap DiZeT + majority vote", "none", y.shape[1])

            # ================================================================
            # D — two-tap multipath + CFO + I/Q + DC
            # ================================================================
            rng = np.random.default_rng(20268300 + int(snr)*1000 + seed)
            idx = rng.integers(256, size=B)
            h2 = (rng.normal(size=(B,2))+1j*rng.normal(size=(B,2)))/np.sqrt(2)
            h2 *= np.sqrt(PDP2)[None,:]
            omega = rng.uniform(-np.deg2rad(V8_CFO_MAX_DEG), np.deg2rad(V8_CFO_MAX_DEG), B)
            A = comp_random_A(rng, B, 1.5)
            dcang = rng.uniform(0, 2*np.pi, B)
            d = np.stack([np.cos(dcang), np.sin(dcang)], axis=1)

            s = v8_two_tap_signal(V8_X_SQM, idx, h2)
            Y = v8_real_affine_observation(V8_X_SQM, idx, s, omega, A, d, N0, rng)
            p = decode_tdla_bank(Y, V8_SQM_2T_BANK, 4)
            v8_append(rows, "D: two-tap + HWI", snr, seed, "SQM", p, idx,
                      "matched rank-4 two-tap quotient + CFO search", "none", 16)

            s = v8_two_tap_signal(V8_X_HWI, idx, h2)
            Y = v8_real_affine_observation(V8_X_HWI, idx, s, omega, A, d, N0, rng)
            p = decode_tdla_bank(Y, V8_HWI_2T_BANK, 4)
            v8_append(rows, "D: two-tap + HWI", snr, seed, "HWI-Grassmannian", p, idx,
                      "matched rank-4 two-tap quotient + CFO search", "none", 16)

            s = v8_two_tap_signal(V8_X_CUBE, idx, h2)
            Y = v8_real_affine_observation(V8_X_CUBE, idx, s, omega, A, d, N0, rng)
            p = decode_tdla_bank(Y, V8_CUBE_2T_BANK, 4)
            v8_append(rows, "D: two-tap + HWI", snr, seed, "Cube-Split", p, idx,
                      "matched rank-4 two-tap quotient + CFO search", "none", 16)

            # Again, give MOCZ an oracle HWI-cleaned upper reference.
            y = simulate_from_marginal_bank(idx, h2, se_two_bank, N0, rng)
            p = decode_gaussian_marginal(y, se_two_bank)
            v8_append(rows, "D: two-tap + HWI", snr, seed, "SE-MOCZ", p, idx,
                      "two-tap marginal ML after oracle HWI cleanup",
                      "oracle front-end cleanup", y.shape[1])

            y = conv_channel_batch(V8_X_IM[idx], h2) + cnoise2((B, V8_X_IM.shape[1]+1))
            p = im_dizet_decode(y, 8, 6, R_IM)
            v8_append(rows, "D: two-tap + HWI", snr, seed, "IM-MOCZ", p, idx,
                      "two-tap DiZeT after oracle HWI cleanup",
                      "oracle front-end cleanup", y.shape[1])

            print(f"V8 SNR={snr:>4} seed={seed:>3}: four panels complete")

    v8_ps = pd.DataFrame(rows)
    v8_ps.to_csv(v8_ps_path, index=False)

v8_summary_df = v8_summary(v8_ps)
meta_cols = ["panel","method","payload_bits","observation_samples",
             "bits_per_observation_sample","receiver","calibration_assumption"]
v8_meta = v8_ps.groupby(["panel","method"], as_index=False)[meta_cols[2:]].first()
v8_summary_df = v8_summary_df.merge(v8_meta, on=["panel","method"], how="left")
v8_summary_df.to_csv(V8_DIR / "complete_1plus4_summary.csv", index=False)

display(v8_summary_df)


## VI.2 — Publication figures: all five schemes on every plot

The next cells produce:

- four individual PER-vs-SNR plots, each containing **SQM + all four competitors**;
- one compact 2×2 overview figure;
- one dedicated **headline combined-stress plot** for panel D;
- and a 10 dB snapshot/ranking table.

For panels B and D, the legend alone is intentionally kept short; the calibration assumptions are reported in the accompanying table rather than hidden in long curve labels.


In [ ]:
# ---------- Five-curve figures for every operating condition ----------

panel_titles = {
    "A: flat clean": "A — Clean flat Rayleigh fading",
    "B: flat + HWI": "B — Flat fading + CFO + I/Q + DC",
    "C: two-tap clean": "C — Clean two-tap multipath",
    "D: two-tap + HWI": "D — Two-tap multipath + CFO + I/Q + DC",
}

for panel, title in panel_titles.items():
    plt.figure(figsize=(7.4, 4.8))
    for marker, method in zip(V8_MARKERS, V8_METHODS):
        g = v8_summary_df[(v8_summary_df.panel == panel) &
                          (v8_summary_df.method == method)].sort_values("snr_db")
        plt.errorbar(g.snr_db, g.PER_mean, yerr=g.PER_95CI_halfwidth,
                     marker=marker, capsize=3, label=method)
    plt.yscale("log")
    plt.xlabel(r"$E_b/N_0$ (dB)")
    plt.ylabel("Packet error rate")
    plt.title(title)
    plt.grid(True, which="both", alpha=0.25)
    plt.legend(fontsize=8)
    plt.tight_layout()
    stem = panel.split(":")[0].lower() + "_five_scheme_per"
    plt.savefig(V8_DIR / f"{stem}.png", dpi=220, bbox_inches="tight")
    plt.savefig(V8_DIR / f"{stem}.pdf", bbox_inches="tight")
    plt.show()

# Compact overview for manuscript drafting.
fig, axes = plt.subplots(2, 2, figsize=(12.0, 8.0), sharex=True)
for ax, (panel, title) in zip(axes.ravel(), panel_titles.items()):
    for marker, method in zip(V8_MARKERS, V8_METHODS):
        g = v8_summary_df[(v8_summary_df.panel == panel) &
                          (v8_summary_df.method == method)].sort_values("snr_db")
        ax.errorbar(g.snr_db, g.PER_mean, yerr=g.PER_95CI_halfwidth,
                    marker=marker, capsize=2, label=method)
    ax.set_yscale("log")
    ax.set_title(title, fontsize=10)
    ax.set_xlabel(r"$E_b/N_0$ (dB)")
    ax.set_ylabel("PER")
    ax.grid(True, which="both", alpha=0.22)
axes[0,0].legend(fontsize=7)
fig.tight_layout()
fig.savefig(V8_DIR / "complete_1plus4_four_panel_overview.png", dpi=220, bbox_inches="tight")
fig.savefig(V8_DIR / "complete_1plus4_four_panel_overview.pdf", bbox_inches="tight")
plt.show()

# Dedicated headline figure: hardest combined condition.
headline = "D: two-tap + HWI"
plt.figure(figsize=(7.6, 4.9))
for marker, method in zip(V8_MARKERS, V8_METHODS):
    g = v8_summary_df[(v8_summary_df.panel == headline) &
                      (v8_summary_df.method == method)].sort_values("snr_db")
    plt.errorbar(g.snr_db, g.PER_mean, yerr=g.PER_95CI_halfwidth,
                 marker=marker, capsize=3, label=method)
plt.yscale("log")
plt.xlabel(r"$E_b/N_0$ (dB)")
plt.ylabel("Packet error rate")
plt.title("Headline 1+4 stress test: multipath + CFO + I/Q + DC")
plt.grid(True, which="both", alpha=0.25)
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(V8_DIR / "headline_five_scheme_combined_stress.png", dpi=240, bbox_inches="tight")
plt.savefig(V8_DIR / "headline_five_scheme_combined_stress.pdf", bbox_inches="tight")
plt.show()


## VI.3 — Quantify the 1+4 result rather than relying only on curves

For each panel and SNR, the notebook computes the paired difference

\[
\Delta = \mathrm{PER}_{\text{competitor}} - \mathrm{PER}_{\text{SQM}}.
\]

A positive value means SQM has lower PER. The paired interval is across the same independent seeds. This is especially useful for the combined-stress panel, where the MOCZ curves are already oracle-aided with respect to hardware calibration.


In [ ]:
# ---------- Pairwise SQM-vs-competitor deltas and 10 dB ranking ----------

pair_rows = []
for (panel, snr), g in v8_ps.groupby(["panel","snr_db"]):
    p = g.pivot(index="seed", columns="method", values="PER")
    if "SQM" not in p.columns:
        continue
    for method in V8_METHODS[1:]:
        d = p[method] - p["SQM"]
        pair_rows.append({
            "panel": panel,
            "snr_db": snr,
            "competitor": method,
            "mean_delta_PER_competitor_minus_SQM": float(d.mean()),
            "paired_95CI_halfwidth": float(t_halfwidth(d.to_numpy())),
            "seeds": int(len(d)),
            "SQM_lower_PER_if_positive": bool(d.mean() > 0),
        })

v8_deltas = pd.DataFrame(pair_rows)
v8_deltas.to_csv(V8_DIR / "sqm_vs_four_paired_deltas.csv", index=False)
display(v8_deltas)

# Ranking snapshot near 10 dB.
target_snr = min(V8_SNRS, key=lambda x: abs(float(x)-10.0))
v8_rank10 = (
    v8_summary_df[np.isclose(v8_summary_df.snr_db, float(target_snr))]
    .sort_values(["panel","PER_mean"])
    .copy()
)
v8_rank10["rank"] = v8_rank10.groupby("panel")["PER_mean"].rank(method="min")
v8_rank10.to_csv(V8_DIR / "ranking_at_10dB.csv", index=False)
display(v8_rank10[["panel","rank","method","PER_mean","PER_95CI_halfwidth",
                   "receiver","calibration_assumption"]])

# Combined-stress snapshot at 10 dB.
bar = v8_rank10[v8_rank10.panel.eq("D: two-tap + HWI")].copy()
bar["method"] = pd.Categorical(bar["method"], categories=V8_METHODS, ordered=True)
bar = bar.sort_values("method")
plt.figure(figsize=(7.5, 4.6))
plt.bar(bar.method.astype(str), bar.PER_mean,
        yerr=bar.PER_95CI_halfwidth, capsize=4)
plt.ylabel("Packet error rate")
plt.xlabel("Scheme")
plt.title(f"Combined-stress five-scheme snapshot at {float(target_snr):g} dB")
plt.xticks(rotation=20, ha="right")
plt.grid(True, axis="y", alpha=0.25)
plt.tight_layout()
plt.savefig(V8_DIR / "combined_stress_snapshot_10dB.png", dpi=220, bbox_inches="tight")
plt.savefig(V8_DIR / "combined_stress_snapshot_10dB.pdf", bbox_inches="tight")
plt.show()


## VI.4 — Interpretation guardrails for the manuscript

Use the V8 matrix in the manuscript as follows:

- **Panel A** is the neutral common-channel baseline.
- **Panel C** asks whether SQM remains competitive when the channel becomes frequency selective.
- **Panel B** and especially **Panel D** are robustness stress tests. The MOCZ-family points in those panels are **oracle hardware-cleaned upper references**, not blind implementations under the full SQM impairment model.
- The **headline panel D** is therefore conservative with respect to SE-MOCZ and IM-MOCZ: if SQM has lower PER, the result cannot be attributed to simply denying those schemes hardware calibration.
- HWI-Grassmannian and Cube-Split are additionally given matched selective-quotient decoding in the multipath panels. This prevents the comparison from being won merely because their native flat-fading line receiver is mismatched to multipath.
- Do **not** claim universal state-of-the-art superiority from these plots. Claim only what the matched/adapted assumptions support.

The V7 TDL-A SQM-vs-HWI result should remain alongside the V8 all-five matrix: V8 gives breadth; the TDL-A pairwise experiment gives depth against the closest competitor.


In [ ]:
# ---------- Final V8 comparison ledger / output index ----------

v8_ledger = pd.DataFrame([
    ["A: flat clean", "all five", "common 8-bit / Eb=1 flat-Rayleigh benchmark", "direct common-channel benchmark"],
    ["B: flat + HWI", "all five", "SQM/HWI/Cube see hardware nuisance; MOCZ gets oracle front-end cleanup", "stress test; MOCZ competitor-favorable upper reference"],
    ["C: two-tap clean", "all five", "common PDP [0.8,0.2]; HWI/Cube get matched quotient receiver", "multipath waveform/receiver comparison"],
    ["D: two-tap + HWI", "all five", "combined multipath + CFO + I/Q + DC; MOCZ oracle-cleaned", "headline conservative robustness stress test"],
    ["V7 TDL-A decisive test", "SQM vs HWI", "full fractional TDL-A + CFO + I/Q + DC", "closest-competitor depth test"],
], columns=["experiment","schemes","assumptions","manuscript_role"])
v8_ledger.to_csv(V8_DIR / "v8_comparison_ledger.csv", index=False)
display(v8_ledger)

v8_outputs = pd.DataFrame([
    {"file": str(p.relative_to(ROOT)), "size_bytes": p.stat().st_size}
    for p in sorted(V8_DIR.rglob("*")) if p.is_file()
])
v8_outputs.to_csv(V8_DIR / "output_index.csv", index=False)
display(v8_outputs)

print("\n=== V8 COMPLETE 1+4 MATRIX READY ===")
print("Output directory:", V8_DIR)
print("For the final high-confidence run set SQM_V8_SCALE=publication and rerun Part VI.")


## Part VII — Consolidated comparison ledger and manuscript-use rules

The comparative experiments intentionally answer **different scientific questions**:

1. **Factorial ablation:** are the two claimed SQM ingredients—incidence-aware training and selective rank—individually useful?
2. **Cube-Split:** is SQM competitive with a structured noncoherent Grassmannian line code under flat fading?
3. **Hardware-aware Grassmannian, flat/HWI:** does SQM remain competitive against a constellation explicitly robustified for CFO/IQ uncertainty?
4. **MOCZ family:** how does SQM compare against blind polynomial short-packet schemes, including spectral-efficiency and index-modulated variants?
5. **Decisive matched HWI experiment:** when SQM and HWI-Grassmannian see the same TDL-A/two-tap multipath, CFO, I/Q, DC, noise, payload, energy and blocklength, does SQM retain an advantage?

### Manuscript rule

Only call a result a **direct numerical comparison** when the payload, channel law, energy convention, blocklength, and impairment assumptions are actually matched. Otherwise call it a **native/source-model benchmark** or **stress-test adaptation** and report the mismatch explicitly.

The released `train_selective_incidence.py` initializes incidence-aware training from `Artifact/stage3_codebooks.npz::X_dmm`, so the stage-3 codebook remains the natural released **pre-incidence** control.


The V8 complete 1+4 matrix in Part VI is the recommended comparison package for the manuscript; retain the earlier family-specific plots as supporting diagnostics rather than substitutes for the all-five figures.


In [ ]:
ledger_rows = [
    ["SQM factorial ablation", "DIRECT / SAME MODEL",
     "All four cells use identical TDL-A nuisance/noise draws.",
     "Use in main manuscript or ablation subsection."],

    ["Cube-Split CS(4,1) native", "SOURCE-FAITHFUL NATIVE CHECK",
     "Exact 256-line source construction; native T=4 differs from SQM L=16.",
     "Use as related-work validation, not as a same-budget headline comparison."],

    ["Cube-Split 16-sample embedding", "MATCHED FLAT-FADING DIAGNOSTIC",
     "Same 8 bits and 16 samples; fixed isometry preserves Grassmannian distances.",
     "Suitable as a controlled flat-fading comparison."],

    ["HWI-aware Grassmannian", "MATCHED SINGLE-STREAM ADAPTATION",
     "Source Algorithm-1 ideas adapted from M=2,K=64 to M=1,K=256,T=16.",
     "Use with explicit 'single-stream adaptation' wording."],

    ["BMOCZ", "DIRECT TWO-TAP SHORT-PACKET BASELINE",
     "8 bits, matched energy, Gaussian true-PDP marginal ML.",
     "Strong direct blind baseline."],

    ["SE-MOCZ", "SOURCE-MODEL ADAPTATION",
     "tau=0.9, beta=0.3, 17-tap matched-pulse model; comparative study uses Gaussian marginal ML rather than paper RFMD.",
     "Report as independent source-model implementation; do not call exact paper reproduction."],

    ["IM-MOCZ", "MATCHED-PAYLOAD ADAPTATION",
     "N=8,K=6 instead of source N=10 examples; DiZeT + majority vote follows source equations.",
     "Report adaptation and native efficiency separately."],

    ["Unified five-scheme flat-fading benchmark", "COMMON-CHANNEL CROSS-SCHEME BENCHMARK",
     "Same 8-bit payload, Eb=1 and one-tap Rayleigh fading; native waveform lengths/receivers retained.",
     "Use as the single all-five comparison figure; do not describe it as universal superiority."],

    ["Decisive SQM vs HWI-Grassmannian TDL-A + CFO + I/Q + DC",
     "DIRECT MATCHED ENVIRONMENT + CONSERVATIVE HWI STRESS TEST",
     "Same 8-bit payload, L=16, energy, TDL-A draw, CFO, static 2x2 I/Q transform, DC, AWGN, seeds and packet budget. HWI is tested both with its closest line receiver and with a held-out-tuned selective-quotient receiver.",
     "This is the decisive performance-novelty comparison. Interpret the two HWI receivers separately."],

    ["Secondary SQM vs HWI-Grassmannian two-tap + CFO + I/Q + DC",
     "DIRECT MATCHED TWO-TAP STRESS TEST",
     "Same 8-bit payload, L=16, energy, two-tap PDP [0.8,0.2], CFO, I/Q, DC and AWGN.",
     "Use as supporting evidence showing whether the conclusion depends on full TDL-A richness."],
]

comp_ledger = pd.DataFrame(
    ledger_rows,
    columns=["item","comparison_status","why","recommended_manuscript_use"]
)
display(comp_ledger)
comp_ledger.to_csv(COMP_DIR / "comparison_ledger.csv", index=False)

# Compact machine-readable index of all comparative-study outputs, including
# the nested decisive_hwi_vs_sqm directory.
output_index = []
for p in sorted(COMP_DIR.rglob("*")):
    if p.is_file():
        output_index.append({
            "file": str(p.relative_to(ROOT)),
            "size_bytes": p.stat().st_size,
        })
output_index = pd.DataFrame(output_index)
output_index.to_csv(COMP_DIR / "output_index.csv", index=False)

print("\n=== COMPARATIVE STUDY COMPLETE ===")
print("Output directory:", COMP_DIR)
display(output_index)

print("\nInterpretation checklist for the manuscript:")
print("1. Quote ablation effects only from the paired factorial table.")
print("2. Keep Cube-Split native T=4 separate from equal-L=16 embedding.")
print("3. Call HWI-aware Grassmannian a single-stream matched adaptation.")
print("4. Call SE-MOCZ a source-model independent implementation, not an exact reproduction.")
print("5. Call IM-MOCZ a matched N=8,K=6 adaptation.")
print("6. Use the unified five-scheme curve only as the stated common flat-fading energy benchmark.")
print("7. Keep family-specific HWI and multipath comparisons for the corresponding novelty claims.")
print("8. Treat the matched TDL-A/two-tap + CFO + I/Q + DC experiment as the decisive SQM-vs-HWI performance-novelty test.")
print("9. If HWI+matched-selective-quotient catches SQM, attribute the gain primarily to selective quotienting rather than to SQM waveform design alone.")


print("\nV8 recommendation: use Part VI all-five matrix as the main comparison package, with the V7 TDL-A SQM-vs-HWI test as the closest-competitor depth experiment.")
